# **Metriche PostgreSQL - DSpace**

## **Phase zero**

### Install requirements

In [1]:
!pip install -r ../requirements.txt

You should consider upgrading via the 'C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\virtual_env\Scripts\python.exe -m pip install --upgrade pip' command.


### Import packages and configuration loading

In [2]:
import os
import psycopg2
import pandas as pd
from datetime import date, timedelta
from pathlib import Path
import yaml
from sshtunnel import SSHTunnelForwarder

EXPORT_SOURCE = "db"
MONTHLY_EXPORT_FILES = {
    "db_active_users_in_month_last_active.csv",
    "db_users_inactive_over_12_months.csv",
    "db_active_users_last_12_months.csv",
    "db_items_uploaded_in_month_with_bitstreams.csv",
    "db_items_uploaded_in_month_by_submitter.csv",
    "db_items_uploaded_in_month_by_collection.csv",
    "db_items_uploaded_in_month_by_language.csv",
    "db_items_modified_in_month_from_provenance.csv",
    "db_avg_file_size_month.csv",
    "db_items_uploaded_in_month_details.csv",
    "db_active_collections_combined_in_month.csv",
}

def load_config(config_path=None):
    config_path = config_path or os.environ.get("DSPACE_REPORTING_CONFIG", "../config/settings.local.yaml")
    with open(config_path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def next_month_start(month_label):
    year, month = map(int, month_label.split("-"))
    if month == 12:
        return date(year + 1, 1, 1)
    return date(year, month + 1, 1)

def get_scoped_export_dir(scope):
    export_root = PROJECT_ROOT / config["paths"].get("exports_root", "exports")
    base_dir = export_root / EXPORT_SOURCE / scope
    if config.get("run", {}).get("create_month_subfolders", True):
        return base_dir / REPORT_MONTH
    return base_dir

def export_path(filename):
    scope = "monthly" if filename in MONTHLY_EXPORT_FILES else "always"
    output_path = get_scoped_export_dir(scope) / filename
    if EXPORT_OUTPUTS:
        output_path.parent.mkdir(parents=True, exist_ok=True)
    return output_path

config = load_config()

REPORT_MONTH = config["run"]["reference_month"]          # es. 2026-03
REPORT_MONTH_DATE = f"{REPORT_MONTH}-01"                 # es. 2026-03-01
REPORT_MONTH_END_DATE = next_month_start(REPORT_MONTH).isoformat()
REPORT_AS_OF_DATE = (next_month_start(REPORT_MONTH) - timedelta(days=1)).isoformat()
OVERWRITE_EXPORTS = config.get("run", {}).get("overwrite_exports", False)
EXPORT_OUTPUTS = config.get("run", {}).get("export_outputs", True)
ENVIRONMENT = config["project"]["environment"]
TIMEZONE = config["project"]["timezone"]

PROJECT_ROOT = Path.cwd().parent
EXPORT_ROOT = PROJECT_ROOT / config["paths"].get("exports_root", "exports")
source_export_dir = EXPORT_ROOT / EXPORT_SOURCE
monthly_export_dir = get_scoped_export_dir("monthly")
always_export_dir = get_scoped_export_dir("always")
export_dir = source_export_dir  # compatibilita con celle esistenti che creano la directory
_original_to_csv = pd.DataFrame.to_csv

def guarded_to_csv(self, path_or_buf=None, *args, **kwargs):
    if path_or_buf is not None and not EXPORT_OUTPUTS:
        output_path = Path(path_or_buf)
        print(f"Display-only mode, export skipped: {output_path.resolve()}")
        return None
    if path_or_buf is not None and not OVERWRITE_EXPORTS:
        output_path = Path(path_or_buf)
        if output_path.exists():
            print(f"Export exists and overwrite_exports=false, skipped: {output_path.resolve()}")
            return None
    return _original_to_csv(self, path_or_buf, *args, **kwargs)

pd.DataFrame.to_csv = guarded_to_csv


C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\virtual_env\lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\virtual_env\lib\site-packages\paramiko\transport.py:253: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "class": algorithms.TripleDES,


### SSH Tunnel

In [3]:
print("=== Context of implementation ===")
print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Environment: {ENVIRONMENT}")
print(f"Timezone: {TIMEZONE}")
print(f"Reference month: {REPORT_MONTH}")
print(f"Reference period: {REPORT_MONTH_DATE} to {REPORT_AS_OF_DATE}")
print(f"Export outputs: {EXPORT_OUTPUTS}")
print(f"DB monthly export directory: {monthly_export_dir.resolve()}")
print(f"DB always export directory: {always_export_dir.resolve()}")
print()

tunnel = None
conn = None


# === SSH TUNNEL ===
if config["ssh"]["enabled"]:
    tunnel = SSHTunnelForwarder(
        (config["ssh"]["host"], config["ssh"]["port"]),
        ssh_username=config["ssh"]["username"],
        ssh_password=config["ssh"]["password"],
        remote_bind_address=(
            config["ssh"]["remote_bind_host"],
            config["ssh"]["remote_bind_port"]
        ),
        local_bind_address=(
            config["ssh"]["local_bind_host"],
            config["ssh"]["local_bind_port"]
        )
    )
    tunnel.start()
    print("SSH tunnel established successfully.")

# === CONNESSIONE DB ===
conn = psycopg2.connect(
    host=config["postgres"]["host"],
    port=config["postgres"]["port"],
    database=config["postgres"]["database"],
    user=config["postgres"]["user"],
    password=config["postgres"]["password"],
    connect_timeout=10
)

print("Database connection successful.")

2026-05-11 17:37:38,495| ERROR   | Password is required for key C:\Users\Michele Mallia/.ssh\id_ed25519


=== Context of implementation ===
Project root: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks
Environment: preprod
Timezone: Europe/Rome
Reference month: 2026-04
DB monthly export directory: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\monthly\2026-04
DB always export directory: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04

SSH tunnel established successfully.


Database connection successful.


## **Item**

### Total items

In [4]:
# - item_id
# - deposit_date
# - title
# - handles
# - has_handle
# - in_archive
# - withdrawn
# - discoverable
# - publicly_reachable_candidate
# - item_public_status
# - owning_collection_id
# - collections_count
# - bitstreams_count
# - has_bitstreams
# - total_size_bytes
# - total_size_mb
# - last_modified
# - submitter_id
# - submitter_email
# - reference_month
# - environment
# - source
# - metric_definition

query_all_items_by_archive_status = """
WITH provenance_dates AS (
    SELECT
        mv.dspace_object_id AS item_id,
        MIN(
            ((regexp_match(
                mv.text_value,
                'on ([0-9]{4}-[0-9]{2}-[0-9]{2})'
            ))[1])::date
        ) AS deposit_date
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'description'
      AND mfr.qualifier = 'provenance'
      AND mv.text_value ~ 'on [0-9]{4}-[0-9]{2}-[0-9]{2}'
    GROUP BY mv.dspace_object_id
),

item_titles AS (
    SELECT
        mv.dspace_object_id AS item_id,
        STRING_AGG(DISTINCT mv.text_value, '; ' ORDER BY mv.text_value) AS title
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mfr.qualifier IS NULL
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),

item_handles AS (
    SELECT
        mv.dspace_object_id AS item_id,
        STRING_AGG(DISTINCT mv.text_value, '; ' ORDER BY mv.text_value) AS handles
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'identifier'
      AND mfr.qualifier = 'uri'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
      AND (
            mv.text_value ILIKE '%%handle%%'
         OR mv.text_value ILIKE '%%hdl.handle.net%%'
      )
    GROUP BY mv.dspace_object_id
),

item_collections AS (
    SELECT
        i.uuid AS item_id,
        i.owning_collection AS owning_collection_id,
        COUNT(DISTINCT c2i.collection_id) AS collections_count
    FROM item i
    LEFT JOIN collection2item c2i
        ON i.uuid = c2i.item_id
    GROUP BY
        i.uuid,
        i.owning_collection
),

item_bitstreams AS (
    SELECT
        i.uuid AS item_id,
        COUNT(DISTINCT bs.uuid) AS bitstreams_count,
        COALESCE(SUM(bs.size_bytes), 0) AS total_size_bytes,
        ROUND(COALESCE(SUM(bs.size_bytes), 0) / 1024.0 / 1024.0, 2) AS total_size_mb
    FROM item i
    LEFT JOIN item2bundle i2b
        ON i.uuid = i2b.item_id
    LEFT JOIN bundle b
        ON i2b.bundle_id = b.uuid
    LEFT JOIN bundle2bitstream b2bs
        ON b.uuid = b2bs.bundle_id
    LEFT JOIN bitstream bs
        ON b2bs.bitstream_id = bs.uuid
       AND (bs.deleted = false OR bs.deleted IS NULL)
    GROUP BY i.uuid
)

SELECT
    i.uuid AS item_id,
    pd.deposit_date,
    it.title,
    ih.handles,

    CASE
        WHEN ih.handles IS NOT NULL THEN true
        ELSE false
    END AS has_handle,

    i.in_archive,
    i.withdrawn,
    i.discoverable,

    CASE
        WHEN i.in_archive = true
         AND i.withdrawn = false
         AND i.discoverable = true
            THEN true
        ELSE false
    END AS publicly_reachable_candidate,

    CASE
        WHEN i.in_archive = true
         AND i.withdrawn = false
         AND i.discoverable = true
            THEN 'publicly_reachable'
        WHEN i.in_archive = true
         AND i.withdrawn = false
         AND i.discoverable = false
            THEN 'archived_not_discoverable'
        WHEN i.in_archive = true
         AND i.withdrawn = true
            THEN 'archived_withdrawn'
        WHEN i.in_archive = false
         AND i.withdrawn = true
            THEN 'not_archived_withdrawn'
        WHEN i.in_archive = false
         AND COALESCE(i.withdrawn, false) = false
            THEN 'not_in_archive'
        ELSE 'other'
    END AS item_public_status,

    ic.owning_collection_id,
    COALESCE(ic.collections_count, 0) AS collections_count,

    COALESCE(ib.bitstreams_count, 0) AS bitstreams_count,

    CASE
        WHEN COALESCE(ib.bitstreams_count, 0) > 0 THEN true
        ELSE false
    END AS has_bitstreams,

    COALESCE(ib.total_size_bytes, 0) AS total_size_bytes,
    COALESCE(ib.total_size_mb, 0) AS total_size_mb,

    i.last_modified,
    i.submitter_id,
    e.email AS submitter_email

FROM item i
LEFT JOIN provenance_dates pd
    ON i.uuid = pd.item_id
LEFT JOIN item_titles it
    ON i.uuid = it.item_id
LEFT JOIN item_handles ih
    ON i.uuid = ih.item_id
LEFT JOIN item_collections ic
    ON i.uuid = ic.item_id
LEFT JOIN item_bitstreams ib
    ON i.uuid = ib.item_id
LEFT JOIN eperson e
    ON i.submitter_id = e.uuid

ORDER BY
    i.in_archive DESC NULLS LAST,
    i.withdrawn ASC NULLS LAST,
    i.discoverable DESC NULLS LAST,
    has_handle ASC,
    has_bitstreams DESC,
    bitstreams_count DESC,
    i.last_modified DESC NULLS LAST,
    i.uuid;
"""

df_all_items_by_archive_status = pd.read_sql_query(
    query_all_items_by_archive_status,
    conn
)

df_all_items_by_archive_status["reference_month"] = REPORT_MONTH
df_all_items_by_archive_status["environment"] = ENVIRONMENT
df_all_items_by_archive_status["source"] = "postgres"
df_all_items_by_archive_status["metric_definition"] = (
    "all items from the item table, including archived, non-archived and withdrawn items; "
    "public reachability classified using in_archive, withdrawn and discoverable; "
    "includes title, handle presence, collection mappings, non-deleted bitstream count and total size"
)

output_all_items_by_archive_status = export_path("db_all_items_by_archive_status.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_all_items_by_archive_status.to_csv(output_all_items_by_archive_status, index=False)

print(f"Total items extracted from the item table: {len(df_all_items_by_archive_status)}")

print("Distribution by public status:")
display(
    df_all_items_by_archive_status["item_public_status"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "item_public_status", "item_public_status": "count"})
)

print("Distribution by in_archive:")
display(
    df_all_items_by_archive_status["in_archive"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "in_archive", "in_archive": "count"})
)

print("Distribution by withdrawn:")
display(
    df_all_items_by_archive_status["withdrawn"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "withdrawn", "withdrawn": "count"})
)

print("Distribution by discoverable:")
display(
    df_all_items_by_archive_status["discoverable"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "discoverable", "discoverable": "count"})
)

print("Distribution by handle availability:")
display(
    df_all_items_by_archive_status["has_handle"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "has_handle", "has_handle": "count"})
)

print("Distribution by bitstream availability:")
display(
    df_all_items_by_archive_status["has_bitstreams"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "has_bitstreams", "has_bitstreams": "count"})
)

display(df_all_items_by_archive_status.head())

print(f"File saved to: {output_all_items_by_archive_status.resolve()}")

Total items extracted from the item table: 1028
Distribution by public status:


,count,count
0,publicly_reachable,969
1,not_in_archive,56
2,not_archived_withdrawn,3


Distribution by in_archive:


,count,count
0,True,969
1,False,59


Distribution by withdrawn:


...

,count,count
0,True,999
1,False,29


Distribution by bitstream availability:


,count,count
0,True,552
1,False,476


,item_id,handles,has_handle,in_archive,withdrawn,discoverable,publicly_reachable_candidate,item_public_status,owning_collection_id,collections_count,...,has_bitstreams,total_size_bytes,total_size_mb,last_modified,submitter_id,submitter_email,reference_month,environment,source,metric_definition
0,776f944f-c44b-4230-9960-f13a861593da,http://hdl.handle.net/20.500.11752/OPEN-1012,True,True,False,True,True,publicly_reachable,57c1de63-635d-41ef-8e64-119865cb8f80,1,...,True,12188826.0,11.62,2025-11-18 15:56:50.363000+00:00,68df5541-8dba-47fa-a729-e924d8bda0e9,enrica.salvatori@unipi.it,2026-04,preprod,postgres,"all items from the item table, including archi..."
1,7b7e2e48-fbb4-452c-bf40-b67577af70f6,http://hdl.handle.net/20.500.11752/OPEN-1023,True,True,False,True,True,publicly_reachable,57c1de63-635d-41ef-8e64-119865cb8f80,1,...,True,6682349.0,6.37,2025-11-18 16:19:24.222000+00:00,daca5197-c4a3-45e8-a31e-2566a36ea518,francesca.murano@unifi.it,2026-04,preprod,postgres,"all items from the item table, including archi..."
2,93b3c336-98c7-413f-bde7-713601992914,http://hdl.handle.net/20.500.11752/OPEN-1032,True,True,False,True,True,publicly_reachable,57c1de63-635d-41ef-8e64-119865cb8f80,1,...,True,2238561.0,2.13,2025-12-11 14:29:37.351000+00:00,dac4c1b3-f4ca-4229-ba2f-3ddb0c69f6b9,luca.rigobianco@unive.it,2026-04,preprod,postgres,"all items from the item table, including archi..."
3,be9a4bee-0fe2-458f-8d7a-8979d5b3fb77,http://hdl.handle.net/20.500.11752/ILC-1040,True,True,False,True,True,publicly_reachable,79c6fbcd-aaac-42b5-aa7d-9413eb906511,1,...,True,159530.0,0.15,2025-11-18 16:19:09.625000+00:00,56b85948-99f2-4d72-9ec7-59081008408e,sebastiano.giacomin2@unibo.it,2026-04,preprod,postgres,"all items from the item table, including archi..."
4,89168f1e-52e0-4184-b761-942ccdc7de14,http://hdl.handle.net/20.500.11752/OPEN-1031,True,True,False,True,True,publicly_reachable,57c1de63-635d-41ef-8e64-119865cb8f80,1,...,True,542255.0,0.52,2025-11-18 16:19:12.128000+00:00,dac4c1b3-f4ca-4229-ba2f-3ddb0c69f6b9,luca.rigobianco@unive.it,2026-04,preprod,postgres,"all items from the item table, including archi..."


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_all_items_by_archive_status.csv


### Items without handle

In [5]:
# - item_id
# - in_archive
# - withdrawn
# - discoverable
# - publicly_reachable_candidate
# - item_public_status
# - bitstreams_count
# - has_bitstreams
# - total_size_bytes
# - total_size_mb
# - last_modified
# - submitter_id
# - submitter_email
# - reference_month
# - environment
# - source
# - metric_definition


query_items_without_handle = """
WITH item_handles AS (
    SELECT
        mv.dspace_object_id AS item_id,
        STRING_AGG(DISTINCT mv.text_value, '; ' ORDER BY mv.text_value) AS handles
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'identifier'
      AND mfr.qualifier = 'uri'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
      AND (
            mv.text_value ILIKE '%%handle%%'
         OR mv.text_value ILIKE '%%hdl.handle.net%%'
      )
    GROUP BY mv.dspace_object_id
),
item_bitstreams AS (
    SELECT
        i.uuid AS item_id,
        COUNT(DISTINCT bs.uuid) AS bitstreams_count,
        COALESCE(SUM(bs.size_bytes), 0) AS total_size_bytes,
        ROUND(COALESCE(SUM(bs.size_bytes), 0) / 1024.0 / 1024.0, 2) AS total_size_mb
    FROM item i
    LEFT JOIN item2bundle i2b
        ON i.uuid = i2b.item_id
    LEFT JOIN bundle b
        ON i2b.bundle_id = b.uuid
    LEFT JOIN bundle2bitstream b2bs
        ON b.uuid = b2bs.bundle_id
    LEFT JOIN bitstream bs
        ON b2bs.bitstream_id = bs.uuid
       AND (bs.deleted = false OR bs.deleted IS NULL)
    GROUP BY i.uuid
)
SELECT
    i.uuid AS item_id,
    i.in_archive,
    i.withdrawn,
    i.discoverable,
    CASE
        WHEN i.in_archive = true
         AND i.withdrawn = false
         AND i.discoverable = true
            THEN true
        ELSE false
    END AS publicly_reachable_candidate,
    CASE
        WHEN i.in_archive = true
         AND i.withdrawn = false
         AND i.discoverable = true
            THEN 'publicly_reachable'
        WHEN i.in_archive = true
         AND i.withdrawn = false
         AND i.discoverable = false
            THEN 'archived_not_discoverable'
        WHEN i.withdrawn = true
            THEN 'withdrawn'
        WHEN i.in_archive = false
            THEN 'not_in_archive'
        ELSE 'other'
    END AS item_public_status,
    COALESCE(ib.bitstreams_count, 0) AS bitstreams_count,
    CASE
        WHEN COALESCE(ib.bitstreams_count, 0) > 0 THEN true
        ELSE false
    END AS has_bitstreams,
    COALESCE(ib.total_size_bytes, 0) AS total_size_bytes,
    COALESCE(ib.total_size_mb, 0) AS total_size_mb,
    i.last_modified,
    i.submitter_id,
    e.email AS submitter_email
FROM item i
LEFT JOIN item_handles ih
    ON i.uuid = ih.item_id
LEFT JOIN item_bitstreams ib
    ON i.uuid = ib.item_id
LEFT JOIN eperson e
    ON i.submitter_id = e.uuid
WHERE ih.item_id IS NULL
ORDER BY
    publicly_reachable_candidate DESC,
    has_bitstreams DESC,
    bitstreams_count DESC,
    i.in_archive DESC,
    i.discoverable DESC,
    i.withdrawn ASC,
    i.last_modified DESC NULLS LAST,
    i.uuid;
"""

df_items_without_handle = pd.read_sql_query(query_items_without_handle, conn)

df_items_without_handle["reference_month"] = REPORT_MONTH
df_items_without_handle["environment"] = ENVIRONMENT
df_items_without_handle["source"] = "postgres"
df_items_without_handle["metric_definition"] = (
    "items without dc.identifier.uri containing a handle; "
    "public reachability classified using in_archive, withdrawn and discoverable; "
    "includes non-deleted bitstream count and total size"
)

output_items_without_handle = export_path("db_items_without_handle_all_statuses.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_items_without_handle.to_csv(output_items_without_handle, index=False)

print(f"Item senza handle: {len(df_items_without_handle)}")

print("Distribution by public sector:")
display(df_items_without_handle["item_public_status"].value_counts().reset_index())

print("Bitstream distribution:")
display(df_items_without_handle["has_bitstreams"].value_counts().reset_index())

display(df_items_without_handle.head())
print(f"File salvato in: {output_items_without_handle.resolve()}")

Item senza handle: 29
Distribution by public sector:


,item_public_status,count
0,not_in_archive,29


Bitstream distribution:


,has_bitstreams,count
0,False,22
1,True,7


,item_id,in_archive,withdrawn,discoverable,publicly_reachable_candidate,item_public_status,bitstreams_count,has_bitstreams,total_size_bytes,total_size_mb,last_modified,submitter_id,submitter_email,reference_month,environment,source,metric_definition
0,b7ce9294-eee3-4b4f-a32d-34c85441f06e,False,False,True,False,not_in_archive,3,True,3738676.0,3.57,2025-11-18 14:53:50.315000+00:00,dbdec1f3-d5db-4f5e-8097-debc7e6eec22,f.poli12@studenti.unipi.it,2026-04,preprod,postgres,items without dc.identifier.uri containing a h...
1,94c37c91-2d57-4618-8c61-592b38e951e0,False,False,True,False,not_in_archive,2,True,330886.0,0.32,2025-11-18 14:53:37.014000+00:00,29c07a05-b333-4496-891f-09557a19ca4a,francesca.frontini@ilc.cnr.it,2026-04,preprod,postgres,items without dc.identifier.uri containing a h...
2,a72ef9c3-0c6b-4672-9520-36192f309796,False,False,True,False,not_in_archive,1,True,1684.0,0.00,2026-03-04 15:40:30.800000+00:00,b217ca93-fa47-488d-87cb-1ad4f9155853,valeria.quochi@ilc.cnr.it,2026-04,preprod,postgres,items without dc.identifier.uri containing a h...
3,425da4f3-616c-44aa-b2c7-b523c090301c,False,False,True,False,not_in_archive,1,True,3680970.0,3.51,2025-11-18 16:17:35.945000+00:00,3a6d692b-a795-4ca2-851f-d4d2bac81e7d,andreschauta81@gmail.com,2026-04,preprod,postgres,items without dc.identifier.uri containing a h...
4,43c0b3c1-ef55-4853-88cb-1f9842dbd041,False,False,True,False,not_in_archive,1,True,19446.0,0.02,2025-11-18 16:15:33.153000+00:00,b346c704-a402-4a57-8800-1ac31d559d16,martin.critelli@ilc.cnr.it,2026-04,preprod,postgres,items without dc.identifier.uri containing a h...


File salvato in: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_items_without_handle_all_statuses.csv


### Items without collection

In [6]:
# - item_id
# - deposit_date
# - handles
# - in_archive
# - withdrawn
# - discoverable
# - last_modified
# - submitter_id
# - submitter_email
# - bitstreams_count
# - total_size_bytes
# - total_size_mb
# - reference_month
# - environment
# - source
# - metric_definition

query_items_without_collection = """
WITH provenance_dates AS (
    SELECT
        mv.dspace_object_id AS item_id,
        MIN(
            ((regexp_match(
                mv.text_value,
                'on ([0-9]{4}-[0-9]{2}-[0-9]{2})'
            ))[1])::date
        ) AS deposit_date
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'description'
      AND mfr.qualifier = 'provenance'
      AND mv.text_value ~ 'on [0-9]{4}-[0-9]{2}-[0-9]{2}'
    GROUP BY mv.dspace_object_id
),
item_handles AS (
    SELECT
        h.resource_id AS item_id,
        STRING_AGG(DISTINCT h.handle, '; ' ORDER BY h.handle) AS handles
    FROM handle h
    WHERE h.resource_type_id = 2
    GROUP BY h.resource_id
),
item_bitstreams AS (
    SELECT
        i.uuid AS item_id,
        COUNT(DISTINCT bs.uuid) AS bitstreams_count,
        COALESCE(SUM(bs.size_bytes), 0) AS total_size_bytes,
        ROUND(COALESCE(SUM(bs.size_bytes), 0) / 1024.0 / 1024.0, 2) AS total_size_mb
    FROM item i
    LEFT JOIN item2bundle i2b
        ON i.uuid = i2b.item_id
    LEFT JOIN bundle b
        ON i2b.bundle_id = b.uuid
    LEFT JOIN bundle2bitstream b2bs
        ON b.uuid = b2bs.bundle_id
    LEFT JOIN bitstream bs
        ON b2bs.bitstream_id = bs.uuid
       AND (bs.deleted = false OR bs.deleted IS NULL)
    GROUP BY i.uuid
)
SELECT
    i.uuid AS item_id,
    pd.deposit_date,
    ih.handles,
    i.in_archive,
    i.withdrawn,
    i.discoverable,
    i.last_modified,
    i.submitter_id,
    e.email AS submitter_email,
    COALESCE(ib.bitstreams_count, 0) AS bitstreams_count,
    COALESCE(ib.total_size_bytes, 0) AS total_size_bytes,
    COALESCE(ib.total_size_mb, 0) AS total_size_mb
FROM item i
LEFT JOIN provenance_dates pd
    ON i.uuid = pd.item_id
LEFT JOIN item_handles ih
    ON i.uuid = ih.item_id
LEFT JOIN collection2item c2i
    ON i.uuid = c2i.item_id
LEFT JOIN eperson e
    ON i.submitter_id = e.uuid
LEFT JOIN item_bitstreams ib
    ON i.uuid = ib.item_id
WHERE c2i.collection_id IS NULL
ORDER BY
    pd.deposit_date DESC NULLS LAST,
    i.in_archive DESC,
    i.withdrawn ASC,
    i.discoverable DESC,
    bitstreams_count DESC,
    total_size_bytes DESC,
    i.last_modified DESC NULLS LAST,
    i.uuid;
"""

df_items_without_collection = pd.read_sql_query(
    query_items_without_collection,
    conn
)

df_items_without_collection["reference_month"] = REPORT_MONTH
df_items_without_collection["environment"] = ENVIRONMENT
df_items_without_collection["source"] = "postgres"
df_items_without_collection["metric_definition"] = (
    "items not associated with any collection through collection2item, "
    "including handles, provenance deposit date, non-deleted bitstream count and total size"
)

output_items_without_collection = export_path("db_items_without_collection.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_items_without_collection.to_csv(output_items_without_collection, index=False)

print(f"Item senza collection: {len(df_items_without_collection)}")

print("Distribution by deposit_date:")
display(
    df_items_without_collection["deposit_date"]
    .notna()
    .value_counts()
    .reset_index()
    .rename(columns={"index": "has_deposit_date", "deposit_date": "count"})
)

display(df_items_without_collection.head())

print(f"File saved in: {output_items_without_collection.resolve()}")

Item senza collection: 56
Distribution by deposit_date:


,count,count
0,False,44
1,True,12


,item_id,deposit_date,in_archive,withdrawn,discoverable,last_modified,submitter_id,submitter_email,bitstreams_count,total_size_bytes,total_size_mb,reference_month,environment,source,metric_definition
0,6eae7867-9e95-408a-aa9d-a571fdcb2994,2026-03-04,False,False,True,2026-03-04 16:19:01.141000+00:00,b217ca93-fa47-488d-87cb-1ad4f9155853,valeria.quochi@ilc.cnr.it,2,3573.0,0.00,2026-04,preprod,postgres,items not associated with any collection throu...
1,425da4f3-616c-44aa-b2c7-b523c090301c,2025-06-06,False,False,True,2025-11-18 16:17:35.945000+00:00,3a6d692b-a795-4ca2-851f-d4d2bac81e7d,andreschauta81@gmail.com,1,3680970.0,3.51,2026-04,preprod,postgres,items not associated with any collection throu...
2,9d20175e-ad75-4b04-bee2-a76700a578eb,2021-12-07,False,False,True,2025-11-18 16:04:01.125000+00:00,f3c900f8-e230-4304-8933-f8b384474a36,martina.ali@unicatt.it,0,0.0,0.00,2026-04,preprod,postgres,items not associated with any collection throu...
3,4679ecf6-9127-441e-b9ba-e07c4a7472db,2021-12-07,False,False,True,2025-11-18 16:03:46.448000+00:00,a4bbfffd-f955-4861-9861-7f93a50b1c29,silvia.calvi1@unicatt.it,0,0.0,0.00,2026-04,preprod,postgres,items not associated with any collection throu...
4,b7ce9294-eee3-4b4f-a32d-34c85441f06e,2016-07-06,False,False,True,2025-11-18 14:53:50.315000+00:00,dbdec1f3-d5db-4f5e-8097-debc7e6eec22,f.poli12@studenti.unipi.it,3,3738676.0,3.57,2026-04,preprod,postgres,items not associated with any collection throu...


File saved in: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_items_without_collection.csv


## **Metadata**

### Item with missing metadata

In [7]:
# - item_id
# - handle
# - missing_metadata_field
# - in_archive
# - withdrawn
# - discoverable
# - last_modified
# - reference_month
# - environment
# - source
# - metric_definition

required_metadata_fields = [
    "dc.title",
    "dc.contributor.author",
    "dc.date.issued",
    "dc.language.iso",
    "dc.type"
]

required_metadata_values = ",\n        ".join(
    [f"('{field}')" for field in required_metadata_fields]
)

query_items_missing_metadata = f"""
WITH required_fields(metadata_field) AS (
    VALUES
        {required_metadata_values}
),

archived_items AS (
    SELECT
        i.uuid AS item_id,
        h.handle,
        i.in_archive,
        i.withdrawn,
        i.discoverable,
        i.last_modified
    FROM item i
    LEFT JOIN handle h
        ON h.resource_id = i.uuid
    WHERE i.in_archive = true
),

existing_metadata AS (
    SELECT DISTINCT
        i.uuid AS item_id,
        CASE
            WHEN mfr.qualifier IS NULL OR TRIM(mfr.qualifier) = ''
                THEN msr.short_id || '.' || mfr.element
            ELSE msr.short_id || '.' || mfr.element || '.' || mfr.qualifier
        END AS metadata_field
    FROM item i
    JOIN metadatavalue mv
        ON i.uuid = mv.dspace_object_id
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE i.in_archive = true
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
)

SELECT
    ai.item_id,
    ai.handle,
    rf.metadata_field AS missing_metadata_field,
    ai.in_archive,
    ai.withdrawn,
    ai.discoverable,
    ai.last_modified

FROM archived_items ai
CROSS JOIN required_fields rf
LEFT JOIN existing_metadata em
    ON ai.item_id = em.item_id
   AND rf.metadata_field = em.metadata_field
WHERE em.metadata_field IS NULL
ORDER BY
    ai.last_modified DESC NULLS LAST,
    ai.item_id,
    rf.metadata_field;
"""

df_items_missing_metadata = pd.read_sql_query(
    query_items_missing_metadata,
    conn
)

df_items_missing_metadata["reference_month"] = REPORT_MONTH
df_items_missing_metadata["environment"] = ENVIRONMENT
df_items_missing_metadata["source"] = "postgres"
df_items_missing_metadata["metric_definition"] = (
    "archived items missing one or more required metadata fields or having only NULL/empty values for those fields"
)

output_items_missing_metadata = export_path("db_items_missing_metadata.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_items_missing_metadata.to_csv(output_items_missing_metadata, index=False)

print(f"Missing required metadata found: {len(df_items_missing_metadata)}")
print(f"Items involved: {df_items_missing_metadata['item_id'].nunique()}")
display(df_items_missing_metadata.head())
print(f"File saved to: {output_items_missing_metadata.resolve()}")

Missing required metadata found: 175
Items involved: 175


,item_id,handle,missing_metadata_field,in_archive,withdrawn,discoverable,last_modified,reference_month,environment,source,metric_definition
0,b5a7d156-6700-4ac3-8075-039c43767bd7,20.500.11752/ILC-1039,dc.language.iso,True,False,True,2025-12-11 10:22:08.713000+00:00,2026-04,preprod,postgres,archived items missing one or more required me...
1,ae41c614-4b41-40b2-834f-1f6804de34bd,20.500.11752/ILC-1028,dc.language.iso,True,False,True,2025-11-18 16:16:16.544000+00:00,2026-04,preprod,postgres,archived items missing one or more required me...
2,aea8299d-d19d-4860-a027-6f6f086d3c0a,20.500.11752/ILC-1004,dc.language.iso,True,False,True,2025-11-18 16:15:41.353000+00:00,2026-04,preprod,postgres,archived items missing one or more required me...
3,b253faf6-3b8a-4112-809e-ac90c5f86ab2,20.500.11752/OPEN-924,dc.contributor.author,True,False,True,2025-11-18 16:13:56.867000+00:00,2026-04,preprod,postgres,archived items missing one or more required me...
4,98bf5d0a-3861-41d0-9742-cf93488b91a4,20.500.11752/OPEN-922,dc.contributor.author,True,False,True,2025-11-18 16:13:54.860000+00:00,2026-04,preprod,postgres,archived items missing one or more required me...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_items_missing_metadata.csv


### Items with empty metadata

In [8]:
# - item_id
# - metadata_value_id
# - schema
# - element
# - qualifier
# - metadata_field
# - text_value
# - text_lang
# - place
# - in_archive
# - withdrawn
# - discoverable
# - last_modified
# - reference_month
# - environment
# - source
# - metric_definition

query_items_empty_metadata_values = """
SELECT
    i.uuid AS item_id,
    mv.metadata_value_id,
    msr.short_id AS schema,
    mfr.element,
    mfr.qualifier,

    CASE
        WHEN mfr.qualifier IS NULL OR TRIM(mfr.qualifier) = ''
            THEN msr.short_id || '.' || mfr.element
        ELSE msr.short_id || '.' || mfr.element || '.' || mfr.qualifier
    END AS metadata_field,

    mv.text_value,
    mv.text_lang,
    mv.place,
    i.in_archive,
    i.withdrawn,
    i.discoverable,
    i.last_modified

FROM item i
JOIN metadatavalue mv
    ON i.uuid = mv.dspace_object_id
JOIN metadatafieldregistry mfr
    ON mv.metadata_field_id = mfr.metadata_field_id
JOIN metadataschemaregistry msr
    ON mfr.metadata_schema_id = msr.metadata_schema_id
WHERE i.in_archive = true
  AND (
        mv.text_value IS NULL
        OR TRIM(mv.text_value) = ''
  )
ORDER BY
    i.last_modified DESC NULLS LAST,
    item_id,
    metadata_field,
    mv.place;
"""

df_items_empty_metadata_values = pd.read_sql_query(
    query_items_empty_metadata_values,
    conn
)

df_items_empty_metadata_values["reference_month"] = REPORT_MONTH
df_items_empty_metadata_values["environment"] = ENVIRONMENT
df_items_empty_metadata_values["source"] = "postgres"
df_items_empty_metadata_values["metric_definition"] = (
    "archived items with metadata values where text_value is NULL or empty after trim"
)

output_items_empty_metadata_values = export_path("db_items_empty_metadata_values.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_items_empty_metadata_values.to_csv(output_items_empty_metadata_values, index=False)

print(f"Empty metadata values found: {len(df_items_empty_metadata_values)}")
print(f"Items involved: {df_items_empty_metadata_values['item_id'].nunique()}")
display(df_items_empty_metadata_values.head())
print(f"File saved to: {output_items_empty_metadata_values.resolve()}")

Empty metadata values found: 2


Items involved: 1


,item_id,metadata_value_id,schema,element,qualifier,metadata_field,text_value,text_lang,place,in_archive,withdrawn,discoverable,last_modified,reference_month,environment,source,metric_definition
0,e9e60322-568e-4fe7-a87e-06b60926585f,80221,dc,contributor,advisor,dc.contributor.advisor,,,0,True,False,True,2025-11-18 15:55:33.310000+00:00,2026-04,preprod,postgres,archived items with metadata values where text...
1,e9e60322-568e-4fe7-a87e-06b60926585f,80222,dc,contributor,advisor,dc.contributor.advisor,,,1,True,False,True,2025-11-18 15:55:33.310000+00:00,2026-04,preprod,postgres,archived items with metadata values where text...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_items_empty_metadata_values.csv


### Average metadata by item

In [9]:
# - item_id
# - in_archive
# - withdrawn
# - discoverable
# - last_modified
# - metadata_count
# - non_empty_metadata_count
# - empty_metadata_count
# - reference_month
# - environment
# - source
# - metric_definition

query_metadata_count_per_item = """
WITH metadata_counts AS (
    SELECT
        i.uuid AS item_id,
        i.in_archive,
        i.withdrawn,
        i.discoverable,
        i.last_modified,

        COUNT(mv.metadata_value_id) AS metadata_count,

        COUNT(mv.metadata_value_id) FILTER (
            WHERE mv.text_value IS NOT NULL
              AND TRIM(mv.text_value) <> ''
        ) AS non_empty_metadata_count,

        COUNT(mv.metadata_value_id) FILTER (
            WHERE mv.text_value IS NULL
               OR TRIM(mv.text_value) = ''
        ) AS empty_metadata_count

    FROM item i
    LEFT JOIN metadatavalue mv
        ON i.uuid = mv.dspace_object_id
    WHERE i.in_archive = true
    GROUP BY
        i.uuid,
        i.in_archive,
        i.withdrawn,
        i.discoverable,
        i.last_modified
)
SELECT
    item_id,
    in_archive,
    withdrawn,
    discoverable,
    last_modified,
    metadata_count
FROM metadata_counts
ORDER BY
    metadata_count DESC,
    last_modified DESC NULLS LAST,
    item_id;
"""

df_metadata_count_per_item = pd.read_sql_query(
    query_metadata_count_per_item,
    conn
)

df_metadata_count_per_item["reference_month"] = REPORT_MONTH
df_metadata_count_per_item["environment"] = ENVIRONMENT
df_metadata_count_per_item["source"] = "postgres"
df_metadata_count_per_item["metric_definition"] = (
    "metadata value counts per archived item, including total, non-empty and empty metadata values"
)

output_metadata_count_per_item = export_path("db_metadata_count_per_item.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_metadata_count_per_item.to_csv(output_metadata_count_per_item, index=False)

print(f"Items analyzed: {len(df_metadata_count_per_item)}")
print(f"Average number of metadata values per item: {df_metadata_count_per_item['metadata_count'].mean():.2f}")
display(df_metadata_count_per_item.head())
print(f"File saved to: {output_metadata_count_per_item.resolve()}")

Items analyzed: 969


Average number of metadata values per item: 34.71


,item_id,in_archive,withdrawn,discoverable,last_modified,metadata_count,reference_month,environment,source,metric_definition
0,93b3c336-98c7-413f-bde7-713601992914,True,False,True,2025-12-11 14:29:37.351000+00:00,164,2026-04,preprod,postgres,"metadata value counts per archived item, inclu..."
1,b4452e1f-56a4-4d4b-aa69-4047cf393230,True,False,True,2025-11-18 15:57:49.163000+00:00,139,2026-04,preprod,postgres,"metadata value counts per archived item, inclu..."
2,f41ea177-a164-4f30-8a7c-7626a734f002,True,False,True,2025-11-18 16:15:47.835000+00:00,80,2026-04,preprod,postgres,"metadata value counts per archived item, inclu..."
3,db53a863-a70f-496e-a6a2-6f6f354e09ba,True,False,True,2025-11-18 15:08:54.538000+00:00,77,2026-04,preprod,postgres,"metadata value counts per archived item, inclu..."
4,c33d60f2-8473-4a57-bd10-3e0a24a4d177,True,False,True,2025-12-11 11:39:40.075000+00:00,65,2026-04,preprod,postgres,"metadata value counts per archived item, inclu..."


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_metadata_count_per_item.csv


## **Bitstreams and files**

### Items without bitstreams

In [10]:
# - item_id
# - last_modified
# - submitter_id
# - submitter_email
# - reference_month
# - environment
# - source
# - metric_definition

query_items_without_files = """
SELECT
    i.uuid AS item_id,
    i.last_modified,
    i.submitter_id,
    e.email AS submitter_email
FROM item i
LEFT JOIN eperson e
    ON i.submitter_id = e.uuid
LEFT JOIN item2bundle i2b
    ON i.uuid = i2b.item_id
LEFT JOIN bundle b
    ON i2b.bundle_id = b.uuid
LEFT JOIN bundle2bitstream b2bs
    ON b.uuid = b2bs.bundle_id
LEFT JOIN bitstream bs
    ON b2bs.bitstream_id = bs.uuid
WHERE i.in_archive = true
GROUP BY i.uuid, i.last_modified, i.submitter_id, e.email
HAVING COUNT(bs.uuid) = 0
ORDER BY i.last_modified DESC;
"""

df_items_without_files = pd.read_sql_query(
    query_items_without_files,
    conn
)

# =========================================================
# METADATI EXPORT
# =========================================================
df_items_without_files["reference_month"] = REPORT_MONTH
df_items_without_files["environment"] = ENVIRONMENT
df_items_without_files["source"] = "postgres"
df_items_without_files["metric_definition"] = "archived items without any associated bitstream"

# =========================================================
# EXPORT CSV
# =========================================================
output_items_without_files = export_path("db_items_without_files.csv")
df_items_without_files.to_csv(output_items_without_files, index=False)

# =========================================================
# PRINT
# =========================================================
print(f"Items without files: {len(df_items_without_files)}")
print(df_items_without_files.head())
print(f"File saved to: {output_items_without_files.resolve()}")

Items without files: 434
                                item_id                    last_modified  \
0  ae41c614-4b41-40b2-834f-1f6804de34bd 2025-11-18 16:16:16.544000+00:00   
1  f41ea177-a164-4f30-8a7c-7626a734f002 2025-11-18 16:15:47.835000+00:00   
2  aea8299d-d19d-4860-a027-6f6f086d3c0a 2025-11-18 16:15:41.353000+00:00   
3  134926b8-9608-492e-9aba-745aabb02979 2025-11-18 16:04:12.552000+00:00   
4  981f8908-2b54-42c3-86e8-fcb30be57df1 2025-11-18 16:04:11.620000+00:00   

                           submitter_id             submitter_email  \
0  b217ca93-fa47-488d-87cb-1ad4f9155853   valeria.quochi@ilc.cnr.it   
1  629e80b3-c20b-41b0-9aa8-ea950e765417       olja.perisic@unito.it   
2  42bc516b-f18f-4019-8cd2-d74cbbcb850b  andrea.bellandi@ilc.cnr.it   
3  5c9e77eb-8c67-4a5e-8ba3-69f96d093a39    duccio.piccardi@unisi.it   
4  5c9e77eb-8c67-4a5e-8ba3-69f96d093a39    duccio.piccardi@unisi.it   

  reference_month environment    source  \
0         2026-04     preprod  postgres   
1    

File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_items_without_files.csv


### Total number of bitstreams and MIME type

In [11]:
# - bitstream_id
# - item_id
# - bundle_id
# - bundle_name
# - filename
# - file_extension
# - size_bytes
# - size_mb
# - deleted
# - internal_id
# - checksum
# - checksum_algorithm
# - mimetype
# - bitstream_status
# - reference_month
# - environment
# - source
# - metric_definition

query_bitstreams = """
WITH bitstream_names AS (
    SELECT
        mv.dspace_object_id AS bitstream_id,
        MIN(mv.text_value) AS filename
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
bundle_names AS (
    SELECT
        mv.dspace_object_id AS bundle_id,
        MIN(mv.text_value) AS bundle_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
)
SELECT
    bs.uuid AS bitstream_id,
    i.uuid AS item_id,
    b.uuid AS bundle_id,
    COALESCE(bn_bundle.bundle_name, '[no bundle]') AS bundle_name,
    bn.filename,

    LOWER(
        CASE
            WHEN bn.filename LIKE '%%.%%'
            THEN SPLIT_PART(
                bn.filename,
                '.',
                array_length(string_to_array(bn.filename, '.'), 1)
            )
            ELSE NULL
        END
    ) AS file_extension,

    bs.size_bytes,
    ROUND(COALESCE(bs.size_bytes, 0) / 1024.0 / 1024.0, 2) AS size_mb,
    bs.deleted,
    bs.internal_id,
    bs.checksum,
    bs.checksum_algorithm,
    bfr.mimetype,

    CASE
        WHEN b2bs.bitstream_id IS NULL AND bs.deleted = true
            THEN 'deleted_orphan_bitstream_without_bundle'
        WHEN b2bs.bitstream_id IS NULL
            THEN 'orphan_bitstream_without_bundle'
        WHEN i.uuid IS NULL AND bs.deleted = true
            THEN 'deleted_bitstream_without_item'
        WHEN i.uuid IS NULL
            THEN 'bitstream_without_item'
        WHEN bs.deleted = true
            THEN 'deleted_bitstream_linked_to_item'
        ELSE 'bitstream_linked_to_item'
    END AS bitstream_status

FROM bitstream bs
LEFT JOIN bundle2bitstream b2bs
    ON bs.uuid = b2bs.bitstream_id
LEFT JOIN bundle b
    ON b2bs.bundle_id = b.uuid
LEFT JOIN item2bundle i2b
    ON b.uuid = i2b.bundle_id
LEFT JOIN item i
    ON i2b.item_id = i.uuid
LEFT JOIN bitstream_names bn
    ON bs.uuid = bn.bitstream_id
LEFT JOIN bundle_names bn_bundle
    ON b.uuid = bn_bundle.bundle_id
LEFT JOIN bitstreamformatregistry bfr
    ON bs.bitstream_format_id = bfr.bitstream_format_id
ORDER BY
    bitstream_status,
    i.uuid NULLS LAST,
    b.uuid NULLS LAST,
    bs.uuid;
"""

df_bitstreams = pd.read_sql_query(query_bitstreams, conn)

df_bitstreams["reference_month"] = REPORT_MONTH
df_bitstreams["environment"] = ENVIRONMENT
df_bitstreams["source"] = "postgres"
df_bitstreams["metric_definition"] = (
    "all bitstreams including linked, deleted and orphan bitstreams, "
    "with optional item and bundle links, filename, extension, mimetype and checksum"
)

output_bitstreams = export_path("db_bitstreams_with_filename_extension.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_bitstreams.to_csv(output_bitstreams, index=False)

print(f"Total bitstreams: {len(df_bitstreams)}")

print("\\nDistribution by status:")
display(
    df_bitstreams["bitstream_status"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "bitstream_status", "bitstream_status": "count"})
)

display(df_bitstreams.head())
print(f"File saved to: {output_bitstreams.resolve()}")

Total bitstreams: 2502
\nDistribution by status:


,count,count
0,bitstream_linked_to_item,2296
1,deleted_orphan_bitstream_without_bundle,191
2,orphan_bitstream_without_bundle,13
3,deleted_bitstream_linked_to_item,2


,bitstream_id,item_id,bundle_id,bundle_name,filename,file_extension,size_bytes,size_mb,deleted,internal_id,checksum,checksum_algorithm,mimetype,bitstream_status,reference_month,environment,source,metric_definition
0,d2201e8e-dc2c-4e09-ae0e-04c610f97b13,0086cbb7-2307-48ab-991f-bce666b490fc,03f7f061-3353-4955-9e50-eb36a252148d,ORIGINAL,dlt000565.xml,xml,7148,0.01,False,75272747552975445089121803531664605847,f6addd34893bfacc908f7a18ba38a55a,MD5,text/xml,bitstream_linked_to_item,2026-04,preprod,postgres,"all bitstreams including linked, deleted and o..."
1,e7e82410-803e-49bd-9027-b51b6c5bcacb,0086cbb7-2307-48ab-991f-bce666b490fc,45d38046-d8ab-4220-8b64-2d6d375e41e9,LICENSE,license.txt,txt,174,0.00,False,15828566470898408312078197386509399691,4e59c7d44b709f689e3a2024e5f0b18b,MD5,text/plain,bitstream_linked_to_item,2026-04,preprod,postgres,"all bitstreams including linked, deleted and o..."
2,95acebaf-412d-4cab-9326-52eec88b8e83,01ce7407-a52b-491b-9e58-690a95576da9,47becebb-b2aa-4e4f-a1f9-63060235d4d1,ORIGINAL,dlt000533.xml,xml,71225,0.07,False,144511887360203423359601137920152548733,eeef77940385edbba19dde31277fd3af,MD5,text/xml,bitstream_linked_to_item,2026-04,preprod,postgres,"all bitstreams including linked, deleted and o..."
3,848c53c7-6a7e-40d0-beb4-2682e9bf4de7,01ce7407-a52b-491b-9e58-690a95576da9,be1948c2-09b7-44e3-a7f4-9da427cc44e0,LICENSE,license.txt,txt,174,0.00,False,124074873750854498498859616352627345643,4e59c7d44b709f689e3a2024e5f0b18b,MD5,text/plain,bitstream_linked_to_item,2026-04,preprod,postgres,"all bitstreams including linked, deleted and o..."
4,fb1a8f18-ee63-494c-81a7-5dabfdc20990,02effd82-7425-49a9-99fd-7957539dc74f,1a440f7d-3ee7-4efb-9f4f-4631561e758c,LICENSE,license.txt,txt,174,0.00,False,102245963654053343660304646125522068836,4e59c7d44b709f689e3a2024e5f0b18b,MD5,text/plain,bitstream_linked_to_item,2026-04,preprod,postgres,"all bitstreams including linked, deleted and o..."


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_bitstreams_with_filename_extension.csv


### Total number of bitstreams per item

In [12]:
# - item_id
# - in_archive
# - withdrawn
# - discoverable
# - bitstreams_count
# - total_size_bytes
# - total_size_mb
# - reference_month
# - environment
# - source
# - metric_definition

query_bitstreams_per_item = """
SELECT
    i.uuid AS item_id,
    i.in_archive,
    i.withdrawn,
    i.discoverable,
    COUNT(DISTINCT bs.uuid) AS bitstreams_count,
    COALESCE(SUM(bs.size_bytes), 0) AS total_size_bytes,
    ROUND(COALESCE(SUM(bs.size_bytes), 0) / 1024.0 / 1024.0, 2) AS total_size_mb
FROM item i
LEFT JOIN item2bundle i2b
    ON i.uuid = i2b.item_id
LEFT JOIN bundle b
    ON i2b.bundle_id = b.uuid
LEFT JOIN bundle2bitstream b2bs
    ON b.uuid = b2bs.bundle_id
LEFT JOIN bitstream bs
    ON b2bs.bitstream_id = bs.uuid
   AND (bs.deleted = false OR bs.deleted IS NULL)
GROUP BY
    i.uuid,
    i.in_archive,
    i.withdrawn,
    i.discoverable
ORDER BY
    i.in_archive DESC,
    bitstreams_count DESC,
    total_size_bytes DESC;
"""

df_bitstreams_per_item = pd.read_sql_query(query_bitstreams_per_item, conn)

df_bitstreams_per_item["reference_month"] = REPORT_MONTH
df_bitstreams_per_item["environment"] = ENVIRONMENT
df_bitstreams_per_item["source"] = "postgres"
df_bitstreams_per_item["metric_definition"] = "total non-deleted bitstreams per item including archived and non-archived items"

output_bitstreams_per_item = export_path("db_bitstreams_per_item_all_statuses.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_bitstreams_per_item.to_csv(output_bitstreams_per_item, index=False)

print(f"Items analyzed: {len(df_bitstreams_per_item)}")
display(df_bitstreams_per_item.head())
print(f"File saved to: {output_bitstreams_per_item.resolve()}")

Items analyzed: 1028


,item_id,in_archive,withdrawn,discoverable,bitstreams_count,total_size_bytes,total_size_mb,reference_month,environment,source,metric_definition
0,776f944f-c44b-4230-9960-f13a861593da,True,False,True,540,12188826.0,11.62,2026-04,preprod,postgres,total non-deleted bitstreams per item includin...
1,7b7e2e48-fbb4-452c-bf40-b67577af70f6,True,False,True,343,6682349.0,6.37,2026-04,preprod,postgres,total non-deleted bitstreams per item includin...
2,93b3c336-98c7-413f-bde7-713601992914,True,False,True,121,2238561.0,2.13,2026-04,preprod,postgres,total non-deleted bitstreams per item includin...
3,be9a4bee-0fe2-458f-8d7a-8979d5b3fb77,True,False,True,84,159530.0,0.15,2026-04,preprod,postgres,total non-deleted bitstreams per item includin...
4,89168f1e-52e0-4184-b761-942ccdc7de14,True,False,True,29,542255.0,0.52,2026-04,preprod,postgres,total non-deleted bitstreams per item includin...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_bitstreams_per_item_all_statuses.csv


### Bitstreams without checksum

In [13]:
# - bitstream_id
# - item_id
# - bundle_id
# - bundle_name
# - filename
# - size_bytes
# - size_mb
# - deleted
# - internal_id
# - checksum
# - checksum_algorithm
# - mimetype
# - bitstream_status
# - reference_month
# - environment
# - source
# - metric_definition

query_bitstreams_without_checksum = """
WITH bitstream_names AS (
    SELECT
        mv.dspace_object_id AS bitstream_id,
        MIN(mv.text_value) AS filename
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
bundle_names AS (
    SELECT
        mv.dspace_object_id AS bundle_id,
        MIN(mv.text_value) AS bundle_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
)
SELECT
    bs.uuid AS bitstream_id,
    i.uuid AS item_id,
    b.uuid AS bundle_id,
    COALESCE(bn_bundle.bundle_name, '[no bundle]') AS bundle_name,
    bn.filename,
    bs.size_bytes,
    ROUND(COALESCE(bs.size_bytes, 0) / 1024.0 / 1024.0, 2) AS size_mb,
    bs.deleted,
    bs.internal_id,
    bs.checksum,
    bs.checksum_algorithm,
    bfr.mimetype,

    CASE
        WHEN b2bs.bitstream_id IS NULL AND bs.deleted = true
            THEN 'deleted_orphan_bitstream_without_bundle'
        WHEN b2bs.bitstream_id IS NULL
            THEN 'orphan_bitstream_without_bundle'
        WHEN i.uuid IS NULL AND bs.deleted = true
            THEN 'deleted_bitstream_without_item'
        WHEN i.uuid IS NULL
            THEN 'bitstream_without_item'
        WHEN bs.deleted = true
            THEN 'deleted_bitstream_linked_to_item'
        ELSE 'bitstream_linked_to_item'
    END AS bitstream_status

FROM bitstream bs
LEFT JOIN bundle2bitstream b2bs
    ON bs.uuid = b2bs.bitstream_id
LEFT JOIN bundle b
    ON b2bs.bundle_id = b.uuid
LEFT JOIN item2bundle i2b
    ON b.uuid = i2b.bundle_id
LEFT JOIN item i
    ON i2b.item_id = i.uuid
LEFT JOIN bitstream_names bn
    ON bs.uuid = bn.bitstream_id
LEFT JOIN bundle_names bn_bundle
    ON b.uuid = bn_bundle.bundle_id
LEFT JOIN bitstreamformatregistry bfr
    ON bs.bitstream_format_id = bfr.bitstream_format_id
WHERE
       bs.checksum IS NULL
    OR TRIM(bs.checksum) = ''
ORDER BY
    bitstream_status,
    bs.deleted DESC,
    bs.size_bytes DESC NULLS LAST,
    i.uuid NULLS LAST,
    bs.uuid;
"""

df_bitstreams_without_checksum = pd.read_sql_query(
    query_bitstreams_without_checksum,
    conn
)

df_bitstreams_without_checksum["reference_month"] = REPORT_MONTH
df_bitstreams_without_checksum["environment"] = ENVIRONMENT
df_bitstreams_without_checksum["source"] = "postgres"
df_bitstreams_without_checksum["metric_definition"] = (
    "bitstreams with missing checksum, including linked, deleted and orphan bitstreams"
)

output_bitstreams_without_checksum = export_path("db_bitstreams_without_checksum.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_bitstreams_without_checksum.to_csv(output_bitstreams_without_checksum, index=False)

print(f"Bitstreams without checksum: {len(df_bitstreams_without_checksum)}")

print("Distribution by bitstream status:")
display(
    df_bitstreams_without_checksum["bitstream_status"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "bitstream_status", "bitstream_status": "count"})
)

display(df_bitstreams_without_checksum.head())

print(f"File saved to: {output_bitstreams_without_checksum.resolve()}")

Bitstreams without checksum: 2
Distribution by bitstream status:


,count,count
0,deleted_orphan_bitstream_without_bundle,2


,bitstream_id,item_id,bundle_id,bundle_name,filename,size_bytes,size_mb,deleted,internal_id,checksum,checksum_algorithm,mimetype,bitstream_status,reference_month,environment,source,metric_definition
0,10abe4f2-d95f-4fde-b670-781036297c9e,None,None,[no bundle],None,0,0.0,True,108232242537923732315604715343450606696,None,None,application/octet-stream,deleted_orphan_bitstream_without_bundle,2026-04,preprod,postgres,"bitstreams with missing checksum, including li..."
1,25b3831f-1343-4b61-bc7e-3453c323398d,None,None,[no bundle],None,0,0.0,True,118300303630022810999979858179649290053,None,None,application/octet-stream,deleted_orphan_bitstream_without_bundle,2026-04,preprod,postgres,"bitstreams with missing checksum, including li..."


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_bitstreams_without_checksum.csv


## **Accessibility**

### Items not publicly visible

In [14]:
# - item_uuid
# - item_id
# - item_name
# - in_archive
# - withdrawn
# - discoverable
# - visible_to_anonymous
# - visibility_problem
# - bitstreams_count
# - has_bitstreams
# - total_size_bytes
# - total_size_mb
# - reference_month
# - environment
# - source
# - metric_definition

query_items_anonymous_visibility = """
WITH anonymous_group AS (
    SELECT uuid AS group_uuid
    FROM epersongroup
    WHERE name = 'Anonymous'
),
item_titles AS (
    SELECT
        mv.dspace_object_id AS item_uuid,
        STRING_AGG(mv.text_value, ' | ' ORDER BY mv.place) AS item_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mfr.qualifier IS NULL
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
item_bitstreams AS (
    SELECT
        i.uuid AS item_uuid,
        COUNT(DISTINCT bs.uuid) AS bitstreams_count,
        COALESCE(SUM(bs.size_bytes), 0) AS total_size_bytes,
        ROUND(COALESCE(SUM(bs.size_bytes), 0) / 1024.0 / 1024.0, 2) AS total_size_mb
    FROM item i
    LEFT JOIN item2bundle i2b
        ON i.uuid = i2b.item_id
    LEFT JOIN bundle b
        ON i2b.bundle_id = b.uuid
    LEFT JOIN bundle2bitstream b2bs
        ON b.uuid = b2bs.bundle_id
    LEFT JOIN bitstream bs
        ON b2bs.bitstream_id = bs.uuid
       AND (bs.deleted = false OR bs.deleted IS NULL)
    GROUP BY i.uuid
),
item_visibility AS (
    SELECT
        i.item_id,
        i.uuid AS item_uuid,
        i.in_archive,
        i.withdrawn,
        i.discoverable,

        CASE
            WHEN i.discoverable = true
             AND EXISTS (
                SELECT 1
                FROM resourcepolicy rp
                JOIN anonymous_group ag
                    ON rp.epersongroup_id = ag.group_uuid
                WHERE rp.dspace_object = i.uuid
                  AND rp.action_id = 0
                  AND (
                      rp.start_date IS NULL
                      OR rp.start_date <= CURRENT_DATE
                  )
                  AND (
                      rp.end_date IS NULL
                      OR rp.end_date >= CURRENT_DATE
                  )
            ) THEN true
            ELSE false
        END AS visible_to_anonymous,

        CASE
            WHEN i.discoverable = false
                THEN 'NOT_DISCOVERABLE'
            WHEN NOT EXISTS (
                SELECT 1
                FROM resourcepolicy rp
                JOIN anonymous_group ag
                    ON rp.epersongroup_id = ag.group_uuid
                WHERE rp.dspace_object = i.uuid
                  AND rp.action_id = 0
                  AND (
                      rp.start_date IS NULL
                      OR rp.start_date <= CURRENT_DATE
                  )
                  AND (
                      rp.end_date IS NULL
                      OR rp.end_date >= CURRENT_DATE
                  )
            ) THEN 'NO_ACTIVE_ANONYMOUS_READ_POLICY'
            ELSE 'VISIBLE'
        END AS visibility_problem

    FROM item i
    WHERE i.in_archive = true
      AND i.withdrawn = false
)
SELECT
    iv.item_uuid,
    iv.item_id,
    COALESCE(it.item_name, '[no title]') AS item_name,
    iv.in_archive,
    iv.withdrawn,
    iv.discoverable,
    iv.visible_to_anonymous,
    iv.visibility_problem,
    COALESCE(ib.bitstreams_count, 0) AS bitstreams_count,

    CASE
        WHEN COALESCE(ib.bitstreams_count, 0) > 0 THEN true
        ELSE false
    END AS has_bitstreams,

    COALESCE(ib.total_size_bytes, 0) AS total_size_bytes,
    COALESCE(ib.total_size_mb, 0) AS total_size_mb

FROM item_visibility iv
LEFT JOIN item_titles it
    ON iv.item_uuid = it.item_uuid
LEFT JOIN item_bitstreams ib
    ON iv.item_uuid = ib.item_uuid
WHERE iv.visibility_problem <> 'VISIBLE'
ORDER BY
    iv.visibility_problem,
    it.item_name,
    iv.item_uuid;
"""

df_items_anonymous_visibility = pd.read_sql_query(
    query_items_anonymous_visibility,
    conn
)

df_items_anonymous_visibility["reference_month"] = REPORT_MONTH
df_items_anonymous_visibility["environment"] = ENVIRONMENT
df_items_anonymous_visibility["source"] = "postgres"
df_items_anonymous_visibility["metric_definition"] = (
    "archived and non-withdrawn items not publicly visible to Anonymous; "
    "visibility is based on discoverable=true and active READ policy for Anonymous; "
    "includes item UUIDs, title and non-deleted bitstream count/size"
)

output_items_anonymous_visibility = export_path("db_items_anonymous_visibility.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_items_anonymous_visibility.to_csv(output_items_anonymous_visibility, index=False)

print(f"Archived and non-withdrawn items not publicly visible: {len(df_items_anonymous_visibility)}")

print("Distribution by visibility problem:")
display(
    df_items_anonymous_visibility["visibility_problem"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "visibility_problem", "visibility_problem": "count"})
)

print("Distribution by Anonymous visibility:")
display(
    df_items_anonymous_visibility["visible_to_anonymous"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "visible_to_anonymous", "visible_to_anonymous": "count"})
)

print("Distribution by bitstream availability:")
display(
    df_items_anonymous_visibility["has_bitstreams"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "has_bitstreams", "has_bitstreams": "count"})
)

display(df_items_anonymous_visibility.head())

print(f"File saved to: {output_items_anonymous_visibility.resolve()}")

Archived and non-withdrawn items not publicly visible: 145


Distribution by visibility problem:


,count,count
0,NO_ACTIVE_ANONYMOUS_READ_POLICY,145


Distribution by Anonymous visibility:


,count,count
0,False,145


Distribution by bitstream availability:


,count,count
0,False,129
1,True,16


,item_uuid,item_id,item_name,in_archive,withdrawn,discoverable,visible_to_anonymous,visibility_problem,bitstreams_count,has_bitstreams,total_size_bytes,total_size_mb,reference_month,environment,source,metric_definition
0,90aaa25a-3e6b-43d1-ab39-23306a257a08,None,Actius,True,False,True,False,NO_ACTIVE_ANONYMOUS_READ_POLICY,0,False,0.0,0.0,2026-04,preprod,postgres,archived and non-withdrawn items not publicly ...
1,ebbde94f-aafe-486d-a6e6-6f7b98884242,None,Aesopi Fabulae,True,False,True,False,NO_ACTIVE_ANONYMOUS_READ_POLICY,0,False,0.0,0.0,2026-04,preprod,postgres,archived and non-withdrawn items not publicly ...
2,8863a2be-55f6-40d4-b94d-8bb27cf02a50,None,Alfonseis,True,False,True,False,NO_ACTIVE_ANONYMOUS_READ_POLICY,0,False,0.0,0.0,2026-04,preprod,postgres,archived and non-withdrawn items not publicly ...
3,23ef59b5-f4e7-4b44-8616-6e637710c1eb,None,Andreae Bergomatis Historia,True,False,True,False,NO_ACTIVE_ANONYMOUS_READ_POLICY,0,False,0.0,0.0,2026-04,preprod,postgres,archived and non-withdrawn items not publicly ...
4,a3dc1817-3527-4ecd-8269-54507dace473,None,Annales Ceccanenses,True,False,True,False,NO_ACTIVE_ANONYMOUS_READ_POLICY,0,False,0.0,0.0,2026-04,preprod,postgres,archived and non-withdrawn items not publicly ...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_items_anonymous_visibility.csv


### Bitstreams not publicly visible

In [15]:
# - bitstream_id
# - item_id
# - bundle_id
# - bundle_name
# - filename
# - size_bytes
# - size_mb
# - deleted
# - internal_id
# - checksum
# - checksum_algorithm
# - mimetype
# - item_in_archive
# - item_withdrawn
# - item_discoverable
# - has_active_anonymous_read_policy
# - bitstream_visibility_problem
# - reference_month
# - environment
# - source
# - metric_definition

query_non_public_bitstreams = """
WITH anonymous_group AS (
    SELECT uuid AS group_uuid
    FROM epersongroup
    WHERE name = 'Anonymous'
),
bitstream_names AS (
    SELECT
        mv.dspace_object_id AS bitstream_id,
        MIN(mv.text_value) AS filename
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
bundle_names AS (
    SELECT
        mv.dspace_object_id AS bundle_id,
        MIN(mv.text_value) AS bundle_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
bitstream_details AS (
    SELECT
        bs.uuid AS bitstream_id,
        i.uuid AS item_id,
        b.uuid AS bundle_id,
        COALESCE(bn_bundle.bundle_name, '[no bundle]') AS bundle_name,
        bn.filename,
        bs.size_bytes,
        ROUND(COALESCE(bs.size_bytes, 0) / 1024.0 / 1024.0, 2) AS size_mb,
        bs.deleted,
        bs.internal_id,
        bs.checksum,
        bs.checksum_algorithm,
        bfr.mimetype,
        i.in_archive AS item_in_archive,
        i.withdrawn AS item_withdrawn,
        i.discoverable AS item_discoverable,

        CASE
            WHEN EXISTS (
                SELECT 1
                FROM resourcepolicy rp
                JOIN anonymous_group ag
                    ON rp.epersongroup_id = ag.group_uuid
                WHERE rp.dspace_object = bs.uuid
                  AND rp.action_id = 0
                  AND (
                      rp.start_date IS NULL
                      OR rp.start_date <= CURRENT_DATE
                  )
                  AND (
                      rp.end_date IS NULL
                      OR rp.end_date >= CURRENT_DATE
                  )
            ) THEN true
            ELSE false
        END AS has_active_anonymous_read_policy

    FROM bitstream bs
    LEFT JOIN bundle2bitstream b2bs
        ON bs.uuid = b2bs.bitstream_id
    LEFT JOIN bundle b
        ON b2bs.bundle_id = b.uuid
    LEFT JOIN item2bundle i2b
        ON b.uuid = i2b.bundle_id
    LEFT JOIN item i
        ON i2b.item_id = i.uuid
    LEFT JOIN bitstream_names bn
        ON bs.uuid = bn.bitstream_id
    LEFT JOIN bundle_names bn_bundle
        ON b.uuid = bn_bundle.bundle_id
    LEFT JOIN bitstreamformatregistry bfr
        ON bs.bitstream_format_id = bfr.bitstream_format_id
)
SELECT
    bitstream_id,
    item_id,
    bundle_id,
    bundle_name,
    filename,
    size_bytes,
    size_mb,
    deleted,
    internal_id,
    checksum,
    checksum_algorithm,
    mimetype,
    item_in_archive,
    item_withdrawn,
    item_discoverable,
    has_active_anonymous_read_policy,

    CASE
        WHEN deleted = true
            THEN 'DELETED_BITSTREAM'
        WHEN item_id IS NULL
            THEN 'BITSTREAM_NOT_LINKED_TO_ITEM'
        WHEN item_in_archive = false
            THEN 'ITEM_NOT_IN_ARCHIVE'
        WHEN item_withdrawn = true
            THEN 'ITEM_WITHDRAWN'
        WHEN item_discoverable = false
            THEN 'ITEM_NOT_DISCOVERABLE'
        WHEN has_active_anonymous_read_policy = false
            THEN 'NO_ACTIVE_ANONYMOUS_READ_POLICY_ON_BITSTREAM'
        ELSE 'PUBLIC'
    END AS bitstream_visibility_problem

FROM bitstream_details
WHERE
       deleted = true
    OR item_id IS NULL
    OR item_in_archive = false
    OR item_withdrawn = true
    OR item_discoverable = false
    OR has_active_anonymous_read_policy = false
ORDER BY
    bitstream_visibility_problem,
    item_id NULLS LAST,
    bundle_name,
    filename,
    bitstream_id;
"""

df_non_public_bitstreams = pd.read_sql_query(
    query_non_public_bitstreams,
    conn
)

df_non_public_bitstreams["reference_month"] = REPORT_MONTH
df_non_public_bitstreams["environment"] = ENVIRONMENT
df_non_public_bitstreams["source"] = "postgres"
df_non_public_bitstreams["metric_definition"] = (
    "bitstreams not publicly accessible because they are deleted, orphan, linked to non-public items, "
    "or missing an active READ policy for the Anonymous group"
)

output_non_public_bitstreams = export_path("db_non_public_bitstreams.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_non_public_bitstreams.to_csv(output_non_public_bitstreams, index=False)

print(f"Non-public bitstreams: {len(df_non_public_bitstreams)}")

print("Distribution by visibility problem:")
display(
    df_non_public_bitstreams["bitstream_visibility_problem"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "bitstream_visibility_problem", "bitstream_visibility_problem": "count"})
)

display(df_non_public_bitstreams.head())

print(f"File saved to: {output_non_public_bitstreams.resolve()}")

Non-public bitstreams: 308
Distribution by visibility problem:


,count,count
0,DELETED_BITSTREAM,193
1,NO_ACTIVE_ANONYMOUS_READ_POLICY_ON_BITSTREAM,74
2,ITEM_NOT_IN_ARCHIVE,28
3,BITSTREAM_NOT_LINKED_TO_ITEM,13


,bitstream_id,item_id,bundle_id,bundle_name,filename,size_bytes,size_mb,deleted,internal_id,checksum,...,mimetype,item_in_archive,item_withdrawn,item_discoverable,has_active_anonymous_read_policy,bitstream_visibility_problem,reference_month,environment,source,metric_definition
0,bd122f32-03dc-4b2d-979e-161cfb67850b,None,None,[no bundle],Epilexo-0.1.0.zip,582005,0.56,False,31557465314215602498804916333761504127,54428ebdc58fa0b176756055d2fc15b9,...,application/zip,None,None,None,True,BITSTREAM_NOT_LINKED_TO_ITEM,2026-04,preprod,postgres,bitstreams not publicly accessible because the...
1,3908baca-d846-4e6d-977b-bccfdce10cbe,None,None,[no bundle],bulk-access-control11.log,160,0.00,False,55643282521779568435027239647808807864,5b9d062e9ab5a80cd688e639e754d666,...,application/octet-stream,None,None,None,False,BITSTREAM_NOT_LINKED_TO_ITEM,2026-04,preprod,postgres,bitstreams not publicly accessible because the...
2,d90b2391-637e-4294-bec0-5d9b916d4570,None,None,[no bundle],curate7.log,491,0.00,False,72490303611535495588063238730548903464,bff116bcacbbd45e5cb2c6f6df978838,...,application/octet-stream,None,None,None,False,BITSTREAM_NOT_LINKED_TO_ITEM,2026-04,preprod,postgres,bitstreams not publicly accessible because the...
3,74974948-53ef-47de-84ec-294bf2181eba,None,None,[no bundle],data.json,221,0.00,False,69667965558273175101138172781609447492,c72005c3f084da01050c00ff2ceeb207,...,application/octet-stream,None,None,None,False,BITSTREAM_NOT_LINKED_TO_ITEM,2026-04,preprod,postgres,bitstreams not publicly accessible because the...
4,0770dd21-2299-406f-b768-a2cd0bef22ec,None,None,[no bundle],None,47021,0.04,False,79294109196295998886906684116495965072,19a98b47e4708d1fc9f74304911945a4,...,application/octet-stream,None,None,None,False,BITSTREAM_NOT_LINKED_TO_ITEM,2026-04,preprod,postgres,bitstreams not publicly accessible because the...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_non_public_bitstreams.csv


### Public items with non public bitstreams

In [16]:
# - item_id
# - item_name
# - item_in_archive
# - item_withdrawn
# - item_discoverable
# - item_visible_to_anonymous
# - bitstream_id
# - bundle_id
# - bundle_name
# - filename
# - size_bytes
# - size_mb
# - deleted
# - internal_id
# - checksum
# - checksum_algorithm
# - mimetype
# - bitstream_has_active_anonymous_read_policy
# - bitstream_visibility_problem
# - reference_month
# - environment
# - source
# - metric_definition

query_public_items_with_non_public_bitstreams = """
WITH anonymous_group AS (
    SELECT uuid AS group_uuid
    FROM epersongroup
    WHERE name = 'Anonymous'
),
item_titles AS (
    SELECT
        mv.dspace_object_id AS item_id,
        STRING_AGG(mv.text_value, ' | ' ORDER BY mv.place) AS item_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mfr.qualifier IS NULL
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
bitstream_names AS (
    SELECT
        mv.dspace_object_id AS bitstream_id,
        MIN(mv.text_value) AS filename
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
bundle_names AS (
    SELECT
        mv.dspace_object_id AS bundle_id,
        MIN(mv.text_value) AS bundle_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
public_items AS (
    SELECT
        i.uuid AS item_id,
        i.in_archive,
        i.withdrawn,
        i.discoverable,
        true AS item_visible_to_anonymous
    FROM item i
    WHERE i.in_archive = true
      AND i.withdrawn = false
      AND i.discoverable = true
      AND EXISTS (
          SELECT 1
          FROM resourcepolicy rp
          JOIN anonymous_group ag
              ON rp.epersongroup_id = ag.group_uuid
          WHERE rp.dspace_object = i.uuid
            AND rp.action_id = 0
            AND (
                rp.start_date IS NULL
                OR rp.start_date <= CURRENT_DATE
            )
            AND (
                rp.end_date IS NULL
                OR rp.end_date >= CURRENT_DATE
            )
      )
),
item_bitstreams AS (
    SELECT
        pi.item_id,
        pi.in_archive AS item_in_archive,
        pi.withdrawn AS item_withdrawn,
        pi.discoverable AS item_discoverable,
        pi.item_visible_to_anonymous,

        b.uuid AS bundle_id,
        COALESCE(bn_bundle.bundle_name, '[no bundle]') AS bundle_name,

        bs.uuid AS bitstream_id,
        bn.filename,
        bs.size_bytes,
        ROUND(COALESCE(bs.size_bytes, 0) / 1024.0 / 1024.0, 2) AS size_mb,
        bs.deleted,
        bs.internal_id,
        bs.checksum,
        bs.checksum_algorithm,
        bfr.mimetype,

        CASE
            WHEN EXISTS (
                SELECT 1
                FROM resourcepolicy rp
                JOIN anonymous_group ag
                    ON rp.epersongroup_id = ag.group_uuid
                WHERE rp.dspace_object = bs.uuid
                  AND rp.action_id = 0
                  AND (
                      rp.start_date IS NULL
                      OR rp.start_date <= CURRENT_DATE
                  )
                  AND (
                      rp.end_date IS NULL
                      OR rp.end_date >= CURRENT_DATE
                  )
            ) THEN true
            ELSE false
        END AS bitstream_has_active_anonymous_read_policy

    FROM public_items pi
    JOIN item2bundle i2b
        ON pi.item_id = i2b.item_id
    JOIN bundle b
        ON i2b.bundle_id = b.uuid
    JOIN bundle2bitstream b2bs
        ON b.uuid = b2bs.bundle_id
    JOIN bitstream bs
        ON b2bs.bitstream_id = bs.uuid
    LEFT JOIN bitstream_names bn
        ON bs.uuid = bn.bitstream_id
    LEFT JOIN bundle_names bn_bundle
        ON b.uuid = bn_bundle.bundle_id
    LEFT JOIN bitstreamformatregistry bfr
        ON bs.bitstream_format_id = bfr.bitstream_format_id
)
SELECT
    ib.item_id,
    COALESCE(it.item_name, '[no title]') AS item_name,
    ib.item_in_archive,
    ib.item_withdrawn,
    ib.item_discoverable,
    ib.item_visible_to_anonymous,

    ib.bitstream_id,
    ib.bundle_id,
    ib.bundle_name,
    ib.filename,
    ib.size_bytes,
    ib.size_mb,
    ib.deleted,
    ib.internal_id,
    ib.checksum,
    ib.checksum_algorithm,
    ib.mimetype,
    ib.bitstream_has_active_anonymous_read_policy,

    CASE
        WHEN ib.deleted = true
            THEN 'DELETED_BITSTREAM'
        WHEN ib.bitstream_has_active_anonymous_read_policy = false
            THEN 'NO_ACTIVE_ANONYMOUS_READ_POLICY_ON_BITSTREAM'
        ELSE 'PUBLIC'
    END AS bitstream_visibility_problem

FROM item_bitstreams ib
LEFT JOIN item_titles it
    ON ib.item_id = it.item_id
WHERE
       ib.deleted = true
    OR ib.bitstream_has_active_anonymous_read_policy = false
ORDER BY
    bitstream_visibility_problem,
    item_name,
    bundle_name,
    filename,
    bitstream_id;
"""

df_public_items_with_non_public_bitstreams = pd.read_sql_query(
    query_public_items_with_non_public_bitstreams,
    conn
)

df_public_items_with_non_public_bitstreams["reference_month"] = REPORT_MONTH
df_public_items_with_non_public_bitstreams["environment"] = ENVIRONMENT
df_public_items_with_non_public_bitstreams["source"] = "postgres"
df_public_items_with_non_public_bitstreams["metric_definition"] = (
    "publicly visible archived items that contain deleted bitstreams or bitstreams "
    "without an active READ policy for the Anonymous group"
)

output_public_items_with_non_public_bitstreams = (
    export_path("db_public_items_with_non_public_bitstreams.csv")
)

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_public_items_with_non_public_bitstreams.to_csv(
    output_public_items_with_non_public_bitstreams,
    index=False
)

print(
    "Public items with at least one non-public bitstream: "
    f"{df_public_items_with_non_public_bitstreams['item_id'].nunique()}"
)
print(
    "Non-public bitstreams associated with public items: "
    f"{len(df_public_items_with_non_public_bitstreams)}"
)

print("Distribution by bitstream visibility problem:")
display(
    df_public_items_with_non_public_bitstreams["bitstream_visibility_problem"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "bitstream_visibility_problem", "bitstream_visibility_problem": "count"})
)

display(df_public_items_with_non_public_bitstreams.head())

print(f"File saved to: {output_public_items_with_non_public_bitstreams.resolve()}")

Public items with at least one non-public bitstream: 7
Non-public bitstreams associated with public items: 20
Distribution by bitstream visibility problem:


,count,count
0,NO_ACTIVE_ANONYMOUS_READ_POLICY_ON_BITSTREAM,18
1,DELETED_BITSTREAM,2


,item_id,item_name,item_in_archive,item_withdrawn,item_discoverable,item_visible_to_anonymous,bitstream_id,bundle_id,bundle_name,filename,...,internal_id,checksum,checksum_algorithm,mimetype,bitstream_has_active_anonymous_read_policy,bitstream_visibility_problem,reference_month,environment,source,metric_definition
0,93b3c336-98c7-413f-bde7-713601992914,ItAnt Faliscan Corpus,True,False,True,True,4ea9bec6-d01d-4418-aaa5-d3e1454da98d,9e5e02d2-9793-4bc3-81c8-2cdd036f4f17,ORIGINAL,ItAnt_Faliscan_3_checked.xml,...,88525493042167469863601843059492005108,c75e4dd777b25474bd0fdb3f897baa3f,MD5,text/xml,False,DELETED_BITSTREAM,2026-04,preprod,postgres,publicly visible archived items that contain d...
1,93b3c336-98c7-413f-bde7-713601992914,ItAnt Faliscan Corpus,True,False,True,True,62816166-e45b-46db-9651-48a6ca036edb,9e5e02d2-9793-4bc3-81c8-2cdd036f4f17,ORIGINAL,ItAnt_Faliscan_7_checked.xml,...,127879013987716035454542329615375588242,38ae768c74b857880d633fabc3de9028,MD5,text/xml,False,DELETED_BITSTREAM,2026-04,preprod,postgres,publicly visible archived items that contain d...
2,776f944f-c44b-4230-9960-f13a861593da,Codice Pelavicino,True,False,True,True,bc6aa966-3388-444c-854f-44a86e82fd18,b4820a92-a1ca-401c-bddb-945834f3afd3,ORIGINAL,CCCCLXXXIII_435.xml,...,128132304665963834717157343799276839450,97c76ba803137027f689b3477502260a,MD5,text/xml,False,NO_ACTIVE_ANONYMOUS_READ_POLICY_ON_BITSTREAM,2026-04,preprod,postgres,publicly visible archived items that contain d...
3,776f944f-c44b-4230-9960-f13a861593da,Codice Pelavicino,True,False,True,True,a71ed45b-f3ad-42f1-936a-832e3db467c9,b4820a92-a1ca-401c-bddb-945834f3afd3,ORIGINAL,CCCCLXXXVIII_450.xml,...,96742040867800009597770521020748539142,3f5254ac9e0897904e65f6caa6a20db1,MD5,text/xml,False,NO_ACTIVE_ANONYMOUS_READ_POLICY_ON_BITSTREAM,2026-04,preprod,postgres,publicly visible archived items that contain d...
4,776f944f-c44b-4230-9960-f13a861593da,Codice Pelavicino,True,False,True,True,004ea3ad-a8d4-448f-90f8-1b41301131fe,b4820a92-a1ca-401c-bddb-945834f3afd3,ORIGINAL,CCCLXXXXVI_357.xml,...,35032487370190723109246389344228408681,ce2fdbe7f3420ba65c06e34485e88674,MD5,text/xml,False,NO_ACTIVE_ANONYMOUS_READ_POLICY_ON_BITSTREAM,2026-04,preprod,postgres,publicly visible archived items that contain d...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_public_items_with_non_public_bitstreams.csv


### Items under embargo

In [17]:
# - object_id
# - object_type
# - action_id
# - action_name
# - policy_id
# - policy_start_date
# - policy_end_date
# - epersongroup_id
# - group_name
# - eperson_id
# - eperson_email
# - item_id
# - item_name
# - bundle_id
# - bundle_name
# - bitstream_id
# - filename
# - embargo_status
# - reference_month
# - environment
# - source
# - metric_definition

query_embargoed_objects = """
WITH item_titles AS (
    SELECT
        mv.dspace_object_id AS item_id,
        STRING_AGG(mv.text_value, ' | ' ORDER BY mv.place) AS item_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mfr.qualifier IS NULL
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
bundle_names AS (
    SELECT
        mv.dspace_object_id AS bundle_id,
        MIN(mv.text_value) AS bundle_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
bitstream_names AS (
    SELECT
        mv.dspace_object_id AS bitstream_id,
        MIN(mv.text_value) AS filename
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
bitstream_item_links AS (
    SELECT
        bs.uuid AS bitstream_id,
        b.uuid AS bundle_id,
        i.uuid AS item_id
    FROM bitstream bs
    LEFT JOIN bundle2bitstream b2bs
        ON bs.uuid = b2bs.bitstream_id
    LEFT JOIN bundle b
        ON b2bs.bundle_id = b.uuid
    LEFT JOIN item2bundle i2b
        ON b.uuid = i2b.bundle_id
    LEFT JOIN item i
        ON i2b.item_id = i.uuid
),
bundle_item_links AS (
    SELECT
        b.uuid AS bundle_id,
        i.uuid AS item_id
    FROM bundle b
    LEFT JOIN item2bundle i2b
        ON b.uuid = i2b.bundle_id
    LEFT JOIN item i
        ON i2b.item_id = i.uuid
)
SELECT
    rp.dspace_object AS object_id,

    CASE
        WHEN i.uuid IS NOT NULL THEN 'item'
        WHEN b.uuid IS NOT NULL THEN 'bundle'
        WHEN bs.uuid IS NOT NULL THEN 'bitstream'
        ELSE 'unknown'
    END AS object_type,

    rp.action_id,

    CASE
        WHEN rp.action_id = 0 THEN 'READ'
        WHEN rp.action_id = 1 THEN 'WRITE'
        WHEN rp.action_id = 2 THEN 'DELETE'
        WHEN rp.action_id = 3 THEN 'ADD'
        WHEN rp.action_id = 4 THEN 'REMOVE'
        WHEN rp.action_id = 5 THEN 'WORKFLOW_STEP_1'
        WHEN rp.action_id = 6 THEN 'WORKFLOW_STEP_2'
        WHEN rp.action_id = 7 THEN 'WORKFLOW_STEP_3'
        WHEN rp.action_id = 8 THEN 'ADMIN'
        ELSE 'UNKNOWN'
    END AS action_name,

    rp.policy_id,
    rp.start_date AS policy_start_date,
    rp.end_date AS policy_end_date,

    rp.epersongroup_id,
    eg.name AS group_name,
    rp.eperson_id,
    e.email AS eperson_email,

    COALESCE(i.uuid, bil.item_id, buil.item_id) AS item_id,
    COALESCE(it.item_name, '[no item title]') AS item_name,

    COALESCE(b.uuid, bil.bundle_id) AS bundle_id,
    COALESCE(bn_bundle.bundle_name, '[no bundle]') AS bundle_name,

    bs.uuid AS bitstream_id,
    COALESCE(bn_bitstream.filename, '[no filename]') AS filename,

    CASE
        WHEN rp.start_date IS NOT NULL
         AND rp.start_date > CURRENT_DATE
            THEN 'FUTURE_START_DATE_EMBARGO'
        WHEN rp.end_date IS NOT NULL
         AND rp.end_date < CURRENT_DATE
            THEN 'EXPIRED_POLICY'
        WHEN rp.start_date IS NOT NULL
         AND rp.start_date <= CURRENT_DATE
         AND (
              rp.end_date IS NULL
              OR rp.end_date >= CURRENT_DATE
         )
            THEN 'ACTIVE_DATE_LIMITED_POLICY'
        ELSE 'OTHER_DATE_POLICY'
    END AS embargo_status

FROM resourcepolicy rp
LEFT JOIN item i
    ON rp.dspace_object = i.uuid
LEFT JOIN bundle b
    ON rp.dspace_object = b.uuid
LEFT JOIN bitstream bs
    ON rp.dspace_object = bs.uuid
LEFT JOIN bitstream_item_links bil
    ON bs.uuid = bil.bitstream_id
LEFT JOIN bundle_item_links buil
    ON b.uuid = buil.bundle_id
LEFT JOIN item_titles it
    ON COALESCE(i.uuid, bil.item_id, buil.item_id) = it.item_id
LEFT JOIN bundle_names bn_bundle
    ON COALESCE(b.uuid, bil.bundle_id) = bn_bundle.bundle_id
LEFT JOIN bitstream_names bn_bitstream
    ON bs.uuid = bn_bitstream.bitstream_id
LEFT JOIN epersongroup eg
    ON rp.epersongroup_id = eg.uuid
LEFT JOIN eperson e
    ON rp.eperson_id = e.uuid

WHERE
       rp.start_date IS NOT NULL
    OR rp.end_date IS NOT NULL

ORDER BY
    embargo_status,
    object_type,
    item_name,
    bundle_name,
    filename,
    object_id;
"""

df_embargoed_objects = pd.read_sql_query(
    query_embargoed_objects,
    conn
)

df_embargoed_objects["reference_month"] = REPORT_MONTH
df_embargoed_objects["environment"] = ENVIRONMENT
df_embargoed_objects["source"] = "postgres"
df_embargoed_objects["metric_definition"] = (
    "objects with date-limited resource policies, including future start dates, "
    "active date-limited policies and expired policies"
)

output_embargoed_objects = export_path("db_embargoed_objects.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_embargoed_objects.to_csv(output_embargoed_objects, index=False)

print(f"Objects with temporal policies / embargo: {len(df_embargoed_objects)}")

print("Distribution by embargo / temporal policy status:")
display(
    df_embargoed_objects["embargo_status"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "embargo_status", "embargo_status": "count"})
)

print("Distribution by object type:")
display(
    df_embargoed_objects["object_type"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "object_type", "object_type": "count"})
)

display(df_embargoed_objects.head())

print(f"File saved to: {output_embargoed_objects.resolve()}")

Objects with temporal policies / embargo: 0
Distribution by embargo / temporal policy status:


,count,count


Distribution by object type:


,count,count


,object_id,object_type,action_id,action_name,policy_id,policy_start_date,policy_end_date,epersongroup_id,group_name,eperson_id,...,item_name,bundle_id,bundle_name,bitstream_id,filename,embargo_status,reference_month,environment,source,metric_definition


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_embargoed_objects.csv


## **Community and collections**

### Total communities

In [18]:
# - community_id
# - community_name
# - collections_count
# - reference_month
# - environment
# - source
# - metric_definition

query_total_communities = """
WITH community_titles AS (
    SELECT
        mv.dspace_object_id AS community_id,
        MIN(mv.text_value) AS community_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
community_collections AS (
    SELECT
        c2c.community_id,
        COUNT(DISTINCT c2c.collection_id) AS collections_count
    FROM community2collection c2c
    GROUP BY c2c.community_id
)
SELECT
    c.uuid AS community_id,
    COALESCE(ct.community_name, '[no community title]') AS community_name,
    COALESCE(cc.collections_count, 0) AS collections_count
FROM community c
LEFT JOIN community_titles ct
    ON c.uuid = ct.community_id
LEFT JOIN community_collections cc
    ON c.uuid = cc.community_id
ORDER BY
    collections_count DESC,
    community_name,
    c.uuid;
"""

df_total_communities = pd.read_sql_query(
    query_total_communities,
    conn
)

df_total_communities["reference_month"] = REPORT_MONTH
df_total_communities["environment"] = ENVIRONMENT
df_total_communities["source"] = "postgres"
df_total_communities["metric_definition"] = "total communities in DSpace with community UUID, title and number of directly associated collections"

output_total_communities = export_path("db_total_communities.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_total_communities.to_csv(output_total_communities, index=False)

print(f"Total communities: {len(df_total_communities)}")
display(df_total_communities.head())
print(f"File saved to: {output_total_communities.resolve()}")

Total communities: 2


,community_id,community_name,collections_count,reference_month,environment,source,metric_definition
0,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,9,2026-04,preprod,postgres,total communities in DSpace with community UUI...
1,bdf96608-5c94-471c-9996-63942e832141,ILC,2,2026-04,preprod,postgres,total communities in DSpace with community UUI...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_total_communities.csv


### Total collections

In [19]:
# - collection_id
# - collection_name
# - community_id
# - community_name
# - reference_month
# - environment
# - source
# - metric_definition

query_total_collections = """
WITH collection_titles AS (
    SELECT
        mv.dspace_object_id AS collection_id,
        MIN(mv.text_value) AS collection_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
community_titles AS (
    SELECT
        mv.dspace_object_id AS community_id,
        MIN(mv.text_value) AS community_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
)
SELECT
    c.uuid AS collection_id,
    COALESCE(ct.collection_name, '[no collection title]') AS collection_name,
    c2c.community_id,
    COALESCE(comt.community_name, '[no community title]') AS community_name
FROM collection c
LEFT JOIN collection_titles ct
    ON c.uuid = ct.collection_id
LEFT JOIN community2collection c2c
    ON c.uuid = c2c.collection_id
LEFT JOIN community_titles comt
    ON c2c.community_id = comt.community_id
ORDER BY
    community_name,
    collection_name,
    c.uuid;
"""

df_total_collections = pd.read_sql_query(
    query_total_collections,
    conn
)

df_total_collections["reference_month"] = REPORT_MONTH
df_total_collections["environment"] = ENVIRONMENT
df_total_collections["source"] = "postgres"
df_total_collections["metric_definition"] = (
    "total collections in DSpace with collection UUID, title and reference community"
)

output_total_collections = export_path("db_total_collections.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_total_collections.to_csv(output_total_collections, index=False)

print(f"Total collections: {len(df_total_collections)}")

print("Distribution of collections by community:")
display(
    df_total_collections
    .groupby(["community_id", "community_name"], dropna=False)
    .size()
    .reset_index(name="collections_count")
    .sort_values(["collections_count", "community_name"], ascending=[False, True])
)

display(df_total_collections.head())

print(f"File saved to: {output_total_collections.resolve()}")

Total collections: 11
Distribution of collections by community:


,community_id,community_name,collections_count
1,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,9
0,bdf96608-5c94-471c-9996-63942e832141,ILC,2


,collection_id,collection_name,community_id,community_name,reference_month,environment,source,metric_definition
0,962a7a1c-3ab2-4291-ac4e-13aec990b65a,ILC WebLicht Web Services,bdf96608-5c94-471c-9996-63942e832141,ILC,2026-04,preprod,postgres,total collections in DSpace with collection UU...
1,79c6fbcd-aaac-42b5-aa7d-9413eb906511,ILC4CLARIN : ILC Data & Tools,bdf96608-5c94-471c-9996-63942e832141,ILC,2026-04,preprod,postgres,total collections in DSpace with collection UU...
2,a24bc33f-a053-42a6-93f0-0cf082732a0d,ALIM Documentary Sources,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,2026-04,preprod,postgres,total collections in DSpace with collection UU...
3,514ba599-9a43-4477-b1a6-e3528048458b,ALIM Literary Sources,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,2026-04,preprod,postgres,total collections in DSpace with collection UU...
4,99107ee7-a7af-4354-81a9-e8312ec03823,BIA-Net FONTES,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,2026-04,preprod,postgres,total collections in DSpace with collection UU...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_total_collections.csv


### Bitstreams per collection

In [20]:
# - collection_id
# - collection_name
# - community_id
# - community_name
# - items_count
# - bitstreams_count
# - total_size_bytes
# - total_size_kb
# - total_size_mb
# - total_size_gb
# - reference_month
# - environment
# - source
# - metric_definition

query_total_size_per_collection = """
WITH collection_titles AS (
    SELECT
        mv.dspace_object_id AS collection_id,
        MIN(mv.text_value) AS collection_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
community_titles AS (
    SELECT
        mv.dspace_object_id AS community_id,
        MIN(mv.text_value) AS community_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
collection_storage AS (
    SELECT
        c.uuid AS collection_id,
        COUNT(DISTINCT i.uuid) AS items_count,
        COUNT(DISTINCT bs.uuid) AS bitstreams_count,
        COALESCE(SUM(bs.size_bytes), 0) AS total_size_bytes
    FROM collection c
    LEFT JOIN collection2item c2i
        ON c.uuid = c2i.collection_id
    LEFT JOIN item i
        ON c2i.item_id = i.uuid
       AND i.in_archive = true
       AND i.withdrawn = false
    LEFT JOIN item2bundle i2b
        ON i.uuid = i2b.item_id
    LEFT JOIN bundle b
        ON i2b.bundle_id = b.uuid
    LEFT JOIN bundle2bitstream b2bs
        ON b.uuid = b2bs.bundle_id
    LEFT JOIN bitstream bs
        ON b2bs.bitstream_id = bs.uuid
       AND (bs.deleted = false OR bs.deleted IS NULL)
    GROUP BY c.uuid
)
SELECT
    c.uuid AS collection_id,
    COALESCE(ct.collection_name, '[no collection title]') AS collection_name,
    c2c.community_id,
    COALESCE(comt.community_name, '[no community title]') AS community_name,

    COALESCE(cs.items_count, 0) AS items_count,
    COALESCE(cs.bitstreams_count, 0) AS bitstreams_count,
    COALESCE(cs.total_size_bytes, 0) AS total_size_bytes,
    ROUND(COALESCE(cs.total_size_bytes, 0) / 1024.0, 2) AS total_size_kb,
    ROUND(COALESCE(cs.total_size_bytes, 0) / 1024.0 / 1024.0, 2) AS total_size_mb,
    ROUND(COALESCE(cs.total_size_bytes, 0) / 1024.0 / 1024.0 / 1024.0, 4) AS total_size_gb

FROM collection c
LEFT JOIN collection_titles ct
    ON c.uuid = ct.collection_id
LEFT JOIN community2collection c2c
    ON c.uuid = c2c.collection_id
LEFT JOIN community_titles comt
    ON c2c.community_id = comt.community_id
LEFT JOIN collection_storage cs
    ON c.uuid = cs.collection_id

ORDER BY
    total_size_bytes DESC,
    bitstreams_count DESC,
    items_count DESC,
    community_name,
    collection_name;
"""

df_total_size_per_collection = pd.read_sql_query(
    query_total_size_per_collection,
    conn
)

df_total_size_per_collection["reference_month"] = REPORT_MONTH
df_total_size_per_collection["environment"] = ENVIRONMENT
df_total_size_per_collection["source"] = "postgres"
df_total_size_per_collection["metric_definition"] = (
    "total non-deleted bitstream size per collection, considering archived and non-withdrawn items"
)

output_total_size_per_collection = export_path("db_total_size_per_collection.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_total_size_per_collection.to_csv(output_total_size_per_collection, index=False)

print(f"Collections analyzed: {len(df_total_size_per_collection)}")
print(f"Overall total size in GB: {df_total_size_per_collection['total_size_gb'].sum():.4f}")

display(df_total_size_per_collection.head())

print(f"File saved to: {output_total_size_per_collection.resolve()}")

Collections analyzed: 11
Overall total size in GB: 16.8608


,collection_id,collection_name,community_id,community_name,items_count,bitstreams_count,total_size_bytes,total_size_kb,total_size_mb,total_size_gb,reference_month,environment,source,metric_definition
0,831e86ef-dd49-4926-ab6d-ce25ccc8a7ff,CIRCSE,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,11,20,9.945157e+09,9712067.78,9484.44,9.2621,2026-04,preprod,postgres,total non-deleted bitstream size per collectio...
1,57c1de63-635d-41ef-8e64-119865cb8f80,ILC4CLARIN : OPEN Data & Tools,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,48,1188,7.489313e+09,7313782.38,7142.37,6.9750,2026-04,preprod,postgres,total non-deleted bitstream size per collectio...
2,79c6fbcd-aaac-42b5-aa7d-9413eb906511,ILC4CLARIN : ILC Data & Tools,bdf96608-5c94-471c-9996-63942e832141,ILC,78,205,3.261329e+08,318489.12,311.02,0.3037,2026-04,preprod,postgres,total non-deleted bitstream size per collectio...
3,97bbce33-6bba-4e9a-af5d-d51b2fbc0c36,REALITER - OTPL,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,21,22,1.112569e+08,108649.29,106.10,0.1036,2026-04,preprod,postgres,total non-deleted bitstream size per collectio...
4,1c610d27-1c9d-485d-b27c-3aadccaffd4c,Corpus KIParla Collection,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,8,40,1.076554e+08,105132.19,102.67,0.1003,2026-04,preprod,postgres,total non-deleted bitstream size per collectio...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_total_size_per_collection.csv


## **Workflow and workspace**

### Item in workspace

In [21]:
# - workspace_item_id
# - item_id
# - collection_id
# - stage_reached
# - page_reached
# - multiple_titles
# - published_before
# - multiple_files
# - submitter_id
# - submitter_email
# - last_modified
# - in_archive
# - withdrawn
# - discoverable
# - stage_label
# - reference_month
# - environment
# - source
# - metric_definition

query_workspace_items_details = """
SELECT
    wi.workspace_item_id,
    wi.item_id,
    wi.collection_id,
    wi.stage_reached,
    wi.page_reached,
    wi.multiple_titles,
    wi.published_before,
    wi.multiple_files,
    i.submitter_id,
    e.email AS submitter_email,
    i.last_modified,
    i.in_archive,
    i.withdrawn,
    i.discoverable
FROM workspaceitem wi
LEFT JOIN item i
    ON wi.item_id = i.uuid
LEFT JOIN eperson e
    ON i.submitter_id = e.uuid
ORDER BY i.last_modified DESC NULLS LAST;
"""

df_workspace_items_details = pd.read_sql_query(
    query_workspace_items_details,
    conn
)


stage_mapping = {
    -1: "not_initialized",
    0: "collection",
    1: "identifiers",
    2: "traditionalpageone",
    3: "traditionalpagetwo",
    4: "upload",
    5: "license",
    6: "clarin-license",
    7: "specialFields"
}

df_workspace_items_details.insert(
    df_workspace_items_details.columns.get_loc("stage_reached") + 1,
    "stage_label",
    df_workspace_items_details["stage_reached"].map(stage_mapping).fillna("unknown")
)


df_workspace_items_details["reference_month"] = REPORT_MONTH
df_workspace_items_details["environment"] = ENVIRONMENT
df_workspace_items_details["source"] = "postgres"
df_workspace_items_details["metric_definition"] = "workspace item details with stage label mapping"

output_workspace_items_details = export_path("db_workspace_items_details.csv")
df_workspace_items_details.to_csv(output_workspace_items_details, index=False)

print("Workspace item details:")
display(df_workspace_items_details.head())
print(f"File saved to: {output_workspace_items_details.resolve()}")

Workspace item details:


,workspace_item_id,item_id,collection_id,stage_reached,stage_label,page_reached,multiple_titles,published_before,multiple_files,submitter_id,submitter_email,last_modified,in_archive,withdrawn,discoverable,reference_month,environment,source,metric_definition
0,2345,e26ab61c-2fe9-4eb9-8532-566ac66bc269,57c1de63-635d-41ef-8e64-119865cb8f80,-1,not_initialized,-1,False,False,False,29eaad3f-dbff-4a98-b009-210e322113a4,a.boccioli@student.unisi.it,2026-04-26 07:45:30.966000+00:00,False,False,True,2026-04,preprod,postgres,workspace item details with stage label mapping
1,2344,8fc1ca77-4c03-499a-931d-dc6c20cc0be9,1c610d27-1c9d-485d-b27c-3aadccaffd4c,-1,not_initialized,-1,False,False,False,7aa6bb7b-678e-46d6-aaa8-6a216786aed2,martina.rossi@grupposcai.it,2026-04-24 15:29:42.932000+00:00,False,False,True,2026-04,preprod,postgres,workspace item details with stage label mapping
2,2342,c8c33670-1b7e-4b9c-bb9b-04c721764f14,57c1de63-635d-41ef-8e64-119865cb8f80,-1,not_initialized,-1,False,False,False,09b44a74-775a-4094-b9a9-62df27717f3f,irene.buttazzi@upf.edu,2026-04-23 12:28:31.020000+00:00,False,False,True,2026-04,preprod,postgres,workspace item details with stage label mapping
3,2341,16b495fc-78da-47fb-9690-f32fcf2d35c7,57c1de63-635d-41ef-8e64-119865cb8f80,-1,not_initialized,-1,False,False,False,7aa6bb7b-678e-46d6-aaa8-6a216786aed2,martina.rossi@grupposcai.it,2026-04-22 08:38:49.325000+00:00,False,False,True,2026-04,preprod,postgres,workspace item details with stage label mapping
4,2337,4f53bd9c-534d-4109-964f-4de361f71255,79c6fbcd-aaac-42b5-aa7d-9413eb906511,-1,not_initialized,-1,False,False,False,29c07a05-b333-4496-891f-09557a19ca4a,francesca.frontini@ilc.cnr.it,2026-04-15 15:17:32.780000+00:00,False,False,True,2026-04,preprod,postgres,workspace item details with stage label mapping


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_workspace_items_details.csv


### Items under review

In [22]:
# - workflowitem_id
# - item_id
# - collection_id
# - multiple_titles
# - published_before
# - multiple_files
# - submitter_id
# - submitter_email
# - last_modified
# - in_archive
# - withdrawn
# - discoverable
# - reference_month
# - environment
# - source
# - metric_definition


query_workflow_items_details = """
SELECT
    cwi.workflowitem_id,
    cwi.item_id,
    cwi.collection_id,
    cwi.multiple_titles,
    cwi.published_before,
    cwi.multiple_files,
    i.submitter_id,
    e.email AS submitter_email,
    i.last_modified,
    i.in_archive,
    i.withdrawn,
    i.discoverable
FROM cwf_workflowitem cwi
LEFT JOIN item i
    ON cwi.item_id = i.uuid
LEFT JOIN eperson e
    ON i.submitter_id = e.uuid
ORDER BY i.last_modified DESC NULLS LAST;
"""

df_workflow_items_details = pd.read_sql_query(query_workflow_items_details, conn)

df_workflow_items_details["reference_month"] = REPORT_MONTH
df_workflow_items_details["environment"] = ENVIRONMENT
df_workflow_items_details["source"] = "postgres"
df_workflow_items_details["metric_definition"] = "current workflow item details with submitter when available"

output_workflow_items_details = export_path("db_workflow_items_details.csv")
df_workflow_items_details.to_csv(output_workflow_items_details, index=False)

print("Workflow / review item details:")
print(df_workflow_items_details.head())
print(f"File saved to: {output_workflow_items_details.resolve()}")

Workflow / review item details:
Empty DataFrame
Columns: [workflowitem_id, item_id, collection_id, multiple_titles, published_before, multiple_files, submitter_id, submitter_email, last_modified, in_archive, withdrawn, discoverable, reference_month, environment, source, metric_definition]
Index: []
File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_workflow_items_details.csv


### Workspace items per users

In [23]:
# - submitter_id
# - submitter_email
# - workspace_items_count
# - workspace_item_ids
# - item_ids
# - collection_ids
# - stage_values
# - stage_labels
# - last_modified_max
# - items_with_multiple_titles
# - items_published_before
# - items_with_multiple_files
# - reference_month
# - environment
# - source
# - metric_definition

query_workspace_items_per_user = """
WITH workspace_details AS (
    SELECT
        wi.workspace_item_id,
        wi.item_id,
        wi.collection_id,
        wi.stage_reached,

        CASE
            WHEN wi.stage_reached = -1 THEN 'not_initialized'
            WHEN wi.stage_reached = 0 THEN 'collection'
            WHEN wi.stage_reached = 1 THEN 'identifiers'
            WHEN wi.stage_reached = 2 THEN 'traditionalpageone'
            WHEN wi.stage_reached = 3 THEN 'traditionalpagetwo'
            WHEN wi.stage_reached = 4 THEN 'upload'
            WHEN wi.stage_reached = 5 THEN 'license'
            WHEN wi.stage_reached = 6 THEN 'clarin-license'
            WHEN wi.stage_reached = 7 THEN 'specialFields'
            ELSE 'unknown'
        END AS stage_label,

        wi.multiple_titles,
        wi.published_before,
        wi.multiple_files,

        i.submitter_id,
        e.email AS submitter_email,
        i.last_modified

    FROM workspaceitem wi
    LEFT JOIN item i
        ON wi.item_id = i.uuid
    LEFT JOIN eperson e
        ON i.submitter_id = e.uuid
)
SELECT
    submitter_id,
    COALESCE(submitter_email, '[no email]') AS submitter_email,

    COUNT(DISTINCT workspace_item_id) AS workspace_items_count,

    STRING_AGG(
        DISTINCT workspace_item_id::text,
        '; '
        ORDER BY workspace_item_id::text
    ) AS workspace_item_ids,

    STRING_AGG(
        DISTINCT item_id::text,
        '; '
        ORDER BY item_id::text
    ) AS item_ids,

    STRING_AGG(
        DISTINCT collection_id::text,
        '; '
        ORDER BY collection_id::text
    ) AS collection_ids,

    STRING_AGG(
        DISTINCT stage_reached::text,
        '; '
        ORDER BY stage_reached::text
    ) AS stage_values,

    STRING_AGG(
        DISTINCT stage_label,
        '; '
        ORDER BY stage_label
    ) AS stage_labels,

    MAX(last_modified) AS last_modified_max,

    COUNT(*) FILTER (
        WHERE multiple_titles = true
    ) AS items_with_multiple_titles,

    COUNT(*) FILTER (
        WHERE published_before = true
    ) AS items_published_before,

    COUNT(*) FILTER (
        WHERE multiple_files = true
    ) AS items_with_multiple_files

FROM workspace_details
GROUP BY
    submitter_id,
    submitter_email
ORDER BY
    workspace_items_count DESC,
    last_modified_max DESC NULLS LAST,
    submitter_email;
"""

df_workspace_items_per_user = pd.read_sql_query(
    query_workspace_items_per_user,
    conn
)

df_workspace_items_per_user["reference_month"] = REPORT_MONTH
df_workspace_items_per_user["environment"] = ENVIRONMENT
df_workspace_items_per_user["source"] = "postgres"
df_workspace_items_per_user["metric_definition"] = (
    "workspace items grouped by submitter, including stage values, stage labels, "
    "collections and workspace flags"
)

output_workspace_items_per_user = export_path("db_workspace_items_per_user.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_workspace_items_per_user.to_csv(output_workspace_items_per_user, index=False)

print(f"Users with workspace items: {len(df_workspace_items_per_user)}")
print(f"Total workspace items: {df_workspace_items_per_user['workspace_items_count'].sum()}")

display(df_workspace_items_per_user.head())

print(f"File saved to: {output_workspace_items_per_user.resolve()}")

Users with workspace items: 30
Total workspace items: 56


,submitter_id,submitter_email,workspace_items_count,workspace_item_ids,item_ids,collection_ids,stage_values,stage_labels,last_modified_max,items_with_multiple_titles,items_published_before,items_with_multiple_files,reference_month,environment,source,metric_definition
0,b217ca93-fa47-488d-87cb-1ad4f9155853,valeria.quochi@ilc.cnr.it,8,1232; 1239; 1240; 2276; 2311; 2318; 2326; 2334,078413e7-2c52-4a2e-8f50-c3abdbdafddd; 1cd4d72f...,514ba599-9a43-4477-b1a6-e3528048458b; 57c1de63...,-1; 2; 3; 5,license; not_initialized; traditionalpageone; ...,2026-04-14 09:50:48.521000+00:00,0,3,0,2026-04,preprod,postgres,"workspace items grouped by submitter, includin..."
1,29eaad3f-dbff-4a98-b009-210e322113a4,a.boccioli@student.unisi.it,5,2312; 2313; 2314; 2322; 2345,334d68e7-5798-4163-8585-b447543d9470; 9ec0585f...,57c1de63-635d-41ef-8e64-119865cb8f80; a24bc33f...,-1,not_initialized,2026-04-26 07:45:30.966000+00:00,0,0,0,2026-04,preprod,postgres,"workspace items grouped by submitter, includin..."
2,b346c704-a402-4a57-8800-1ac31d559d16,martin.critelli@ilc.cnr.it,4,1223; 1229; 1233; 1242,10ecab96-e4ec-4e3e-af7f-e300afde1b87; 2abe38bc...,57c1de63-635d-41ef-8e64-119865cb8f80; 79c6fbcd...,1; 5,identifiers; license,2025-11-18 16:15:33.153000+00:00,0,4,0,2026-04,preprod,postgres,"workspace items grouped by submitter, includin..."
3,29c07a05-b333-4496-891f-09557a19ca4a,francesca.frontini@ilc.cnr.it,3,1220; 2335; 2337,4f53bd9c-534d-4109-964f-4de361f71255; 94c37c91...,79c6fbcd-aaac-42b5-aa7d-9413eb906511,-1; 4,not_initialized; upload,2026-04-15 15:17:32.780000+00:00,0,1,0,2026-04,preprod,postgres,"workspace items grouped by submitter, includin..."
4,d6de8f7d-04a4-4c83-a540-0fb3b1ecc038,rachele.sprugnoli@unicatt.it,3,1218; 2328; 2329,1a2f0f21-4692-4a85-a319-58c8c4266e4d; 5469c181...,57c1de63-635d-41ef-8e64-119865cb8f80; 831e86ef...,-1; 1,identifiers; not_initialized,2026-03-30 13:35:38.366000+00:00,0,1,0,2026-04,preprod,postgres,"workspace items grouped by submitter, includin..."


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_workspace_items_per_user.csv


### Workspace item per collection

In [24]:
# Colonne estratte:
# - collection_id
# - collection_name
# - workspace_items_count
# - workspace_item_ids
# - item_ids
# - submitter_ids
# - submitter_emails
# - stage_values
# - stage_labels
# - last_modified_max
# - items_with_multiple_titles
# - items_published_before
# - items_with_multiple_files
# - reference_month
# - environment
# - source
# - metric_definition

query_workspace_items_per_collection = """
WITH collection_titles AS (
    SELECT
        mv.dspace_object_id AS collection_id,
        MIN(mv.text_value) AS collection_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
workspace_details AS (
    SELECT
        wi.workspace_item_id,
        wi.item_id,
        wi.collection_id,
        wi.stage_reached,

        CASE
            WHEN wi.stage_reached = -1 THEN 'not_initialized'
            WHEN wi.stage_reached = 0 THEN 'collection'
            WHEN wi.stage_reached = 1 THEN 'identifiers'
            WHEN wi.stage_reached = 2 THEN 'traditionalpageone'
            WHEN wi.stage_reached = 3 THEN 'traditionalpagetwo'
            WHEN wi.stage_reached = 4 THEN 'upload'
            WHEN wi.stage_reached = 5 THEN 'license'
            WHEN wi.stage_reached = 6 THEN 'clarin-license'
            WHEN wi.stage_reached = 7 THEN 'specialFields'
            ELSE 'unknown'
        END AS stage_label,

        wi.multiple_titles,
        wi.published_before,
        wi.multiple_files,

        i.submitter_id,
        e.email AS submitter_email,
        i.last_modified

    FROM workspaceitem wi
    LEFT JOIN item i
        ON wi.item_id = i.uuid
    LEFT JOIN eperson e
        ON i.submitter_id = e.uuid
)
SELECT
    wd.collection_id,
    COALESCE(ct.collection_name, '[no collection title]') AS collection_name,

    COUNT(DISTINCT wd.workspace_item_id) AS workspace_items_count,

    STRING_AGG(
        DISTINCT wd.workspace_item_id::text,
        '; '
        ORDER BY wd.workspace_item_id::text
    ) AS workspace_item_ids,

    STRING_AGG(
        DISTINCT wd.item_id::text,
        '; '
        ORDER BY wd.item_id::text
    ) AS item_ids,

    STRING_AGG(
        DISTINCT wd.submitter_id::text,
        '; '
        ORDER BY wd.submitter_id::text
    ) AS submitter_ids,

    STRING_AGG(
        DISTINCT COALESCE(wd.submitter_email, '[no email]'),
        '; '
        ORDER BY COALESCE(wd.submitter_email, '[no email]')
    ) AS submitter_emails,

    STRING_AGG(
        DISTINCT wd.stage_reached::text,
        '; '
        ORDER BY wd.stage_reached::text
    ) AS stage_values,

    STRING_AGG(
        DISTINCT wd.stage_label,
        '; '
        ORDER BY wd.stage_label
    ) AS stage_labels,

    MAX(wd.last_modified) AS last_modified_max,

    COUNT(*) FILTER (
        WHERE wd.multiple_titles = true
    ) AS items_with_multiple_titles,

    COUNT(*) FILTER (
        WHERE wd.published_before = true
    ) AS items_published_before,

    COUNT(*) FILTER (
        WHERE wd.multiple_files = true
    ) AS items_with_multiple_files

FROM workspace_details wd
LEFT JOIN collection_titles ct
    ON wd.collection_id = ct.collection_id
GROUP BY
    wd.collection_id,
    ct.collection_name
ORDER BY
    workspace_items_count DESC,
    last_modified_max DESC NULLS LAST,
    collection_name;
"""

df_workspace_items_per_collection = pd.read_sql_query(
    query_workspace_items_per_collection,
    conn
)

df_workspace_items_per_collection["reference_month"] = REPORT_MONTH
df_workspace_items_per_collection["environment"] = ENVIRONMENT
df_workspace_items_per_collection["source"] = "postgres"
df_workspace_items_per_collection["metric_definition"] = (
    "workspace items grouped by collection, including submitters, stage values, "
    "stage labels and workspace flags"
)

output_workspace_items_per_collection = export_path("db_workspace_items_per_collection.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_workspace_items_per_collection.to_csv(output_workspace_items_per_collection, index=False)

print(f"Collections with workspace items: {len(df_workspace_items_per_collection)}")

if len(df_workspace_items_per_collection) > 0:
    print(f"Total workspace items: {df_workspace_items_per_collection['workspace_items_count'].sum()}")

display(df_workspace_items_per_collection.head())

print(f"File saved to: {output_workspace_items_per_collection.resolve()}")

Collections with workspace items: 7
Total workspace items: 56


,collection_id,collection_name,workspace_items_count,workspace_item_ids,item_ids,submitter_ids,submitter_emails,stage_values,stage_labels,last_modified_max,items_with_multiple_titles,items_published_before,items_with_multiple_files,reference_month,environment,source,metric_definition
0,57c1de63-635d-41ef-8e64-119865cb8f80,ILC4CLARIN : OPEN Data & Tools,28,1213; 1214; 1215; 1222; 1224; 1226; 1229; 1230...,0d691118-7bb2-46ad-b723-b3a07c8e025a; 0e36042b...,016fe80e-534f-4b7e-90a1-027dae2584d0; 09b44a74...,a.boccioli@student.unisi.it ; andreschauta81@g...,-1; 1; 2; 3; 5,identifiers; license; not_initialized; traditi...,2026-04-26 07:45:30.966000+00:00,0,17,0,2026-04,preprod,postgres,"workspace items grouped by collection, includi..."
1,79c6fbcd-aaac-42b5-aa7d-9413eb906511,ILC4CLARIN : ILC Data & Tools,16,1217; 1219; 1220; 1221; 1223; 1225; 1232; 1239...,078413e7-2c52-4a2e-8f50-c3abdbdafddd; 10ecab96...,29c07a05-b333-4496-891f-09557a19ca4a; 8a0ea583...,davide.saponaro1978@gmail.com; f.poli12@studen...,-1; 1; 2; 4; 5,identifiers; license; not_initialized; traditi...,2026-04-15 15:17:32.780000+00:00,0,9,0,2026-04,preprod,postgres,"workspace items grouped by collection, includi..."
2,1c610d27-1c9d-485d-b27c-3aadccaffd4c,Corpus KIParla Collection,6,2225; 2264; 2268; 2269; 2270; 2344,54064453-4658-4f08-9078-94fdc8d9b02a; 59e62c1c...,7aa6bb7b-678e-46d6-aaa8-6a216786aed2; ad229930...,caterina.mauri@unibo.it; ellepannitto@gmail.co...,-1,not_initialized,2026-04-24 15:29:42.932000+00:00,0,0,0,2026-04,preprod,postgres,"workspace items grouped by collection, includi..."
3,a24bc33f-a053-42a6-93f0-0cf082732a0d,ALIM Documentary Sources,2,2312; 2313,334d68e7-5798-4163-8585-b447543d9470; 9ec0585f...,29eaad3f-dbff-4a98-b009-210e322113a4,a.boccioli@student.unisi.it,-1,not_initialized,2026-02-24 17:10:04.277000+00:00,0,0,0,2026-04,preprod,postgres,"workspace items grouped by collection, includi..."
4,97bbce33-6bba-4e9a-af5d-d51b2fbc0c36,REALITER - OTPL,2,1227; 1228,4679ecf6-9127-441e-b9ba-e07c4a7472db; 9d20175e...,a4bbfffd-f955-4861-9861-7f93a50b1c29; f3c900f8...,martina.ali@unicatt.it; silvia.calvi1@unicatt.it,1,identifiers,2025-11-18 16:04:01.125000+00:00,0,2,0,2026-04,preprod,postgres,"workspace items grouped by collection, includi..."


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_workspace_items_per_collection.csv


### Submissions that have been pending for more than 30 days

In [25]:
# - submission_source
# - submission_id
# - item_id
# - collection_id
# - collection_name
# - submitter_id
# - submitter_email
# - stage_reached
# - stage_label
# - last_modified
# - days_since_last_modified
# - in_archive
# - withdrawn
# - discoverable
# - multiple_titles
# - published_before
# - multiple_files
# - reference_month
# - environment
# - source
# - metric_definition

query_stalled_submissions_30_days = """
WITH collection_titles AS (
    SELECT
        mv.dspace_object_id AS collection_id,
        MIN(mv.text_value) AS collection_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
workspace_submissions AS (
    SELECT
        'workspace' AS submission_source,
        wi.workspace_item_id::text AS submission_id,
        wi.item_id,
        wi.collection_id,
        i.submitter_id,
        e.email AS submitter_email,
        wi.stage_reached,

        CASE
            WHEN wi.stage_reached = -1 THEN 'not_initialized'
            WHEN wi.stage_reached = 0 THEN 'collection'
            WHEN wi.stage_reached = 1 THEN 'identifiers'
            WHEN wi.stage_reached = 2 THEN 'traditionalpageone'
            WHEN wi.stage_reached = 3 THEN 'traditionalpagetwo'
            WHEN wi.stage_reached = 4 THEN 'upload'
            WHEN wi.stage_reached = 5 THEN 'license'
            WHEN wi.stage_reached = 6 THEN 'clarin-license'
            WHEN wi.stage_reached = 7 THEN 'specialFields'
            ELSE 'unknown'
        END AS stage_label,

        i.last_modified,
        i.in_archive,
        i.withdrawn,
        i.discoverable,
        wi.multiple_titles,
        wi.published_before,
        wi.multiple_files

    FROM workspaceitem wi
    LEFT JOIN item i
        ON wi.item_id = i.uuid
    LEFT JOIN eperson e
        ON i.submitter_id = e.uuid
),
workflow_submissions AS (
    SELECT
        'workflow' AS submission_source,
        cwi.workflowitem_id::text AS submission_id,
        cwi.item_id,
        cwi.collection_id,
        i.submitter_id,
        e.email AS submitter_email,
        NULL::integer AS stage_reached,
        'workflow_waiting_for_approval' AS stage_label,
        i.last_modified,
        i.in_archive,
        i.withdrawn,
        i.discoverable,
        cwi.multiple_titles,
        cwi.published_before,
        cwi.multiple_files

    FROM cwf_workflowitem cwi
    LEFT JOIN item i
        ON cwi.item_id = i.uuid
    LEFT JOIN eperson e
        ON i.submitter_id = e.uuid
),
all_open_submissions AS (
    SELECT * FROM workspace_submissions
    UNION ALL
    SELECT * FROM workflow_submissions
)
SELECT
    aos.submission_source,
    aos.submission_id,
    aos.item_id,
    aos.collection_id,
    COALESCE(ct.collection_name, '[no collection title]') AS collection_name,
    aos.submitter_id,
    COALESCE(aos.submitter_email, '[no email]') AS submitter_email,
    aos.stage_reached,
    aos.stage_label,
    aos.last_modified,

    CASE
        WHEN aos.last_modified IS NOT NULL
            THEN (CURRENT_DATE - aos.last_modified::date)
        ELSE NULL
    END AS days_since_last_modified,

    aos.in_archive,
    aos.withdrawn,
    aos.discoverable,
    aos.multiple_titles,
    aos.published_before,
    aos.multiple_files

FROM all_open_submissions aos
LEFT JOIN collection_titles ct
    ON aos.collection_id = ct.collection_id
WHERE aos.last_modified IS NULL
   OR aos.last_modified::date < (CURRENT_DATE - INTERVAL '30 days')
ORDER BY
    days_since_last_modified DESC NULLS FIRST,
    aos.submission_source,
    collection_name,
    aos.stage_label,
    aos.item_id;
"""

df_stalled_submissions_30_days = pd.read_sql_query(
    query_stalled_submissions_30_days,
    conn
)

df_stalled_submissions_30_days["reference_month"] = REPORT_MONTH
df_stalled_submissions_30_days["environment"] = ENVIRONMENT
df_stalled_submissions_30_days["source"] = "postgres"
df_stalled_submissions_30_days["metric_definition"] = (
    "workspace and workflow submissions not modified for more than 30 days, "
    "based on item.last_modified"
)

output_stalled_submissions_30_days = export_path("db_stalled_submissions_over_30_days.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_stalled_submissions_30_days.to_csv(output_stalled_submissions_30_days, index=False)

print(f"Submissions stalled for more than 30 days: {len(df_stalled_submissions_30_days)}")

print("Distribution by submission source:")
display(
    df_stalled_submissions_30_days["submission_source"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "submission_source", "submission_source": "count"})
)

print("Distribution by stage:")
display(
    df_stalled_submissions_30_days["stage_label"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "stage_label", "stage_label": "count"})
)

display(df_stalled_submissions_30_days.head())

print(f"File saved to: {output_stalled_submissions_30_days.resolve()}")

Submissions stalled for more than 30 days: 49
Distribution by submission source:


,count,count
0,workspace,49


Distribution by stage:


,count,count
0,not_initialized,20
1,identifiers,15
2,license,8
3,traditionalpageone,3
4,traditionalpagetwo,2
5,upload,1


,submission_source,submission_id,item_id,collection_id,collection_name,submitter_id,submitter_email,stage_reached,stage_label,last_modified,...,in_archive,withdrawn,discoverable,multiple_titles,published_before,multiple_files,reference_month,environment,source,metric_definition
0,workspace,1218,1a2f0f21-4692-4a85-a319-58c8c4266e4d,831e86ef-dd49-4926-ab6d-ce25ccc8a7ff,CIRCSE,d6de8f7d-04a4-4c83-a540-0fb3b1ecc038,rachele.sprugnoli@unicatt.it,1,identifiers,2025-11-18 14:11:45.936000+00:00,...,False,False,True,False,True,False,2026-04,preprod,postgres,workspace and workflow submissions not modifie...
1,workspace,1217,3413c039-7a8f-4dcc-b1e1-c5037c6c944c,79c6fbcd-aaac-42b5-aa7d-9413eb906511,ILC4CLARIN : ILC Data & Tools,dbdc2924-7d0f-4a48-ac31-b89baf9622f4,nella.cucurullo@ilc.cnr.it,1,identifiers,2025-11-18 15:56:03.415000+00:00,...,False,False,True,False,True,False,2026-04,preprod,postgres,workspace and workflow submissions not modifie...
2,workspace,1225,9d8f26ac-708d-4350-9131-8bb5d03d42fd,79c6fbcd-aaac-42b5-aa7d-9413eb906511,ILC4CLARIN : ILC Data & Tools,d872fd3f-c174-43bb-81fd-f5bd7e72a9a4,giovanni.candeo@ilc.cnr.it,1,identifiers,2025-11-18 16:08:37.375000+00:00,...,False,False,True,False,True,False,2026-04,preprod,postgres,workspace and workflow submissions not modifie...
3,workspace,1242,a5acdb41-e470-4997-be47-71f715df56b0,79c6fbcd-aaac-42b5-aa7d-9413eb906511,ILC4CLARIN : ILC Data & Tools,b346c704-a402-4a57-8800-1ac31d559d16,martin.critelli@ilc.cnr.it,1,identifiers,2025-11-18 16:09:41.396000+00:00,...,False,False,True,False,True,False,2026-04,preprod,postgres,workspace and workflow submissions not modifie...
4,workspace,1221,ac4a2fea-8d6b-4a8c-9fd9-83bbe2d37caf,79c6fbcd-aaac-42b5-aa7d-9413eb906511,ILC4CLARIN : ILC Data & Tools,8a0ea583-b817-4f16-ba18-c09d14cf8ba4,davide.saponaro1978@gmail.com,1,identifiers,2025-11-18 16:16:04.156000+00:00,...,False,False,True,False,True,False,2026-04,preprod,postgres,workspace and workflow submissions not modifie...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_stalled_submissions_over_30_days.csv


## **Users**

### Total users

In [26]:
# - eperson_id
# - email
# - netid
# - can_log_in
# - self_registered
# - last_active
# - reference_month
# - environment
# - source
# - metric_definition

query_all_users = """
SELECT
    e.uuid AS eperson_id,
    e.email,
    e.netid,
    e.can_log_in,
    e.self_registered,
    e.last_active
FROM eperson e
ORDER BY e.email;
"""

df_all_users = pd.read_sql_query(query_all_users, conn)

df_all_users["reference_month"] = REPORT_MONTH
df_all_users["environment"] = ENVIRONMENT
df_all_users["source"] = "postgres"
df_all_users["metric_definition"] = "all users in eperson table"

output_all_users = export_path("db_all_users.csv")
df_all_users.to_csv(output_all_users, index=False)

print(f"Total system users: {len(df_all_users)}")
display(df_all_users.head())
print(f"File saved to: {output_all_users.resolve()}")

Total system users: 252


,eperson_id,email,netid,can_log_in,self_registered,last_active,reference_month,environment,source,metric_definition
0,ab600799-fcbd-4634-b034-a3b9bf1f7e58,902442@stud.unive.it,902442@unive.it[https://idp.unive.it/idp/shibb...,False,False,2025-06-19 10:05:30.011,2026-04,preprod,postgres,all users in eperson table
1,29eaad3f-dbff-4a98-b009-210e322113a4,a.boccioli@student.unisi.it,a.boccioli@unisi.it[https://shibboleth.unisi.i...,False,False,2026-04-26 07:44:35.600,2026-04,preprod,postgres,all users in eperson table
2,3e6d04f1-fd25-417b-abd0-50ad328b23f2,a.ciliberti@accademiabari.it,None,True,False,2026-01-27 10:10:35.584,2026-04,preprod,postgres,all users in eperson table
3,0ff5e3b8-9be2-40f7-a1f3-cfa2a3926c60,a.malitska@studenti.unipi.it,a.malitska@unipi.it[https://idp.unipi.it/idp/s...,False,False,2023-06-13 16:32:02.099,2026-04,preprod,postgres,all users in eperson table
4,e6509642-edf0-44e8-8166-429851e9f672,admin@ilc.cnr.it,None,True,False,2026-04-28 10:32:09.098,2026-04,preprod,postgres,all users in eperson table


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_all_users.csv


### Users by collection

In [27]:
# - collection_id
# - collection_name
# - submitters_count
# - submitter_ids
# - submitter_emails
# - items_count
# - reference_month
# - environment
# - source
# - metric_definition

query_users_per_collection = """
WITH collection_titles AS (
    SELECT
        mv.dspace_object_id AS collection_id,
        MIN(mv.text_value) AS collection_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
)
SELECT
    c.uuid AS collection_id,
    COALESCE(ct.collection_name, '[no collection title]') AS collection_name,
    COUNT(DISTINCT i.submitter_id) AS submitters_count,
    STRING_AGG(DISTINCT i.submitter_id::text, '; ' ORDER BY i.submitter_id::text) AS submitter_ids,
    STRING_AGG(DISTINCT e.email, '; ' ORDER BY e.email) AS submitter_emails,
    COUNT(DISTINCT i.uuid) AS items_count
FROM collection c
JOIN collection2item c2i
    ON c.uuid = c2i.collection_id
JOIN item i
    ON c2i.item_id = i.uuid
LEFT JOIN eperson e
    ON i.submitter_id = e.uuid
LEFT JOIN collection_titles ct
    ON c.uuid = ct.collection_id
WHERE i.in_archive = true
  AND i.submitter_id IS NOT NULL
GROUP BY c.uuid, ct.collection_name
ORDER BY submitters_count DESC, items_count DESC, collection_name;
"""

df_users_per_collection = pd.read_sql_query(
    query_users_per_collection,
    conn
)

df_users_per_collection["reference_month"] = REPORT_MONTH
df_users_per_collection["environment"] = ENVIRONMENT
df_users_per_collection["source"] = "postgres"
df_users_per_collection["metric_definition"] = "distinct item submitters per collection based on item.submitter_id"

output_users_per_collection = export_path("db_users_per_collection.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_users_per_collection.to_csv(output_users_per_collection, index=False)

print("Users / submitters per collection:")
display(df_users_per_collection.head())
print(f"File saved to: {output_users_per_collection.resolve()}")

Users / submitters per collection:

,collection_id,collection_name,submitters_count,submitter_ids,submitter_emails,items_count,reference_month,environment,source,metric_definition
0,57c1de63-635d-41ef-8e64-119865cb8f80,ILC4CLARIN : OPEN Data & Tools,30,016fe80e-534f-4b7e-90a1-027dae2584d0; 09b44a74...,902442@stud.unive.it; adriano.ferraresi@unibo....,48,2026-04,preprod,postgres,distinct item submitters per collection based ...
1,79c6fbcd-aaac-42b5-aa7d-9413eb906511,ILC4CLARIN : ILC Data & Tools,19,130129ef-9558-4e0e-b280-81d9b2f0da8b; 1cc051d3...,andrea.bellandi@ilc.cnr.it; batoul.haydar@ird....,78,2026-04,preprod,postgres,distinct item submitters per collection based ...
2,5d5f09d0-12a1-4ac0-8fa1-b71229f43363,digilibLT,2,da9eea31-105b-4c88-97d3-169f974258ce; f18284ec...,dspaceadmin@ilc.cnr.it; riccardo.delgratta@ilc...,375,2026-04,preprod,postgres,distinct item submitters per collection based ...
3,a24bc33f-a053-42a6-93f0-0cf082732a0d,ALIM Documentary Sources,2,da9eea31-105b-4c88-97d3-169f974258ce; f18284ec...,dspaceadmin@ilc.cnr.it; riccardo.delgratta@ilc...,11,2026-04,preprod,postgres,distinct item submitters per collection based ...
4,831e86ef-dd49-4926-ab6d-ce25ccc8a7ff,CIRCSE,2,3a3bce58-2375-4de9-9979-c91244f507b2; d6de8f7d...,federica.iurescia@unicatt.it; rachele.sprugnol...,11,2026-04,preprod,postgres,distinct item submitters per collection based ...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_users_per_collection.csv


### Active users in the reference month

In [28]:
# - eperson_id
# - email
# - last_active
# - reference_month
# - environment
# - source
# - metric_definition

query_active_users_last_active = """
SELECT
    e.uuid AS eperson_id,
    e.email,
    e.last_active
FROM eperson e
WHERE e.last_active IS NOT NULL
  AND DATE_TRUNC('month', e.last_active) = DATE %s
ORDER BY e.last_active DESC;
"""

df_active_users_last_active = pd.read_sql_query(
    query_active_users_last_active,
    conn,
    params=[REPORT_MONTH_DATE]
)

df_active_users_last_active["reference_month"] = REPORT_MONTH
df_active_users_last_active["environment"] = ENVIRONMENT
df_active_users_last_active["source"] = "postgres"
df_active_users_last_active["metric_definition"] = "eperson with last_active in reference month"

output_active_users_last_active = export_path("db_active_users_in_month_last_active.csv")
df_active_users_last_active.to_csv(output_active_users_last_active, index=False)

print(f"Active users in the month (last_active): {len(df_active_users_last_active)}")
print(df_active_users_last_active.head())
print(f"File saved to: {output_active_users_last_active.resolve()}")

Active users in the month (last_active): 21
                             eperson_id                         email  \
0  b217ca93-fa47-488d-87cb-1ad4f9155853     valeria.quochi@ilc.cnr.it   
1  e6509642-edf0-44e8-8166-429851e9f672              admin@ilc.cnr.it   
2  969029e7-6fef-420b-8d05-4a20dcd9ecef    caterina.fratesi2@unibo.it   
3  50eb728a-bff3-416f-a278-ac27681b3e71    cristiana.cervini@unibo.it   
4  29eaad3f-dbff-4a98-b009-210e322113a4  a.boccioli@student.unisi.it    

              last_active reference_month environment    source  \
0 2026-04-30 11:27:10.999         2026-04     preprod  postgres   
1 2026-04-28 10:32:09.098         2026-04     preprod  postgres   
2 2026-04-27 09:13:17.863         2026-04     preprod  postgres   
3 2026-04-26 09:30:30.435         2026-04     preprod  postgres   
4 2026-04-26 07:44:35.600         2026-04     preprod  postgres   

                             metric_definition  
0  eperson with last_active in reference month  
1  eperson with 

### Users who have never logged in

In [29]:
# - eperson_id
# - email
# - netid
# - can_log_in
# - self_registered
# - last_active
# - submitter_items_count
# - workspace_items_count
# - workflow_items_count
# - reference_month
# - environment
# - source
# - metric_definition

query_users_never_logged_in = """
WITH submitted_items AS (
    SELECT
        i.submitter_id AS eperson_id,
        COUNT(DISTINCT i.uuid) AS submitter_items_count
    FROM item i
    WHERE i.submitter_id IS NOT NULL
    GROUP BY i.submitter_id
),
workspace_items AS (
    SELECT
        i.submitter_id AS eperson_id,
        COUNT(DISTINCT wi.workspace_item_id) AS workspace_items_count
    FROM workspaceitem wi
    LEFT JOIN item i
        ON wi.item_id = i.uuid
    WHERE i.submitter_id IS NOT NULL
    GROUP BY i.submitter_id
),
workflow_items AS (
    SELECT
        i.submitter_id AS eperson_id,
        COUNT(DISTINCT cwi.workflowitem_id) AS workflow_items_count
    FROM cwf_workflowitem cwi
    LEFT JOIN item i
        ON cwi.item_id = i.uuid
    WHERE i.submitter_id IS NOT NULL
    GROUP BY i.submitter_id
)
SELECT
    e.uuid AS eperson_id,
    e.email,
    e.netid,
    e.can_log_in,
    e.self_registered,
    e.last_active,
    COALESCE(si.submitter_items_count, 0) AS submitter_items_count,
    COALESCE(wi.workspace_items_count, 0) AS workspace_items_count,
    COALESCE(wfi.workflow_items_count, 0) AS workflow_items_count
FROM eperson e
LEFT JOIN submitted_items si
    ON e.uuid = si.eperson_id
LEFT JOIN workspace_items wi
    ON e.uuid = wi.eperson_id
LEFT JOIN workflow_items wfi
    ON e.uuid = wfi.eperson_id
WHERE e.last_active IS NULL
ORDER BY
    e.email;
"""

df_users_never_logged_in = pd.read_sql_query(
    query_users_never_logged_in,
    conn
)

df_users_never_logged_in["reference_month"] = REPORT_MONTH
df_users_never_logged_in["environment"] = ENVIRONMENT
df_users_never_logged_in["source"] = "postgres"
df_users_never_logged_in["metric_definition"] = (
    "users with last_active IS NULL, interpreted as users who have never logged in "
    "or for whom no login activity was recorded"
)

output_users_never_logged_in = export_path("db_users_never_logged_in.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_users_never_logged_in.to_csv(output_users_never_logged_in, index=False)

print(f"Users who never logged in / without last_active: {len(df_users_never_logged_in)}")

print("Distribution by can_log_in:")
display(
    df_users_never_logged_in["can_log_in"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "can_log_in", "can_log_in": "count"})
)

print("Distribution by self_registered:")
display(
    df_users_never_logged_in["self_registered"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "self_registered", "self_registered": "count"})
)

display(df_users_never_logged_in.head())

print(f"File saved to: {output_users_never_logged_in.resolve()}")

Users who never logged in / without last_active: 2
Distribution by can_log_in:


,count,count
0,False,1
1,True,1


Distribution by self_registered:


,count,count
0,False,2


,eperson_id,email,netid,can_log_in,self_registered,last_active,submitter_items_count,workspace_items_count,workflow_items_count,reference_month,environment,source,metric_definition
0,415562f2-cd70-4273-ad08-6eeaa62ac93e,nannan.liu@unibo.it,nannan.liu@unibo.it[https://shib.unibo.it/idp/...,False,False,None,0,0,0,2026-04,preprod,postgres,"users with last_active IS NULL, interpreted as..."
1,ab6b9914-3ac6-4510-87ee-4bde5e362af3,riccardo.delgratta@gmail.com,None,True,False,None,0,0,0,2026-04,preprod,postgres,"users with last_active IS NULL, interpreted as..."


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_users_never_logged_in.csv


### Users who have been inactive for more than 12 months

In [30]:
# - eperson_id
# - email
# - netid
# - can_log_in
# - self_registered
# - last_active
# - days_since_last_active
# - submitter_items_count
# - workspace_items_count
# - workflow_items_count
# - reference_month
# - environment
# - source
# - metric_definition

query_users_inactive_12_months = """
WITH submitted_items AS (
    SELECT
        i.submitter_id AS eperson_id,
        COUNT(DISTINCT i.uuid) AS submitter_items_count
    FROM item i
    WHERE i.submitter_id IS NOT NULL
    GROUP BY i.submitter_id
),
workspace_items AS (
    SELECT
        i.submitter_id AS eperson_id,
        COUNT(DISTINCT wi.workspace_item_id) AS workspace_items_count
    FROM workspaceitem wi
    LEFT JOIN item i
        ON wi.item_id = i.uuid
    WHERE i.submitter_id IS NOT NULL
    GROUP BY i.submitter_id
),
workflow_items AS (
    SELECT
        i.submitter_id AS eperson_id,
        COUNT(DISTINCT cwi.workflowitem_id) AS workflow_items_count
    FROM cwf_workflowitem cwi
    LEFT JOIN item i
        ON cwi.item_id = i.uuid
    WHERE i.submitter_id IS NOT NULL
    GROUP BY i.submitter_id
)
SELECT
    e.uuid AS eperson_id,
    e.email,
    e.netid,
    e.can_log_in,
    e.self_registered,
    e.last_active,

    CASE
        WHEN e.last_active IS NOT NULL
            THEN DATE %s - e.last_active::date
        ELSE NULL
    END AS days_since_last_active,

    COALESCE(si.submitter_items_count, 0) AS submitter_items_count,
    COALESCE(wi.workspace_items_count, 0) AS workspace_items_count,
    COALESCE(wfi.workflow_items_count, 0) AS workflow_items_count

FROM eperson e
LEFT JOIN submitted_items si
    ON e.uuid = si.eperson_id
LEFT JOIN workspace_items wi
    ON e.uuid = wi.eperson_id
LEFT JOIN workflow_items wfi
    ON e.uuid = wfi.eperson_id

WHERE e.last_active IS NOT NULL
  AND e.last_active::date < (DATE %s - INTERVAL '12 months')

ORDER BY
    days_since_last_active DESC,
    e.email;
"""

df_users_inactive_12_months = pd.read_sql_query(
    query_users_inactive_12_months,
    conn,
    params=[REPORT_AS_OF_DATE, REPORT_AS_OF_DATE]
)

df_users_inactive_12_months["reference_month"] = REPORT_MONTH
df_users_inactive_12_months["environment"] = ENVIRONMENT
df_users_inactive_12_months["source"] = "postgres"
df_users_inactive_12_months["metric_definition"] = (
    "users with last_active older than 12 months, including submitted items, "
    "workspace items and workflow items"
)

output_users_inactive_12_months = export_path("db_users_inactive_over_12_months.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_users_inactive_12_months.to_csv(output_users_inactive_12_months, index=False)

print(f"Users inactive for more than 12 months: {len(df_users_inactive_12_months)}")

print("Distribution by can_log_in:")
display(
    df_users_inactive_12_months["can_log_in"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "can_log_in", "can_log_in": "count"})
)

print("Distribution by self_registered:")
display(
    df_users_inactive_12_months["self_registered"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "self_registered", "self_registered": "count"})
)

display(df_users_inactive_12_months.head())

print(f"File saved to: {output_users_inactive_12_months.resolve()}")

Users inactive for more than 12 months: 182
Distribution by can_log_in:


,count,count
0,False,128
1,True,54


Distribution by self_registered:


,count,count
0,False,182


,eperson_id,email,netid,can_log_in,self_registered,last_active,days_since_last_active,submitter_items_count,workspace_items_count,workflow_items_count,reference_month,environment,source,metric_definition
0,9162a502-e899-4d5f-9cfa-af3f83062567,paola.baroni@ilc.cnr.it,paola.baroni@ilc.cnr.it[https://idem-idp.ilc.c...,False,False,2016-02-08 17:03:23.761,3745,0,0,0,2026-04,preprod,postgres,"users with last_active older than 12 months, i..."
1,eb422669-24ff-4f81-8e62-02277f09aaff,sabrina.tomassini@garr.it,tomassin@garr.it[https://idp.dir.garr.it/idp/s...,False,False,2016-05-05 10:36:58.054,3658,0,0,0,2026-04,preprod,postgres,"users with last_active older than 12 months, i..."
2,5c5fec04-9ca4-4ba6-8b76-9c0cf6eb9992,luca.micellone@unito.it,lmicello@unito.it[https://idp-unito-prod.cinec...,False,False,2016-05-11 09:59:43.595,3652,0,0,0,2026-04,preprod,postgres,"users with last_active older than 12 months, i..."
3,7c7d28d7-6ab0-4369-979f-e493f559d500,barbara.monticini@garr.it,monticini@garr.it[https://idp.dir.garr.it/idp/...,False,False,2016-05-30 12:39:25.833,3633,0,0,0,2026-04,preprod,postgres,"users with last_active older than 12 months, i..."
4,741a493e-aa26-446d-9bf8-4db60c373afb,stefano@unica.it,stefano@unica.it[https://idp.unica.it/idp/shib...,False,False,2016-06-08 13:28:15.165,3624,0,0,0,2026-04,preprod,postgres,"users with last_active older than 12 months, i..."


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\monthly\2026-04\db_users_inactive_over_12_months.csv


### Active users in the last 12 month

In [31]:
# - eperson_id
# - email
# - netid
# - can_log_in
# - self_registered
# - last_active
# - days_since_last_active
# - submitter_items_count
# - workspace_items_count
# - workflow_items_count
# - reference_month
# - environment
# - source
# - metric_definition

query_active_users_last_12_months = """
WITH submitted_items AS (
    SELECT
        i.submitter_id AS eperson_id,
        COUNT(DISTINCT i.uuid) AS submitter_items_count
    FROM item i
    WHERE i.submitter_id IS NOT NULL
    GROUP BY i.submitter_id
),
workspace_items AS (
    SELECT
        i.submitter_id AS eperson_id,
        COUNT(DISTINCT wi.workspace_item_id) AS workspace_items_count
    FROM workspaceitem wi
    LEFT JOIN item i
        ON wi.item_id = i.uuid
    WHERE i.submitter_id IS NOT NULL
    GROUP BY i.submitter_id
),
workflow_items AS (
    SELECT
        i.submitter_id AS eperson_id,
        COUNT(DISTINCT cwi.workflowitem_id) AS workflow_items_count
    FROM cwf_workflowitem cwi
    LEFT JOIN item i
        ON cwi.item_id = i.uuid
    WHERE i.submitter_id IS NOT NULL
    GROUP BY i.submitter_id
)
SELECT
    e.uuid AS eperson_id,
    e.email,
    e.netid,
    e.can_log_in,
    e.self_registered,
    e.last_active,

    CASE
        WHEN e.last_active IS NOT NULL
            THEN DATE %s - e.last_active::date
        ELSE NULL
    END AS days_since_last_active,

    COALESCE(si.submitter_items_count, 0) AS submitter_items_count,
    COALESCE(wi.workspace_items_count, 0) AS workspace_items_count,
    COALESCE(wfi.workflow_items_count, 0) AS workflow_items_count

FROM eperson e
LEFT JOIN submitted_items si
    ON e.uuid = si.eperson_id
LEFT JOIN workspace_items wi
    ON e.uuid = wi.eperson_id
LEFT JOIN workflow_items wfi
    ON e.uuid = wfi.eperson_id

WHERE e.last_active IS NOT NULL
  AND e.last_active::date >= (DATE %s - INTERVAL '12 months')
  AND e.last_active::date <= DATE %s

ORDER BY
    e.last_active DESC,
    e.email;
"""

df_active_users_last_12_months = pd.read_sql_query(
    query_active_users_last_12_months,
    conn,
    params=[REPORT_AS_OF_DATE, REPORT_AS_OF_DATE, REPORT_AS_OF_DATE]
)

df_active_users_last_12_months["reference_month"] = REPORT_MONTH
df_active_users_last_12_months["environment"] = ENVIRONMENT
df_active_users_last_12_months["source"] = "postgres"
df_active_users_last_12_months["metric_definition"] = (
    "users with last_active within the last 12 months, including submitted items, "
    "workspace items and workflow items"
)

output_active_users_last_12_months = export_path("db_active_users_last_12_months.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_active_users_last_12_months.to_csv(output_active_users_last_12_months, index=False)

print(f"Users active in the last 12 months: {len(df_active_users_last_12_months)}")

print("Distribution by can_log_in:")
display(
    df_active_users_last_12_months["can_log_in"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "can_log_in", "can_log_in": "count"})
)

print("Distribution by self_registered:")
display(
    df_active_users_last_12_months["self_registered"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "self_registered", "self_registered": "count"})
)

display(df_active_users_last_12_months.head())

print(f"File saved to: {output_active_users_last_12_months.resolve()}")

Users active in the last 12 months: 68
Distribution by can_log_in:


,count,count
0,False,36
1,True,32


Distribution by self_registered:


,count,count
0,False,68


,eperson_id,email,netid,can_log_in,self_registered,last_active,days_since_last_active,submitter_items_count,workspace_items_count,workflow_items_count,reference_month,environment,source,metric_definition
0,4cda417f-f654-4f2a-8ce9-ef010f467598,michele.mallia@ilc.cnr.it,michele.mallia@ilc.cnr.it[https://idem-idp.ilc...,False,False,2026-05-11 13:38:26.821,0,2,0,0,2026-04,preprod,postgres,users with last_active within the last 12 mont...
1,da9eea31-105b-4c88-97d3-169f974258ce,riccardo.delgratta@ilc.cnr.it,riccardo.delgratta@ilc.cnr.it[https://idem-idp...,False,False,2026-05-05 13:42:01.263,6,27,0,0,2026-04,preprod,postgres,users with last_active within the last 12 mont...
2,b217ca93-fa47-488d-87cb-1ad4f9155853,valeria.quochi@ilc.cnr.it,valeria.quochi@ilc.cnr.it[https://idem-idp.ilc...,False,False,2026-04-30 11:27:10.999,11,20,8,0,2026-04,preprod,postgres,users with last_active within the last 12 mont...
3,e6509642-edf0-44e8-8166-429851e9f672,admin@ilc.cnr.it,None,True,False,2026-04-28 10:32:09.098,13,0,0,0,2026-04,preprod,postgres,users with last_active within the last 12 mont...
4,969029e7-6fef-420b-8d05-4a20dcd9ecef,caterina.fratesi2@unibo.it,None,True,False,2026-04-27 09:13:17.863,14,0,0,0,2026-04,preprod,postgres,users with last_active within the last 12 mont...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\monthly\2026-04\db_active_users_last_12_months.csv


### Users by collection

In [32]:
# - collection_id
# - collection_name
# - community_id
# - community_name
# - submitters_count
# - submitter_ids
# - submitter_emails
# - items_count
# - reference_month
# - environment
# - source
# - metric_definition

query_users_per_collection = """
WITH collection_titles AS (
    SELECT
        mv.dspace_object_id AS collection_id,
        MIN(mv.text_value) AS collection_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
community_titles AS (
    SELECT
        mv.dspace_object_id AS community_id,
        MIN(mv.text_value) AS community_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
)
SELECT
    c.uuid AS collection_id,
    COALESCE(ct.collection_name, '[no collection title]') AS collection_name,
    c2c.community_id,
    COALESCE(comt.community_name, '[no community title]') AS community_name,

    COUNT(DISTINCT i.submitter_id) AS submitters_count,

    STRING_AGG(
        DISTINCT i.submitter_id::text,
        '; '
        ORDER BY i.submitter_id::text
    ) AS submitter_ids,

    STRING_AGG(
        DISTINCT COALESCE(e.email, '[no email]'),
        '; '
        ORDER BY COALESCE(e.email, '[no email]')
    ) AS submitter_emails,

    COUNT(DISTINCT i.uuid) AS items_count

FROM collection c
LEFT JOIN collection2item c2i
    ON c.uuid = c2i.collection_id
LEFT JOIN item i
    ON c2i.item_id = i.uuid
   AND i.in_archive = true
   AND i.submitter_id IS NOT NULL
LEFT JOIN eperson e
    ON i.submitter_id = e.uuid
LEFT JOIN collection_titles ct
    ON c.uuid = ct.collection_id
LEFT JOIN community2collection c2c
    ON c.uuid = c2c.collection_id
LEFT JOIN community_titles comt
    ON c2c.community_id = comt.community_id

GROUP BY
    c.uuid,
    ct.collection_name,
    c2c.community_id,
    comt.community_name

ORDER BY
    submitters_count DESC,
    items_count DESC,
    community_name,
    collection_name;
"""

df_users_per_collection = pd.read_sql_query(
    query_users_per_collection,
    conn
)

df_users_per_collection["reference_month"] = REPORT_MONTH
df_users_per_collection["environment"] = ENVIRONMENT
df_users_per_collection["source"] = "postgres"
df_users_per_collection["metric_definition"] = (
    "distinct submitters per collection based on archived items, "
    "including reference community and item count"
)

output_users_per_collection = export_path("db_users_per_collection.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_users_per_collection.to_csv(output_users_per_collection, index=False)

print("Users / submitters per collection:")
display(df_users_per_collection.head())

print(f"Collections analyzed: {len(df_users_per_collection)}")
print(f"File saved to: {output_users_per_collection.resolve()}")

Export exists and overwrite_exports=false, skipped: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_users_per_collection.csv


Users / submitters per collection:


,collection_id,collection_name,community_id,community_name,submitters_count,submitter_ids,submitter_emails,items_count,reference_month,environment,source,metric_definition
0,57c1de63-635d-41ef-8e64-119865cb8f80,ILC4CLARIN : OPEN Data & Tools,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,30,016fe80e-534f-4b7e-90a1-027dae2584d0; 09b44a74...,902442@stud.unive.it; adriano.ferraresi@unibo....,48,2026-04,preprod,postgres,distinct submitters per collection based on ar...
1,79c6fbcd-aaac-42b5-aa7d-9413eb906511,ILC4CLARIN : ILC Data & Tools,bdf96608-5c94-471c-9996-63942e832141,ILC,19,130129ef-9558-4e0e-b280-81d9b2f0da8b; 1cc051d3...,[no email]; andrea.bellandi@ilc.cnr.it; batoul...,78,2026-04,preprod,postgres,distinct submitters per collection based on ar...
2,5d5f09d0-12a1-4ac0-8fa1-b71229f43363,digilibLT,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,2,da9eea31-105b-4c88-97d3-169f974258ce; f18284ec...,dspaceadmin@ilc.cnr.it; riccardo.delgratta@ilc...,375,2026-04,preprod,postgres,distinct submitters per collection based on ar...
3,a24bc33f-a053-42a6-93f0-0cf082732a0d,ALIM Documentary Sources,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,2,da9eea31-105b-4c88-97d3-169f974258ce; f18284ec...,dspaceadmin@ilc.cnr.it; riccardo.delgratta@ilc...,11,2026-04,preprod,postgres,distinct submitters per collection based on ar...
4,831e86ef-dd49-4926-ab6d-ce25ccc8a7ff,CIRCSE,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,2,3a3bce58-2375-4de9-9979-c91244f507b2; d6de8f7d...,federica.iurescia@unicatt.it; rachele.sprugnol...,11,2026-04,preprod,postgres,distinct submitters per collection based on ar...


Collections analyzed: 11
File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_users_per_collection.csv


## **Groups and authorizations**

### Total groups

In [33]:
# - group_id
# - group_name
# - permanent
# - members_count
# - reference_month
# - environment
# - source
# - metric_definition

query_total_groups = """
WITH group_members AS (
    SELECT
        eperson_group_id AS group_id,
        COUNT(DISTINCT eperson_id) AS members_count
    FROM epersongroup2eperson
    GROUP BY eperson_group_id
)
SELECT
    eg.uuid AS group_id,
    eg.name AS group_name,
    eg.permanent,
    COALESCE(gm.members_count, 0) AS members_count
FROM epersongroup eg
LEFT JOIN group_members gm
    ON eg.uuid = gm.group_id
ORDER BY
    members_count DESC,
    group_name;
"""

df_total_groups = pd.read_sql_query(
    query_total_groups,
    conn
)

df_total_groups["reference_month"] = REPORT_MONTH
df_total_groups["environment"] = ENVIRONMENT
df_total_groups["source"] = "postgres"
df_total_groups["metric_definition"] = (
    "total DSpace groups with group name, permanence flag and number of direct eperson members"
)

output_total_groups = export_path("db_total_groups.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_total_groups.to_csv(output_total_groups, index=False)

print(f"Total groups: {len(df_total_groups)}")

print("Distribution of permanent / non-permanent groups:")
display(
    df_total_groups["permanent"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "permanent", "permanent": "count"})
)

display(df_total_groups.head())

print(f"File saved to: {output_total_groups.resolve()}")

Total groups: 98
Distribution of permanent / non-permanent groups:


,count,count
0,False,96
1,True,2


,group_id,group_name,permanent,members_count,reference_month,environment,source,metric_definition
0,7ffebeee-6aeb-4fd9-acb7-99be6d134972,Authenticated,False,11,2026-04,preprod,postgres,"total DSpace groups with group name, permanenc..."
1,f0cc50e7-d1e2-4984-b407-a2be30f8eef9,Administrator,True,6,2026-04,preprod,postgres,"total DSpace groups with group name, permanenc..."
2,13031b19-a4f9-4daf-b7e1-9bd18befe712,Amministratori ILC,False,3,2026-04,preprod,postgres,"total DSpace groups with group name, permanenc..."
3,dc48f7e7-c616-47f1-a895-ae6c0dfc7ed0,COLLECTION_79c6fbcd-aaac-42b5-aa7d-9413eb90651...,False,3,2026-04,preprod,postgres,"total DSpace groups with group name, permanenc..."
4,143fe065-cc34-42b4-b574-17e99355c337,COLLECTION_79c6fbcd-aaac-42b5-aa7d-9413eb90651...,False,3,2026-04,preprod,postgres,"total DSpace groups with group name, permanenc..."


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_total_groups.csv


### Empty groups

In [34]:
# Colonne estratte:
# - group_id
# - group_name
# - permanent
# - members_count
#
# Colonne aggiunte lato Python:
# - reference_month
# - environment
# - source
# - metric_definition

query_empty_groups = """
WITH group_members AS (
    SELECT
        eperson_group_id AS group_id,
        COUNT(DISTINCT eperson_id) AS members_count
    FROM epersongroup2eperson
    GROUP BY eperson_group_id
)
SELECT
    eg.uuid AS group_id,
    eg.name AS group_name,
    eg.permanent,
    COALESCE(gm.members_count, 0) AS members_count
FROM epersongroup eg
LEFT JOIN group_members gm
    ON eg.uuid = gm.group_id
WHERE COALESCE(gm.members_count, 0) = 0
ORDER BY
    eg.permanent DESC,
    eg.name;
"""

df_empty_groups = pd.read_sql_query(
    query_empty_groups,
    conn
)

df_empty_groups["reference_month"] = REPORT_MONTH
df_empty_groups["environment"] = ENVIRONMENT
df_empty_groups["source"] = "postgres"
df_empty_groups["metric_definition"] = (
    "DSpace groups without direct eperson members"
)

output_empty_groups = export_path("db_empty_groups.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_empty_groups.to_csv(output_empty_groups, index=False)

print(f"Empty groups: {len(df_empty_groups)}")

print("Distribution of empty permanent / non-permanent groups:")
display(
    df_empty_groups["permanent"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "permanent", "permanent": "count"})
)

display(df_empty_groups.head())

print(f"File saved to: {output_empty_groups.resolve()}")

Empty groups: 68
Distribution of empty permanent / non-permanent groups:


,count,count
0,False,67
1,True,1


,group_id,group_name,permanent,members_count,reference_month,environment,source,metric_definition
0,dabf1638-8a74-401b-affa-86bb49819815,Anonymous,True,0,2026-04,preprod,postgres,DSpace groups without direct eperson members
1,50023b2b-d1c0-4927-99ac-8e3d71c3e91c,COLLECTION ILC Submitter,False,0,2026-04,preprod,postgres,DSpace groups without direct eperson members
2,90e21f4c-3059-4e2c-aab9-597335f9e336,COLLECTION OPEN Submitter,False,0,2026-04,preprod,postgres,DSpace groups without direct eperson members
3,effc856b-f547-46ca-b339-8189b2e305ae,COLLECTION_15_WORKFLOW_STEP_3,False,0,2026-04,preprod,postgres,DSpace groups without direct eperson members
4,5df12493-dc02-49af-b7d6-6d8b861cd9f7,COLLECTION_17_WORKFLOW_STEP_3,False,0,2026-04,preprod,postgres,DSpace groups without direct eperson members


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_empty_groups.csv


### Users by group

In [35]:
# - group_id
# - group_name
# - permanent
# - members_count
# - member_ids
# - member_emails
# - reference_month
# - environment
# - source
# - metric_definition

query_users_per_group = """
WITH group_members AS (
    SELECT
        eg.uuid AS group_id,
        eg.name AS group_name,
        eg.permanent,
        e.uuid AS eperson_id,
        e.email AS eperson_email
    FROM epersongroup eg
    LEFT JOIN epersongroup2eperson eg2e
        ON eg.uuid = eg2e.eperson_group_id
    LEFT JOIN eperson e
        ON eg2e.eperson_id = e.uuid
)
SELECT
    group_id,
    group_name,
    permanent,

    COUNT(DISTINCT eperson_id) AS members_count,

    STRING_AGG(
        DISTINCT eperson_id::text,
        '; '
        ORDER BY eperson_id::text
    ) AS member_ids,

    STRING_AGG(
        DISTINCT COALESCE(eperson_email, '[no email]'),
        '; '
        ORDER BY COALESCE(eperson_email, '[no email]')
    ) FILTER (
        WHERE eperson_id IS NOT NULL
    ) AS member_emails

FROM group_members
GROUP BY
    group_id,
    group_name,
    permanent

ORDER BY
    members_count DESC,
    group_name;
"""

df_users_per_group = pd.read_sql_query(
    query_users_per_group,
    conn
)

df_users_per_group["reference_month"] = REPORT_MONTH
df_users_per_group["environment"] = ENVIRONMENT
df_users_per_group["source"] = "postgres"
df_users_per_group["metric_definition"] = (
    "direct eperson members grouped by DSpace group"
)

output_users_per_group = export_path("db_users_per_group.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_users_per_group.to_csv(output_users_per_group, index=False)

print(f"Groups analyzed: {len(df_users_per_group)}")

print("Distribution of users per group:")

df_members_distribution = (
    df_users_per_group["members_count"]
    .value_counts(dropna=False)
    .rename_axis("members_count")
    .reset_index(name="groups_count")
    .sort_values("members_count")
)

display(df_members_distribution)

display(df_users_per_group.head())

print(f"File saved to: {output_users_per_group.resolve()}")

Groups analyzed: 98
Distribution of users per group:


,members_count,groups_count
0,0,68
5,1,1
1,2,22
2,3,5
4,6,1
3,11,1


,group_id,group_name,permanent,members_count,member_ids,member_emails,reference_month,environment,source,metric_definition
0,7ffebeee-6aeb-4fd9-acb7-99be6d134972,Authenticated,False,11,5ff86ef6-57d6-41cb-b27e-6cb25ace3815; 91337806...,alessandro.enea@ilc.cnr.it; ellepannitto@gmail...,2026-04,preprod,postgres,direct eperson members grouped by DSpace group
1,f0cc50e7-d1e2-4984-b407-a2be30f8eef9,Administrator,True,6,4cda417f-f654-4f2a-8ce9-ef010f467598; a0584082...,admin@ilc.cnr.it; alessandro.enea@ilc.cnr.it; ...,2026-04,preprod,postgres,direct eperson members grouped by DSpace group
2,13031b19-a4f9-4daf-b7e1-9bd18befe712,Amministratori ILC,False,3,4cda417f-f654-4f2a-8ce9-ef010f467598; da9eea31...,alessandro.enea@ilc.cnr.it; michele.mallia@ilc...,2026-04,preprod,postgres,direct eperson members grouped by DSpace group
3,dc48f7e7-c616-47f1-a895-ae6c0dfc7ed0,COLLECTION_79c6fbcd-aaac-42b5-aa7d-9413eb90651...,False,3,4cda417f-f654-4f2a-8ce9-ef010f467598; da9eea31...,alessandro.enea@ilc.cnr.it; michele.mallia@ilc...,2026-04,preprod,postgres,direct eperson members grouped by DSpace group
4,143fe065-cc34-42b4-b574-17e99355c337,COLLECTION_79c6fbcd-aaac-42b5-aa7d-9413eb90651...,False,3,4cda417f-f654-4f2a-8ce9-ef010f467598; da9eea31...,alessandro.enea@ilc.cnr.it; michele.mallia@ilc...,2026-04,preprod,postgres,direct eperson members grouped by DSpace group


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_users_per_group.csv


### Submitter/Workflow/Administrator groups per collection

In [36]:
# - collection_id
# - collection_internal_id
# - collection_name
# - submitter_group_ids
# - submitter_group_names
# - submitter_group_members_count
# - workflow_group_ids
# - workflow_group_names
# - workflow_group_members_count
# - administrator_group_ids
# - administrator_group_names
# - administrator_group_members_count
# - reference_month
# - environment
# - source
# - metric_definition

query_collection_groups = """
WITH collection_titles AS (
    SELECT
        mv.dspace_object_id AS collection_id,
        MIN(mv.text_value) AS collection_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
group_members AS (
    SELECT
        eg.uuid AS group_id,
        COUNT(DISTINCT eg2e.eperson_id) AS members_count
    FROM epersongroup eg
    LEFT JOIN epersongroup2eperson eg2e
        ON eg.uuid = eg2e.eperson_group_id
    GROUP BY eg.uuid
),
groups_by_name AS (
    SELECT
        c.uuid AS collection_id,
        c.collection_id AS collection_internal_id,
        eg.uuid AS group_id,
        eg.name AS group_name,

        CASE
            WHEN eg.name = 'COLLECTION_' || c.collection_id || '_SUBMIT'
              OR eg.name = 'COLLECTION_' || c.uuid::text || '_SUBMIT'
                THEN 'submitter'

            WHEN eg.name LIKE 'COLLECTION_' || c.collection_id || '_WORKFLOW_STEP_%'
              OR eg.name LIKE 'COLLECTION_' || c.uuid::text || '_WORKFLOW_STEP_%'
                THEN 'workflow'

            WHEN eg.name = 'COLLECTION_' || c.collection_id || '_ADMIN'
              OR eg.name = 'COLLECTION_' || c.uuid::text || '_ADMIN'
                THEN 'administrator'

            ELSE 'other'
        END AS group_role,

        COALESCE(gm.members_count, 0) AS members_count

    FROM collection c
    JOIN epersongroup eg
        ON eg.name = 'COLLECTION_' || c.collection_id || '_SUBMIT'
        OR eg.name = 'COLLECTION_' || c.uuid::text || '_SUBMIT'
        OR eg.name LIKE 'COLLECTION_' || c.collection_id || '_WORKFLOW_STEP_%'
        OR eg.name LIKE 'COLLECTION_' || c.uuid::text || '_WORKFLOW_STEP_%'
        OR eg.name = 'COLLECTION_' || c.collection_id || '_ADMIN'
        OR eg.name = 'COLLECTION_' || c.uuid::text || '_ADMIN'
    LEFT JOIN group_members gm
        ON eg.uuid = gm.group_id
),
groups_by_policy AS (
    SELECT
        c.uuid AS collection_id,
        c.collection_id AS collection_internal_id,
        eg.uuid AS group_id,
        eg.name AS group_name,

        CASE
            WHEN rp.action_id = 3
                THEN 'submitter'
            WHEN rp.action_id IN (5, 6, 7)
                THEN 'workflow'
            WHEN rp.action_id = 8
                THEN 'administrator'
            ELSE 'other'
        END AS group_role,

        COALESCE(gm.members_count, 0) AS members_count

    FROM collection c
    JOIN resourcepolicy rp
        ON rp.dspace_object = c.uuid
    JOIN epersongroup eg
        ON rp.epersongroup_id = eg.uuid
    LEFT JOIN group_members gm
        ON eg.uuid = gm.group_id
    WHERE rp.epersongroup_id IS NOT NULL
      AND rp.action_id IN (3, 5, 6, 7, 8)
),
collection_groups_union AS (
    SELECT * FROM groups_by_name
    UNION
    SELECT * FROM groups_by_policy
)
SELECT
    c.uuid AS collection_id,
    c.collection_id AS collection_internal_id,
    COALESCE(ct.collection_name, '[no collection title]') AS collection_name,

    STRING_AGG(
        DISTINCT cgu.group_id::text,
        '; '
        ORDER BY cgu.group_id::text
    ) FILTER (
        WHERE cgu.group_role = 'submitter'
    ) AS submitter_group_ids,

    STRING_AGG(
        DISTINCT cgu.group_name,
        '; '
        ORDER BY cgu.group_name
    ) FILTER (
        WHERE cgu.group_role = 'submitter'
    ) AS submitter_group_names,

    COALESCE(
        SUM(DISTINCT cgu.members_count) FILTER (
            WHERE cgu.group_role = 'submitter'
        ),
        0
    ) AS submitter_group_members_count,

    STRING_AGG(
        DISTINCT cgu.group_id::text,
        '; '
        ORDER BY cgu.group_id::text
    ) FILTER (
        WHERE cgu.group_role = 'workflow'
    ) AS workflow_group_ids,

    STRING_AGG(
        DISTINCT cgu.group_name,
        '; '
        ORDER BY cgu.group_name
    ) FILTER (
        WHERE cgu.group_role = 'workflow'
    ) AS workflow_group_names,

    COALESCE(
        SUM(DISTINCT cgu.members_count) FILTER (
            WHERE cgu.group_role = 'workflow'
        ),
        0
    ) AS workflow_group_members_count,

    STRING_AGG(
        DISTINCT cgu.group_id::text,
        '; '
        ORDER BY cgu.group_id::text
    ) FILTER (
        WHERE cgu.group_role = 'administrator'
    ) AS administrator_group_ids,

    STRING_AGG(
        DISTINCT cgu.group_name,
        '; '
        ORDER BY cgu.group_name
    ) FILTER (
        WHERE cgu.group_role = 'administrator'
    ) AS administrator_group_names,

    COALESCE(
        SUM(DISTINCT cgu.members_count) FILTER (
            WHERE cgu.group_role = 'administrator'
        ),
        0
    ) AS administrator_group_members_count

FROM collection c
LEFT JOIN collection_titles ct
    ON c.uuid = ct.collection_id
LEFT JOIN collection_groups_union cgu
    ON c.uuid = cgu.collection_id

GROUP BY
    c.uuid,
    c.collection_id,
    ct.collection_name

ORDER BY
    collection_name,
    c.uuid;
"""

df_collection_groups = pd.read_sql_query(
    query_collection_groups,
    conn
)

df_collection_groups["reference_month"] = REPORT_MONTH
df_collection_groups["environment"] = ENVIRONMENT
df_collection_groups["source"] = "postgres"
df_collection_groups["metric_definition"] = (
    "submitter, workflow and administrator groups per collection based on both "
    "numeric collection id, collection UUID naming conventions and resourcepolicy"
)

output_collection_groups = export_path("db_collection_groups.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_collection_groups.to_csv(output_collection_groups, index=False)

print(f"Collections analyzed: {len(df_collection_groups)}")

print("Collections without submitter group:")
display(
    df_collection_groups[
        df_collection_groups["submitter_group_ids"].isna()
    ][["collection_id", "collection_internal_id", "collection_name"]].head()
)

print("Collections without workflow group:")
display(
    df_collection_groups[
        df_collection_groups["workflow_group_ids"].isna()
    ][["collection_id", "collection_internal_id", "collection_name"]].head()
)

print("Collections without administrator group:")
display(
    df_collection_groups[
        df_collection_groups["administrator_group_ids"].isna()
    ][["collection_id", "collection_internal_id", "collection_name"]].head()
)

display(df_collection_groups.head())

print(f"File saved to: {output_collection_groups.resolve()}")

Collections analyzed: 11
Collections without submitter group:


,collection_id,collection_internal_id,collection_name
5,962a7a1c-3ab2-4291-ac4e-13aec990b65a,None,ILC WebLicht Web Services


Collections without workflow group:


,collection_id,collection_internal_id,collection_name
0,a24bc33f-a053-42a6-93f0-0cf082732a0d,None,ALIM Documentary Sources
2,99107ee7-a7af-4354-81a9-e8312ec03823,None,BIA-Net FONTES
3,831e86ef-dd49-4926-ab6d-ce25ccc8a7ff,None,CIRCSE
4,1c610d27-1c9d-485d-b27c-3aadccaffd4c,None,Corpus KIParla Collection
5,962a7a1c-3ab2-4291-ac4e-13aec990b65a,None,ILC WebLicht Web Services


Collections without administrator group:


,collection_id,collection_internal_id,collection_name
5,962a7a1c-3ab2-4291-ac4e-13aec990b65a,None,ILC WebLicht Web Services


,collection_id,collection_internal_id,collection_name,submitter_group_ids,submitter_group_names,submitter_group_members_count,workflow_group_ids,workflow_group_names,workflow_group_members_count,administrator_group_ids,administrator_group_names,administrator_group_members_count,reference_month,environment,source,metric_definition
0,a24bc33f-a053-42a6-93f0-0cf082732a0d,None,ALIM Documentary Sources,961f3579-a8ad-47be-a219-6edf2b9fb9c2; ddc06b40...,COLLECTION_a24bc33f-a053-42a6-93f0-0cf082732a0...,3.0,None,None,0.0,a3f1a87f-8c9b-494e-aa48-b23fa28ae9e9,COLLECTION_a24bc33f-a053-42a6-93f0-0cf082732a0...,0.0,2026-04,preprod,postgres,"submitter, workflow and administrator groups p..."
1,514ba599-9a43-4477-b1a6-e3528048458b,None,ALIM Literary Sources,0fc25759-eace-4dea-8c02-bcd67310b948; 24e282b7...,COLLECTION_20_WORKFLOW_STEP_1; COLLECTION_20_W...,2.0,0fc25759-eace-4dea-8c02-bcd67310b948; 25888c79...,COLLECTION_20_WORKFLOW_STEP_1; COLLECTION_20_W...,2.0,250b50eb-11d3-46f5-8dff-f67c38314e6c,COLLECTION_514ba599-9a43-4477-b1a6-e3528048458...,0.0,2026-04,preprod,postgres,"submitter, workflow and administrator groups p..."
2,99107ee7-a7af-4354-81a9-e8312ec03823,None,BIA-Net FONTES,65684a4f-ef7c-4515-af39-a500327d4b31; a270fecd...,COLLECTION_99107ee7-a7af-4354-81a9-e8312ec0382...,0.0,None,None,0.0,45e50922-cd6f-4f94-8b2f-007e77e88833,COLLECTION_99107ee7-a7af-4354-81a9-e8312ec0382...,0.0,2026-04,preprod,postgres,"submitter, workflow and administrator groups p..."
3,831e86ef-dd49-4926-ab6d-ce25ccc8a7ff,None,CIRCSE,2b107fe6-fdb7-48f4-b479-aa51c6ccbf8a; 5dd4248d...,COLLECTION_831e86ef-dd49-4926-ab6d-ce25ccc8a7f...,0.0,None,None,0.0,3c7269e0-0d94-4618-bf40-971aa0e2b9fc,COLLECTION_831e86ef-dd49-4926-ab6d-ce25ccc8a7f...,0.0,2026-04,preprod,postgres,"submitter, workflow and administrator groups p..."
4,1c610d27-1c9d-485d-b27c-3aadccaffd4c,None,Corpus KIParla Collection,55624afd-6332-483b-b914-7acbb1419383; 6641cd81...,COLLECTION_1c610d27-1c9d-485d-b27c-3aadccaffd4...,1.0,None,None,0.0,16b4fea2-bd0e-4720-b96e-a1f8bbbcc13c,COLLECTION_1c610d27-1c9d-485d-b27c-3aadccaffd4...,0.0,2026-04,preprod,postgres,"submitter, workflow and administrator groups p..."


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_collection_groups.csv


## **Technical integrity**

### Items without ORIGINAL bundle

In [37]:
# - item_id
# - in_archive
# - withdrawn
# - discoverable
# - last_modified
# - submitter_id
# - submitter_email
# - bundles_count
# - bundle_names
# - bitstreams_count
# - total_size_bytes
# - total_size_mb
# - reference_month
# - environment
# - source
# - metric_definition

query_items_without_original_bundle = """
WITH bundle_titles AS (
    SELECT
        mv.dspace_object_id AS bundle_id,
        MIN(mv.text_value) AS bundle_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
item_bundle_summary AS (
    SELECT
        i.uuid AS item_id,

        COUNT(DISTINCT b.uuid) AS bundles_count,

        STRING_AGG(
            DISTINCT COALESCE(bt.bundle_name, '[no bundle title]'),
            '; '
            ORDER BY COALESCE(bt.bundle_name, '[no bundle title]')
        ) AS bundle_names,

        BOOL_OR(UPPER(COALESCE(bt.bundle_name, '')) = 'ORIGINAL') AS has_original_bundle,

        COUNT(DISTINCT bs.uuid) FILTER (
            WHERE bs.deleted = false OR bs.deleted IS NULL
        ) AS bitstreams_count,

        COALESCE(
            SUM(bs.size_bytes) FILTER (
                WHERE bs.deleted = false OR bs.deleted IS NULL
            ),
            0
        ) AS total_size_bytes

    FROM item i
    LEFT JOIN item2bundle i2b
        ON i.uuid = i2b.item_id
    LEFT JOIN bundle b
        ON i2b.bundle_id = b.uuid
    LEFT JOIN bundle_titles bt
        ON b.uuid = bt.bundle_id
    LEFT JOIN bundle2bitstream b2bs
        ON b.uuid = b2bs.bundle_id
    LEFT JOIN bitstream bs
        ON b2bs.bitstream_id = bs.uuid
    GROUP BY i.uuid
)
SELECT
    i.uuid AS item_id,
    i.in_archive,
    i.withdrawn,
    i.discoverable,
    i.last_modified,
    i.submitter_id,
    e.email AS submitter_email,

    COALESCE(ibs.bundles_count, 0) AS bundles_count,
    COALESCE(ibs.bundle_names, '[no bundles]') AS bundle_names,
    COALESCE(ibs.bitstreams_count, 0) AS bitstreams_count,
    COALESCE(ibs.total_size_bytes, 0) AS total_size_bytes,
    ROUND(COALESCE(ibs.total_size_bytes, 0) / 1024.0 / 1024.0, 2) AS total_size_mb

FROM item i
LEFT JOIN item_bundle_summary ibs
    ON i.uuid = ibs.item_id
LEFT JOIN eperson e
    ON i.submitter_id = e.uuid
WHERE COALESCE(ibs.has_original_bundle, false) = false
ORDER BY
    i.in_archive DESC,
    i.withdrawn ASC,
    i.discoverable DESC,
    bitstreams_count DESC,
    total_size_bytes DESC,
    i.last_modified DESC NULLS LAST,
    i.uuid;
"""

df_items_without_original_bundle = pd.read_sql_query(
    query_items_without_original_bundle,
    conn
)

df_items_without_original_bundle["reference_month"] = REPORT_MONTH
df_items_without_original_bundle["environment"] = ENVIRONMENT
df_items_without_original_bundle["source"] = "postgres"
df_items_without_original_bundle["metric_definition"] = (
    "items without an ORIGINAL bundle, including item status, existing bundles, "
    "non-deleted bitstream count and total size"
)

output_items_without_original_bundle = export_path("db_items_without_original_bundle.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_items_without_original_bundle.to_csv(output_items_without_original_bundle, index=False)

print(f"Items without ORIGINAL bundle: {len(df_items_without_original_bundle)}")

print("Distribution by archival status:")
display(
    df_items_without_original_bundle[["in_archive", "withdrawn", "discoverable"]]
    .value_counts(dropna=False)
    .reset_index(name="count")
)

display(df_items_without_original_bundle.head())

print(f"File saved to: {output_items_without_original_bundle.resolve()}")

Items without ORIGINAL bundle: 483
Distribution by archival status:


,in_archive,withdrawn,discoverable,count
0,True,False,True,439
1,False,False,True,42
2,False,True,True,2


,item_id,in_archive,withdrawn,discoverable,last_modified,submitter_id,submitter_email,bundles_count,bundle_names,bitstreams_count,total_size_bytes,total_size_mb,reference_month,environment,source,metric_definition
0,0a955db2-73c2-4cba-a0af-edd2376a1c81,True,False,True,2026-05-10 15:28:55.412000+00:00,4cda417f-f654-4f2a-8ce9-ef010f467598,michele.mallia@ilc.cnr.it,1,LICENSE,2,3433.0,0.00,2026-04,preprod,postgres,"items without an ORIGINAL bundle, including it..."
1,cb6a2bbc-a961-4ee6-b87c-1c904de7e312,True,False,True,2025-11-18 15:56:17.386000+00:00,da9eea31-105b-4c88-97d3-169f974258ce,riccardo.delgratta@ilc.cnr.it,1,METADATA,1,6481.0,0.01,2026-04,preprod,postgres,"items without an ORIGINAL bundle, including it..."
2,ba2b96da-d23a-4b9d-8e20-2a6297ecb3ec,True,False,True,2025-11-18 15:56:58.727000+00:00,da9eea31-105b-4c88-97d3-169f974258ce,riccardo.delgratta@ilc.cnr.it,1,METADATA,1,4937.0,0.00,2026-04,preprod,postgres,"items without an ORIGINAL bundle, including it..."
3,5489b60c-e78a-4685-8c6c-0194517adf61,True,False,True,2025-11-18 15:57:00.616000+00:00,da9eea31-105b-4c88-97d3-169f974258ce,riccardo.delgratta@ilc.cnr.it,1,METADATA,1,3836.0,0.00,2026-04,preprod,postgres,"items without an ORIGINAL bundle, including it..."
4,d5a78b7b-2239-4ded-b4ea-a99a738060a1,True,False,True,2025-11-18 16:08:07.992000+00:00,5c9e77eb-8c67-4a5e-8ba3-69f96d093a39,duccio.piccardi@unisi.it,1,LICENSE,1,1685.0,0.00,2026-04,preprod,postgres,"items without an ORIGINAL bundle, including it..."


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_items_without_original_bundle.csv


### Orphan metadatavalue

In [38]:
# - metadata_value_id
# - dspace_object_id
# - schema
# - element
# - qualifier
# - metadata_field
# - text_value
# - text_lang
# - place
# - inferred_object_type
# - reference_month
# - environment
# - source
# - metric_definition

query_orphan_metadatavalue = """
SELECT
    mv.metadata_value_id,
    mv.dspace_object_id,
    msr.short_id AS schema,
    mfr.element,
    mfr.qualifier,

    CASE
        WHEN mfr.qualifier IS NULL OR TRIM(mfr.qualifier) = ''
            THEN msr.short_id || '.' || mfr.element
        ELSE msr.short_id || '.' || mfr.element || '.' || mfr.qualifier
    END AS metadata_field,

    mv.text_value,
    mv.text_lang,
    mv.place,

    CASE
        WHEN i.uuid IS NOT NULL THEN 'item'
        WHEN c.uuid IS NOT NULL THEN 'collection'
        WHEN com.uuid IS NOT NULL THEN 'community'
        WHEN bs.uuid IS NOT NULL THEN 'bitstream'
        WHEN b.uuid IS NOT NULL THEN 'bundle'
        ELSE 'orphan'
    END AS inferred_object_type

FROM metadatavalue mv
JOIN metadatafieldregistry mfr
    ON mv.metadata_field_id = mfr.metadata_field_id
JOIN metadataschemaregistry msr
    ON mfr.metadata_schema_id = msr.metadata_schema_id

LEFT JOIN item i
    ON mv.dspace_object_id = i.uuid
LEFT JOIN collection c
    ON mv.dspace_object_id = c.uuid
LEFT JOIN community com
    ON mv.dspace_object_id = com.uuid
LEFT JOIN bitstream bs
    ON mv.dspace_object_id = bs.uuid
LEFT JOIN bundle b
    ON mv.dspace_object_id = b.uuid

WHERE i.uuid IS NULL
  AND c.uuid IS NULL
  AND com.uuid IS NULL
  AND bs.uuid IS NULL
  AND b.uuid IS NULL

ORDER BY
    metadata_field,
    mv.dspace_object_id,
    mv.place,
    mv.metadata_value_id;
"""

df_orphan_metadatavalue = pd.read_sql_query(
    query_orphan_metadatavalue,
    conn
)

df_orphan_metadatavalue["reference_month"] = REPORT_MONTH
df_orphan_metadatavalue["environment"] = ENVIRONMENT
df_orphan_metadatavalue["source"] = "postgres"
df_orphan_metadatavalue["metric_definition"] = (
    "metadata values whose dspace_object_id does not match any existing item, "
    "collection, community, bitstream or bundle UUID"
)

output_orphan_metadatavalue = export_path("db_orphan_metadatavalue.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_orphan_metadatavalue.to_csv(output_orphan_metadatavalue, index=False)

print(f"Orphan metadatavalue records: {len(df_orphan_metadatavalue)}")

print("Distribution by metadata field:")
display(
    df_orphan_metadatavalue["metadata_field"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "metadata_field", "metadata_field": "count"})
)

display(df_orphan_metadatavalue.head())

print(f"File saved to: {output_orphan_metadatavalue.resolve()}")

Orphan metadatavalue records: 899
Distribution by metadata field:


,count,count
0,eperson.firstname,246
1,eperson.lastname,246
2,eperson.language,242
3,dspace.agreements.cookies,48
4,dspace.agreements.end-user,43
5,dc.description,37
6,eperson.phone,37


,metadata_value_id,dspace_object_id,schema,element,qualifier,metadata_field,text_value,text_lang,place,inferred_object_type,reference_month,environment,source,metric_definition
0,151224,042db788-4960-4472-92bf-4fa2f04afc26,dc,description,None,dc.description,ILC4CLARIN : ILC Data & Tools finaleditor group,None,0,orphan,2026-04,preprod,postgres,metadata values whose dspace_object_id does no...
1,151223,143fe065-cc34-42b4-b574-17e99355c337,dc,description,None,dc.description,ILC4CLARIN : ILC Data & Tools reviewer group,None,0,orphan,2026-04,preprod,postgres,metadata values whose dspace_object_id does no...
2,151691,16b4fea2-bd0e-4720-b96e-a1f8bbbcc13c,dc,description,None,dc.description,Corpus KIParla Collection collection-admin group,None,0,orphan,2026-04,preprod,postgres,metadata values whose dspace_object_id does no...
3,151650,250b50eb-11d3-46f5-8dff-f67c38314e6c,dc,description,None,dc.description,ALIM Literary Sources collection-admin group,None,0,orphan,2026-04,preprod,postgres,metadata values whose dspace_object_id does no...
4,151696,2b107fe6-fdb7-48f4-b479-aa51c6ccbf8a,dc,description,None,dc.description,CIRCSE submitters group,None,0,orphan,2026-04,preprod,postgres,metadata values whose dspace_object_id does no...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_orphan_metadatavalue.csv


### Orphan Hhandle

In [39]:
# - handle_id
# - handle
# - resource_id
# - resource_type_id
# - inferred_resource_type
# - item_exists
# - collection_exists
# - community_exists
# - bitstream_exists
# - bundle_exists
# - reference_month
# - environment
# - source
# - metric_definition

query_orphan_handles = """
SELECT
    h.handle_id,
    h.handle,
    h.resource_id,
    h.resource_type_id,

    CASE
        WHEN i.uuid IS NOT NULL THEN 'item'
        WHEN c.uuid IS NOT NULL THEN 'collection'
        WHEN com.uuid IS NOT NULL THEN 'community'
        WHEN bs.uuid IS NOT NULL THEN 'bitstream'
        WHEN b.uuid IS NOT NULL THEN 'bundle'
        ELSE 'orphan'
    END AS inferred_resource_type,

    CASE WHEN i.uuid IS NOT NULL THEN true ELSE false END AS item_exists,
    CASE WHEN c.uuid IS NOT NULL THEN true ELSE false END AS collection_exists,
    CASE WHEN com.uuid IS NOT NULL THEN true ELSE false END AS community_exists,
    CASE WHEN bs.uuid IS NOT NULL THEN true ELSE false END AS bitstream_exists,
    CASE WHEN b.uuid IS NOT NULL THEN true ELSE false END AS bundle_exists

FROM handle h
LEFT JOIN item i
    ON h.resource_id = i.uuid
LEFT JOIN collection c
    ON h.resource_id = c.uuid
LEFT JOIN community com
    ON h.resource_id = com.uuid
LEFT JOIN bitstream bs
    ON h.resource_id = bs.uuid
LEFT JOIN bundle b
    ON h.resource_id = b.uuid

WHERE i.uuid IS NULL
  AND c.uuid IS NULL
  AND com.uuid IS NULL
  AND bs.uuid IS NULL
  AND b.uuid IS NULL

ORDER BY
    h.resource_type_id,
    h.handle;
"""

df_orphan_handles = pd.read_sql_query(
    query_orphan_handles,
    conn
)

df_orphan_handles["reference_month"] = REPORT_MONTH
df_orphan_handles["environment"] = ENVIRONMENT
df_orphan_handles["source"] = "postgres"
df_orphan_handles["metric_definition"] = (
    "handles whose resource_id does not match any existing item, collection, "
    "community, bitstream or bundle UUID"
)

output_orphan_handles = export_path("db_orphan_handles.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_orphan_handles.to_csv(output_orphan_handles, index=False)

print(f"Orphan handles: {len(df_orphan_handles)}")

print("Distribution by resource_type_id:")
display(
    df_orphan_handles["resource_type_id"]
    .value_counts(dropna=False)
    .rename_axis("resource_type_id")
    .reset_index(name="count")
    .sort_values("resource_type_id")
)

display(df_orphan_handles.head())

print(f"File saved to: {output_orphan_handles.resolve()}")

Orphan handles: 84


Distribution by resource_type_id:


,resource_type_id,count
0,2.0,76
2,4.0,1
1,NaN,7


,handle_id,handle,resource_id,resource_type_id,inferred_resource_type,item_exists,collection_exists,community_exists,bitstream_exists,bundle_exists,reference_month,environment,source,metric_definition
0,7,20.500.11752/ILC-100,None,2.0,orphan,False,False,False,False,False,2026-04,preprod,postgres,handles whose resource_id does not match any e...
1,23,20.500.11752/ILC-1013,None,2.0,orphan,False,False,False,False,False,2026-04,preprod,postgres,handles whose resource_id does not match any e...
2,1982,20.500.11752/ILC-2042,None,2.0,orphan,False,False,False,False,False,2026-04,preprod,postgres,handles whose resource_id does not match any e...
3,1988,20.500.11752/ILC-2046,None,2.0,orphan,False,False,False,False,False,2026-04,preprod,postgres,handles whose resource_id does not match any e...
4,1989,20.500.11752/ILC-2047,None,2.0,orphan,False,False,False,False,False,2026-04,preprod,postgres,handles whose resource_id does not match any e...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_orphan_handles.csv


### Items with non valid license

In [40]:
# - item_id
# - item_name
# - in_archive
# - withdrawn
# - discoverable
# - last_modified
# - license_values
# - license_status
# - reference_month
# - environment
# - source
# - metric_definition

query_items_invalid_license = """
WITH item_titles AS (
    SELECT
        mv.dspace_object_id AS item_id,
        STRING_AGG(mv.text_value, ' | ' ORDER BY mv.place) AS item_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mfr.qualifier IS NULL
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
item_licenses AS (
    SELECT
        i.uuid AS item_id,

        STRING_AGG(
            DISTINCT TRIM(mv.text_value),
            '; '
            ORDER BY TRIM(mv.text_value)
        ) AS license_values,

        COUNT(mv.metadata_value_id) AS license_metadata_count

    FROM item i
    LEFT JOIN metadatavalue mv
        ON i.uuid = mv.dspace_object_id
    LEFT JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    LEFT JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE
           msr.short_id = 'dc'
       AND (
              mfr.element = 'license'
           OR mfr.element = 'rights'
           OR (
                  mfr.element = 'rights'
              AND mfr.qualifier IN ('uri', 'license')
              )
       )
    GROUP BY i.uuid
),
license_classification AS (
    SELECT
        i.uuid AS item_id,
        COALESCE(il.license_values, '[no license]') AS license_values,

        CASE
            WHEN il.license_values IS NULL
                THEN 'MISSING_LICENSE'

            WHEN TRIM(il.license_values) = ''
                THEN 'EMPTY_LICENSE'

            WHEN il.license_values ~* '^(unknown|unkown|not specified|not available|n/a|na|null|none|no license)$'
                THEN 'GENERIC_OR_UNSPECIFIED_LICENSE'

            WHEN il.license_values ~* '(creativecommons\\.org|CC-BY|CC BY|CC0|CC-BY-SA|CC BY-SA|CC-BY-NC|CC BY-NC|CC-BY-ND|CC BY-ND|CC-BY-NC-SA|CC BY-NC-SA|CC-BY-NC-ND|CC BY-NC-ND)'
                THEN 'VALID_OR_RECOGNIZED_LICENSE'

            WHEN il.license_values ~* '(clarin|clariah|metashare|meta-share|olac|rightsstatements\\.org)'
                THEN 'VALID_OR_RECOGNIZED_LICENSE'

            WHEN il.license_values ~* '(Apache License|MIT License|GNU General Public License|GPL|LGPL|BSD|Mozilla Public License|MPL|EUPL)'
                THEN 'VALID_OR_RECOGNIZED_LICENSE'

            WHEN il.license_values ~* '^https?://'
                THEN 'URI_LICENSE_TO_REVIEW'

            ELSE 'UNRECOGNIZED_LICENSE_VALUE'
        END AS license_status

    FROM item i
    LEFT JOIN item_licenses il
        ON i.uuid = il.item_id
)
SELECT
    i.uuid AS item_id,
    COALESCE(it.item_name, '[no title]') AS item_name,
    i.in_archive,
    i.withdrawn,
    i.discoverable,
    i.last_modified,
    lc.license_values,
    lc.license_status

FROM item i
LEFT JOIN item_titles it
    ON i.uuid = it.item_id
LEFT JOIN license_classification lc
    ON i.uuid = lc.item_id

WHERE i.in_archive = true
  AND (
          lc.license_status = 'MISSING_LICENSE'
       OR lc.license_status = 'EMPTY_LICENSE'
       OR lc.license_status = 'GENERIC_OR_UNSPECIFIED_LICENSE'
       OR lc.license_status = 'UNRECOGNIZED_LICENSE_VALUE'
       OR lc.license_status = 'URI_LICENSE_TO_REVIEW'
  )

ORDER BY
    lc.license_status,
    i.last_modified DESC NULLS LAST,
    item_name,
    i.uuid;
"""

df_items_invalid_license = pd.read_sql_query(
    query_items_invalid_license,
    conn
)

df_items_invalid_license["reference_month"] = REPORT_MONTH
df_items_invalid_license["environment"] = ENVIRONMENT
df_items_invalid_license["source"] = "postgres"
df_items_invalid_license["metric_definition"] = (
    "archived items with missing, empty, generic, unrecognized or review-needed license values"
)

output_items_invalid_license = export_path("db_items_invalid_license.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_items_invalid_license.to_csv(output_items_invalid_license, index=False)

print(f"Items with missing, ambiguous, or unrecognized license: {len(df_items_invalid_license)}")

print("Distribution by license status:")
display(
    df_items_invalid_license["license_status"]
    .value_counts(dropna=False)
    .rename_axis("license_status")
    .reset_index(name="count")
)

display(df_items_invalid_license.head())

print(f"File saved to: {output_items_invalid_license.resolve()}")

Items with missing, ambiguous, or unrecognized license: 30
Distribution by license status:


,license_status,count
0,MISSING_LICENSE,30


,item_id,item_name,in_archive,withdrawn,discoverable,last_modified,license_values,license_status,reference_month,environment,source,metric_definition
0,ae41c614-4b41-40b2-834f-1f6804de34bd,CASH - Corpus management Annotation and SearcH,True,False,True,2025-11-18 16:16:16.544000+00:00,[no license],MISSING_LICENSE,2026-04,preprod,postgres,"archived items with missing, empty, generic, u..."
1,f41ea177-a164-4f30-8a7c-7626a734f002,It-Sr-NER,True,False,True,2025-11-18 16:15:47.835000+00:00,[no license],MISSING_LICENSE,2026-04,preprod,postgres,"archived items with missing, empty, generic, u..."
2,aea8299d-d19d-4860-a027-6f6f086d3c0a,LexO-server: REST services for Linguistic Link...,True,False,True,2025-11-18 16:15:41.353000+00:00,[no license],MISSING_LICENSE,2026-04,preprod,postgres,"archived items with missing, empty, generic, u..."
3,134926b8-9608-492e-9aba-745aabb02979,UD-001-001,True,False,True,2025-11-18 16:04:12.552000+00:00,[no license],MISSING_LICENSE,2026-04,preprod,postgres,"archived items with missing, empty, generic, u..."
4,4b3e9ce2-8cb0-4ba9-a04e-5bd22fd931d4,SIMPLE-LOD,True,False,True,2025-11-18 16:03:28.485000+00:00,[no license],MISSING_LICENSE,2026-04,preprod,postgres,"archived items with missing, empty, generic, u..."


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_items_invalid_license.csv


## **Licences**

### Items distribution per licence

In [41]:
# - license_value
# - items_count
# - item_ids
# - archived_items_count
# - withdrawn_items_count
# - non_discoverable_items_count
# - reference_month
# - environment
# - source
# - metric_definition

query_items_by_license = """
WITH item_licenses AS (
    SELECT
        i.uuid AS item_id,
        i.in_archive,
        i.withdrawn,
        i.discoverable,

        COALESCE(
            NULLIF(TRIM(mv.text_value), ''),
            '[empty license]'
        ) AS license_value

    FROM item i
    LEFT JOIN metadatavalue mv
        ON i.uuid = mv.dspace_object_id
    LEFT JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    LEFT JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id

    WHERE i.in_archive = true
      AND (
             (
                msr.short_id = 'dc'
                AND mfr.element = 'rights'
             )
          OR (
                msr.short_id = 'dc'
                AND mfr.element = 'license'
             )
          OR (
                msr.short_id = 'dc'
                AND mfr.element = 'rights'
                AND mfr.qualifier IN ('uri', 'license')
             )
      )
),
items_without_license AS (
    SELECT
        i.uuid AS item_id,
        i.in_archive,
        i.withdrawn,
        i.discoverable,
        '[no license]' AS license_value
    FROM item i
    WHERE i.in_archive = true
      AND NOT EXISTS (
          SELECT 1
          FROM metadatavalue mv
          JOIN metadatafieldregistry mfr
              ON mv.metadata_field_id = mfr.metadata_field_id
          JOIN metadataschemaregistry msr
              ON mfr.metadata_schema_id = msr.metadata_schema_id
          WHERE mv.dspace_object_id = i.uuid
            AND mv.text_value IS NOT NULL
            AND TRIM(mv.text_value) <> ''
            AND (
                   (
                      msr.short_id = 'dc'
                      AND mfr.element = 'rights'
                   )
                OR (
                      msr.short_id = 'dc'
                      AND mfr.element = 'license'
                   )
                OR (
                      msr.short_id = 'dc'
                      AND mfr.element = 'rights'
                      AND mfr.qualifier IN ('uri', 'license')
                   )
            )
      )
),
all_license_values AS (
    SELECT * FROM item_licenses
    UNION ALL
    SELECT * FROM items_without_license
)
SELECT
    license_value,

    COUNT(DISTINCT item_id) AS items_count,

    STRING_AGG(
        DISTINCT item_id::text,
        '; '
        ORDER BY item_id::text
    ) AS item_ids,

    COUNT(DISTINCT item_id) FILTER (
        WHERE in_archive = true
    ) AS archived_items_count,

    COUNT(DISTINCT item_id) FILTER (
        WHERE withdrawn = true
    ) AS withdrawn_items_count,

    COUNT(DISTINCT item_id) FILTER (
        WHERE discoverable = false
    ) AS non_discoverable_items_count

FROM all_license_values
GROUP BY license_value
ORDER BY
    items_count DESC,
    license_value;
"""

df_items_by_license = pd.read_sql_query(
    query_items_by_license,
    conn
)

df_items_by_license["reference_month"] = REPORT_MONTH
df_items_by_license["environment"] = ENVIRONMENT
df_items_by_license["source"] = "postgres"
df_items_by_license["metric_definition"] = (
    "distribution of archived DSpace items by license value, including items without license"
)

output_items_by_license = export_path("db_items_by_license.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_items_by_license.to_csv(output_items_by_license, index=False)

print(f"Distinct licenses found: {len(df_items_by_license)}")
print(f"Total items counted across licenses: {df_items_by_license['items_count'].sum()}")

display(df_items_by_license.head())

print(f"File saved to: {output_items_by_license.resolve()}")

Distinct licenses found: 24
Total items counted across licenses: 2845


,license_value,items_count,item_ids,archived_items_count,withdrawn_items_count,non_discoverable_items_count,reference_month,environment,source,metric_definition
0,PUB,938,00001e22-b7ab-4e13-aa3f-335300ecc4ac; 0086cbb7...,938,0,0,2026-04,preprod,postgres,distribution of archived DSpace items by licen...
1,Creative Commons - Attribution-NonCommercial-N...,413,00001e22-b7ab-4e13-aa3f-335300ecc4ac; 00be2594...,413,0,0,2026-04,preprod,postgres,distribution of archived DSpace items by licen...
2,http://creativecommons.org/licenses/by-nc-nd/4.0/,412,00001e22-b7ab-4e13-aa3f-335300ecc4ac; 00be2594...,412,0,0,2026-04,preprod,postgres,distribution of archived DSpace items by licen...
3,Creative Commons - Attribution-NonCommercial 3...,375,0086cbb7-2307-48ab-991f-bce666b490fc; 01ce7407...,375,0,0,2026-04,preprod,postgres,distribution of archived DSpace items by licen...
4,http://creativecommons.org/licenses/by-nc/3.0/,375,0086cbb7-2307-48ab-991f-bce666b490fc; 01ce7407...,375,0,0,2026-04,preprod,postgres,distribution of archived DSpace items by licen...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_items_by_license.csv


### Items with multiple licences

In [42]:
# - item_id
# - item_name
# - in_archive
# - withdrawn
# - discoverable
# - last_modified
# - licenses_count
# - license_values
# - license_fields
# - reference_month
# - environment
# - source
# - metric_definition

query_items_with_multiple_licenses = """
WITH item_titles AS (
    SELECT
        mv.dspace_object_id AS item_id,
        STRING_AGG(mv.text_value, ' | ' ORDER BY mv.place) AS item_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mfr.qualifier IS NULL
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
item_license_values AS (
    SELECT
        i.uuid AS item_id,
        i.in_archive,
        i.withdrawn,
        i.discoverable,
        i.last_modified,

        TRIM(mv.text_value) AS license_value,

        CASE
            WHEN mfr.qualifier IS NULL OR TRIM(mfr.qualifier) = ''
                THEN msr.short_id || '.' || mfr.element
            ELSE msr.short_id || '.' || mfr.element || '.' || mfr.qualifier
        END AS license_field

    FROM item i
    JOIN metadatavalue mv
        ON i.uuid = mv.dspace_object_id
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id

    WHERE i.in_archive = true
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
      AND msr.short_id = 'dc'
      AND (
             mfr.element = 'rights'
          OR mfr.element = 'license'
          OR (
                mfr.element = 'rights'
            AND mfr.qualifier IN ('uri', 'license')
             )
      )
),
license_summary AS (
    SELECT
        item_id,
        COUNT(DISTINCT license_value) AS licenses_count,

        STRING_AGG(
            DISTINCT license_value,
            '; '
            ORDER BY license_value
        ) AS license_values,

        STRING_AGG(
            DISTINCT license_field,
            '; '
            ORDER BY license_field
        ) AS license_fields

    FROM item_license_values
    GROUP BY item_id
    HAVING COUNT(DISTINCT license_value) > 1
)
SELECT
    ilv.item_id,
    COALESCE(it.item_name, '[no title]') AS item_name,
    MAX(ilv.in_archive::int)::boolean AS in_archive,
    MAX(ilv.withdrawn::int)::boolean AS withdrawn,
    MAX(ilv.discoverable::int)::boolean AS discoverable,
    MAX(ilv.last_modified) AS last_modified,

    ls.licenses_count,
    ls.license_values,
    ls.license_fields

FROM license_summary ls
JOIN item_license_values ilv
    ON ls.item_id = ilv.item_id
LEFT JOIN item_titles it
    ON ls.item_id = it.item_id

GROUP BY
    ilv.item_id,
    it.item_name,
    ls.licenses_count,
    ls.license_values,
    ls.license_fields

ORDER BY
    ls.licenses_count DESC,
    last_modified DESC NULLS LAST,
    item_name,
    item_id;
"""

df_items_with_multiple_licenses = pd.read_sql_query(
    query_items_with_multiple_licenses,
    conn
)

df_items_with_multiple_licenses["reference_month"] = REPORT_MONTH
df_items_with_multiple_licenses["environment"] = ENVIRONMENT
df_items_with_multiple_licenses["source"] = "postgres"
df_items_with_multiple_licenses["metric_definition"] = (
    "archived items with more than one distinct license value across dc.rights, "
    "dc.license and dc.rights.uri/license metadata fields"
)

output_items_with_multiple_licenses = export_path("db_items_with_multiple_licenses.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_items_with_multiple_licenses.to_csv(output_items_with_multiple_licenses, index=False)

print(f"Items with multiple licenses: {len(df_items_with_multiple_licenses)}")

print("Distribution of number of licenses per item:")
display(
    df_items_with_multiple_licenses["licenses_count"]
    .value_counts(dropna=False)
    .rename_axis("licenses_count")
    .reset_index(name="items_count")
    .sort_values("licenses_count")
)

display(df_items_with_multiple_licenses.head())

print(f"File saved to: {output_items_with_multiple_licenses.resolve()}")

Items with multiple licenses: 938
Distribution of number of licenses per item:


,licenses_count,items_count
0,3,938


,item_id,item_name,in_archive,withdrawn,discoverable,last_modified,licenses_count,license_values,license_fields,reference_month,environment,source,metric_definition
0,0a955db2-73c2-4cba-a0af-edd2376a1c81,DigItAnt Search,True,False,True,2026-05-10 15:28:55.412000+00:00,3,"GNU General Public Licence, version 3; PUB; ht...",dc.rights; dc.rights.label; dc.rights.uri,2026-04,preprod,postgres,archived items with more than one distinct lic...
1,349f1389-2ce6-4d62-8ada-39d4d2a4d9c7,al-qāmūs l-muḥīṭ: a digital Arabic dictionary:...,True,False,True,2026-05-09 14:36:43.344000+00:00,3,Creative Commons - Attribution-ShareAlike 4.0 ...,dc.rights; dc.rights.label; dc.rights.uri,2026-04,preprod,postgres,archived items with more than one distinct lic...
2,e38f9258-8508-4838-894c-01c26d4cf9b3,Corpus of Affective Objects in Autobiographica...,True,False,True,2026-04-30 15:37:19.162000+00:00,3,Creative Commons - Attribution 4.0 Internation...,dc.rights; dc.rights.label; dc.rights.uri,2026-04,preprod,postgres,archived items with more than one distinct lic...
3,35b566ce-6199-4f20-aa8e-a2b86133adbc,OIIC - Online Interactions in Intercomprehensi...,True,False,True,2026-04-30 15:36:54.622000+00:00,3,Creative Commons - Attribution-NonCommercial-S...,dc.rights; dc.rights.label; dc.rights.uri,2026-04,preprod,postgres,archived items with more than one distinct lic...
4,aeeec067-2091-401a-9503-be401c8c820a,OIIC - Online Interactions in Intercomprehensi...,True,False,True,2026-04-30 15:36:46.302000+00:00,3,Creative Commons - Attribution-NonCommercial-S...,dc.rights; dc.rights.label; dc.rights.uri,2026-04,preprod,postgres,archived items with more than one distinct lic...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_items_with_multiple_licenses.csv


### Licences by collection

In [43]:
# - collection_id
# - collection_name
# - community_id
# - community_name
# - license_value
# - items_count
# - item_ids
# - archived_items_count
# - withdrawn_items_count
# - non_discoverable_items_count
# - reference_month
# - environment
# - source
# - metric_definition

query_licenses_per_collection = """
WITH collection_titles AS (
    SELECT
        mv.dspace_object_id AS collection_id,
        MIN(mv.text_value) AS collection_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
community_titles AS (
    SELECT
        mv.dspace_object_id AS community_id,
        MIN(mv.text_value) AS community_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
item_license_values AS (
    SELECT
        i.uuid AS item_id,
        i.in_archive,
        i.withdrawn,
        i.discoverable,
        c.uuid AS collection_id,
        COALESCE(
            NULLIF(TRIM(mv.text_value), ''),
            '[empty license]'
        ) AS license_value
    FROM item i
    JOIN collection2item c2i
        ON i.uuid = c2i.item_id
    JOIN collection c
        ON c2i.collection_id = c.uuid
    LEFT JOIN metadatavalue mv
        ON i.uuid = mv.dspace_object_id
    LEFT JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    LEFT JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE i.in_archive = true
      AND (
             (
                msr.short_id = 'dc'
                AND mfr.element = 'rights'
             )
          OR (
                msr.short_id = 'dc'
                AND mfr.element = 'license'
             )
          OR (
                msr.short_id = 'dc'
                AND mfr.element = 'rights'
                AND mfr.qualifier IN ('uri', 'license')
             )
      )
),
items_without_license AS (
    SELECT
        i.uuid AS item_id,
        i.in_archive,
        i.withdrawn,
        i.discoverable,
        c.uuid AS collection_id,
        '[no license]' AS license_value
    FROM item i
    JOIN collection2item c2i
        ON i.uuid = c2i.item_id
    JOIN collection c
        ON c2i.collection_id = c.uuid
    WHERE i.in_archive = true
      AND NOT EXISTS (
          SELECT 1
          FROM metadatavalue mv
          JOIN metadatafieldregistry mfr
              ON mv.metadata_field_id = mfr.metadata_field_id
          JOIN metadataschemaregistry msr
              ON mfr.metadata_schema_id = msr.metadata_schema_id
          WHERE mv.dspace_object_id = i.uuid
            AND mv.text_value IS NOT NULL
            AND TRIM(mv.text_value) <> ''
            AND msr.short_id = 'dc'
            AND (
                   mfr.element = 'rights'
                OR mfr.element = 'license'
                OR (
                      mfr.element = 'rights'
                  AND mfr.qualifier IN ('uri', 'license')
                   )
            )
      )
),
all_license_values AS (
    SELECT * FROM item_license_values
    UNION ALL
    SELECT * FROM items_without_license
)
SELECT
    alv.collection_id,
    COALESCE(ct.collection_name, '[no collection title]') AS collection_name,
    c2c.community_id,
    COALESCE(comt.community_name, '[no community title]') AS community_name,

    alv.license_value,

    COUNT(DISTINCT alv.item_id) AS items_count,

    STRING_AGG(
        DISTINCT alv.item_id::text,
        '; '
        ORDER BY alv.item_id::text
    ) AS item_ids,

    COUNT(DISTINCT alv.item_id) FILTER (
        WHERE alv.in_archive = true
    ) AS archived_items_count,

    COUNT(DISTINCT alv.item_id) FILTER (
        WHERE alv.withdrawn = true
    ) AS withdrawn_items_count,

    COUNT(DISTINCT alv.item_id) FILTER (
        WHERE alv.discoverable = false
    ) AS non_discoverable_items_count

FROM all_license_values alv
LEFT JOIN collection_titles ct
    ON alv.collection_id = ct.collection_id
LEFT JOIN community2collection c2c
    ON alv.collection_id = c2c.collection_id
LEFT JOIN community_titles comt
    ON c2c.community_id = comt.community_id

GROUP BY
    alv.collection_id,
    ct.collection_name,
    c2c.community_id,
    comt.community_name,
    alv.license_value

ORDER BY
    collection_name,
    items_count DESC,
    license_value;
"""

df_licenses_per_collection = pd.read_sql_query(
    query_licenses_per_collection,
    conn
)

df_licenses_per_collection["reference_month"] = REPORT_MONTH
df_licenses_per_collection["environment"] = ENVIRONMENT
df_licenses_per_collection["source"] = "postgres"
df_licenses_per_collection["metric_definition"] = (
    "distribution of archived DSpace items by license value and collection, "
    "including items without license"
)

output_licenses_per_collection = export_path("db_licenses_per_collection.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_licenses_per_collection.to_csv(output_licenses_per_collection, index=False)

print(f"License / collection rows: {len(df_licenses_per_collection)}")
print(f"Collections with at least one detected license: {df_licenses_per_collection['collection_id'].nunique()}")

print("Global license distribution:")
display(
    df_licenses_per_collection
    .groupby("license_value", dropna=False)["items_count"]
    .sum()
    .reset_index(name="items_count")
    .sort_values("items_count", ascending=False)
)

display(df_licenses_per_collection.head())

print(f"File saved to: {output_licenses_per_collection.resolve()}")

License / collection rows: 63
Collections with at least one detected license: 11
Global license distribution:


,license_value,items_count
10,PUB,938
5,Creative Commons - Attribution-NonCommercial-N...,413
13,http://creativecommons.org/licenses/by-nc-nd/4.0/,412
3,Creative Commons - Attribution-NonCommercial 3...,375
16,http://creativecommons.org/licenses/by-nc/3.0/,375
18,http://creativecommons.org/licenses/by-sa/4.0/,64
7,Creative Commons - Attribution-ShareAlike 4.0 ...,64
6,Creative Commons - Attribution-NonCommercial-S...,58
15,http://creativecommons.org/licenses/by-nc-sa/4.0/,58
12,[no license],30


,collection_id,collection_name,community_id,community_name,license_value,items_count,item_ids,archived_items_count,withdrawn_items_count,non_discoverable_items_count,reference_month,environment,source,metric_definition
0,a24bc33f-a053-42a6-93f0-0cf082732a0d,ALIM Documentary Sources,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,Creative Commons - Attribution-NonCommercial-N...,11,09e1359c-55c9-4407-9701-3fd44d0aa78d; 0be1a92f...,11,0,0,2026-04,preprod,postgres,distribution of archived DSpace items by licen...
1,a24bc33f-a053-42a6-93f0-0cf082732a0d,ALIM Documentary Sources,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,PUB,10,09e1359c-55c9-4407-9701-3fd44d0aa78d; 0be1a92f...,10,0,0,2026-04,preprod,postgres,distribution of archived DSpace items by licen...
2,a24bc33f-a053-42a6-93f0-0cf082732a0d,ALIM Documentary Sources,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,http://creativecommons.org/licenses/by-nc-nd/4.0/,10,09e1359c-55c9-4407-9701-3fd44d0aa78d; 0be1a92f...,10,0,0,2026-04,preprod,postgres,distribution of archived DSpace items by licen...
3,514ba599-9a43-4477-b1a6-e3528048458b,ALIM Literary Sources,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,Creative Commons - Attribution-NonCommercial-N...,394,00001e22-b7ab-4e13-aa3f-335300ecc4ac; 00be2594...,394,0,0,2026-04,preprod,postgres,distribution of archived DSpace items by licen...
4,514ba599-9a43-4477-b1a6-e3528048458b,ALIM Literary Sources,dc62d695-9dbb-4b9d-baa5-a49ab45288cd,OPEN,PUB,394,00001e22-b7ab-4e13-aa3f-335300ecc4ac; 00be2594...,394,0,0,2026-04,preprod,postgres,distribution of archived DSpace items by licen...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_licenses_per_collection.csv


### Licences by resource type

In [44]:
# - item_type
# - license_value
# - items_count
# - item_ids
# - archived_items_count
# - withdrawn_items_count
# - non_discoverable_items_count
# - reference_month
# - environment
# - source
# - metric_definition

query_licenses_per_resource_type = """
WITH item_types AS (
    SELECT
        i.uuid AS item_id,
        COALESCE(
            STRING_AGG(
                DISTINCT TRIM(mv.text_value),
                '; '
                ORDER BY TRIM(mv.text_value)
            ),
            '[no type]'
        ) AS item_type
    FROM item i
    LEFT JOIN metadatavalue mv
        ON i.uuid = mv.dspace_object_id
    LEFT JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    LEFT JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE i.in_archive = true
      AND (
            mv.metadata_value_id IS NULL
         OR (
                msr.short_id = 'dc'
            AND mfr.element = 'type'
            AND mv.text_value IS NOT NULL
            AND TRIM(mv.text_value) <> ''
            )
      )
    GROUP BY i.uuid
),
item_license_values AS (
    SELECT
        i.uuid AS item_id,
        i.in_archive,
        i.withdrawn,
        i.discoverable,
        COALESCE(
            NULLIF(TRIM(mv.text_value), ''),
            '[empty license]'
        ) AS license_value
    FROM item i
    LEFT JOIN metadatavalue mv
        ON i.uuid = mv.dspace_object_id
    LEFT JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    LEFT JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE i.in_archive = true
      AND msr.short_id = 'dc'
      AND (
             mfr.element = 'rights'
          OR mfr.element = 'license'
          OR (
                mfr.element = 'rights'
            AND mfr.qualifier IN ('uri', 'license')
             )
      )
),
items_without_license AS (
    SELECT
        i.uuid AS item_id,
        i.in_archive,
        i.withdrawn,
        i.discoverable,
        '[no license]' AS license_value
    FROM item i
    WHERE i.in_archive = true
      AND NOT EXISTS (
          SELECT 1
          FROM metadatavalue mv
          JOIN metadatafieldregistry mfr
              ON mv.metadata_field_id = mfr.metadata_field_id
          JOIN metadataschemaregistry msr
              ON mfr.metadata_schema_id = msr.metadata_schema_id
          WHERE mv.dspace_object_id = i.uuid
            AND mv.text_value IS NOT NULL
            AND TRIM(mv.text_value) <> ''
            AND msr.short_id = 'dc'
            AND (
                   mfr.element = 'rights'
                OR mfr.element = 'license'
                OR (
                      mfr.element = 'rights'
                  AND mfr.qualifier IN ('uri', 'license')
                   )
            )
      )
),
all_license_values AS (
    SELECT * FROM item_license_values
    UNION ALL
    SELECT * FROM items_without_license
)
SELECT
    COALESCE(it.item_type, '[no type]') AS item_type,
    alv.license_value,

    COUNT(DISTINCT alv.item_id) AS items_count,

    STRING_AGG(
        DISTINCT alv.item_id::text,
        '; '
        ORDER BY alv.item_id::text
    ) AS item_ids,

    COUNT(DISTINCT alv.item_id) FILTER (
        WHERE alv.in_archive = true
    ) AS archived_items_count,

    COUNT(DISTINCT alv.item_id) FILTER (
        WHERE alv.withdrawn = true
    ) AS withdrawn_items_count,

    COUNT(DISTINCT alv.item_id) FILTER (
        WHERE alv.discoverable = false
    ) AS non_discoverable_items_count

FROM all_license_values alv
LEFT JOIN item_types it
    ON alv.item_id = it.item_id

GROUP BY
    COALESCE(it.item_type, '[no type]'),
    alv.license_value

ORDER BY
    item_type,
    items_count DESC,
    license_value;
"""

df_licenses_per_resource_type = pd.read_sql_query(
    query_licenses_per_resource_type,
    conn
)

df_licenses_per_resource_type["reference_month"] = REPORT_MONTH
df_licenses_per_resource_type["environment"] = ENVIRONMENT
df_licenses_per_resource_type["source"] = "postgres"
df_licenses_per_resource_type["metric_definition"] = (
    "distribution of archived DSpace items by license value and resource type, "
    "including items without license"
)

output_licenses_per_resource_type = export_path("db_licenses_per_resource_type.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_licenses_per_resource_type.to_csv(output_licenses_per_resource_type, index=False)

print(f"License / resource type rows: {len(df_licenses_per_resource_type)}")
print(f"Detected resource types: {df_licenses_per_resource_type['item_type'].nunique()}")

print("Global license distribution by resource type:")
display(
    df_licenses_per_resource_type
    .groupby(["item_type", "license_value"], dropna=False)["items_count"]
    .sum()
    .reset_index()
    .sort_values(["item_type", "items_count"], ascending=[True, False])
)

display(df_licenses_per_resource_type.head())

print(f"File saved to: {output_licenses_per_resource_type.resolve()}")

License / resource type rows: 44
Detected resource types: 5
Global license distribution by resource type:


,item_type,license_value,items_count
6,corpus,PUB,461
3,corpus,Creative Commons - Attribution-NonCommercial-N...,412
8,corpus,http://creativecommons.org/licenses/by-nc-nd/4.0/,411
4,corpus,Creative Commons - Attribution-NonCommercial-S...,26
10,corpus,http://creativecommons.org/licenses/by-nc-sa/4.0/,26
1,corpus,Creative Commons - Attribution 4.0 Internation...,10
5,corpus,Creative Commons - Attribution-ShareAlike 4.0 ...,10
12,corpus,http://creativecommons.org/licenses/by-sa/4.0/,10
13,corpus,https://creativecommons.org/licenses/by/4.0,10
7,corpus,[no license],9


,item_type,license_value,items_count,item_ids,archived_items_count,withdrawn_items_count,non_discoverable_items_count,reference_month,environment,source,metric_definition
0,corpus,PUB,461,00001e22-b7ab-4e13-aa3f-335300ecc4ac; 00be2594...,461,0,0,2026-04,preprod,postgres,distribution of archived DSpace items by licen...
1,corpus,Creative Commons - Attribution-NonCommercial-N...,412,00001e22-b7ab-4e13-aa3f-335300ecc4ac; 00be2594...,412,0,0,2026-04,preprod,postgres,distribution of archived DSpace items by licen...
2,corpus,http://creativecommons.org/licenses/by-nc-nd/4.0/,411,00001e22-b7ab-4e13-aa3f-335300ecc4ac; 00be2594...,411,0,0,2026-04,preprod,postgres,distribution of archived DSpace items by licen...
3,corpus,Creative Commons - Attribution-NonCommercial-S...,26,23cc9267-a87d-4743-ad41-408c4163453e; 2da9b55e...,26,0,0,2026-04,preprod,postgres,distribution of archived DSpace items by licen...
4,corpus,http://creativecommons.org/licenses/by-nc-sa/4.0/,26,23cc9267-a87d-4743-ad41-408c4163453e; 2da9b55e...,26,0,0,2026-04,preprod,postgres,distribution of archived DSpace items by licen...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\always\2026-04\db_licenses_per_resource_type.csv


## **Activities**

### Items uploaded in the reference month with bitstream and MIME type

In [45]:
# - item_id
# - deposit_date
# - bitstreams_count
# - mime_types
# - total_size_bytes
# - total_size_kb
# - total_size_mb
# - reference_month
# - environment
# - source
# - metric_definition

query_items_in_month = """
WITH provenance_dates AS (
    SELECT
        mv.dspace_object_id AS item_id,
        MIN(
            ((regexp_match(
                mv.text_value,
                'on ([0-9]{4}-[0-9]{2}-[0-9]{2})'
            ))[1])::date
        ) AS deposit_date
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'description'
      AND mfr.qualifier = 'provenance'
      AND mv.text_value ~ 'on [0-9]{4}-[0-9]{2}-[0-9]{2}'
    GROUP BY mv.dspace_object_id
),
item_bitstreams AS (
    SELECT
        i.uuid AS item_id,
        COUNT(DISTINCT bs.uuid) AS bitstreams_count,
        COALESCE(SUM(bs.size_bytes), 0) AS total_size_bytes,
        COALESCE(SUM(bs.size_bytes), 0) / 1024.0 AS total_size_kb,
        COALESCE(SUM(bs.size_bytes), 0) / 1024.0 / 1024.0 AS total_size_mb,
        STRING_AGG(
            DISTINCT COALESCE(bfr.mimetype, '[unknown]'),
            '; '
            ORDER BY COALESCE(bfr.mimetype, '[unknown]')
        ) AS mime_types
    FROM item i
    LEFT JOIN item2bundle i2b
        ON i.uuid = i2b.item_id
    LEFT JOIN bundle bun
        ON i2b.bundle_id = bun.uuid
    LEFT JOIN bundle2bitstream b2bs
        ON bun.uuid = b2bs.bundle_id
    LEFT JOIN bitstream bs
        ON b2bs.bitstream_id = bs.uuid
    LEFT JOIN bitstreamformatregistry bfr
        ON bs.bitstream_format_id = bfr.bitstream_format_id
    WHERE i.in_archive = true
      AND (bs.deleted = false OR bs.deleted IS NULL)
    GROUP BY i.uuid
)
SELECT
    i.uuid AS item_id,
    pd.deposit_date,
    ib.bitstreams_count,
    ib.mime_types,
    ib.total_size_bytes,
    ROUND(ib.total_size_kb, 2) AS total_size_kb,
    ROUND(ib.total_size_mb, 2) AS total_size_mb
FROM item i
JOIN provenance_dates pd
    ON i.uuid = pd.item_id
LEFT JOIN item_bitstreams ib
    ON i.uuid = ib.item_id
WHERE i.in_archive = true
  AND DATE_TRUNC('month', pd.deposit_date) = DATE %s
ORDER BY pd.deposit_date DESC, ib.total_size_bytes DESC;
"""

df_items_in_month = pd.read_sql_query(
    query_items_in_month,
    conn,
    params=[REPORT_MONTH_DATE]
)

df_items_in_month["reference_month"] = REPORT_MONTH
df_items_in_month["environment"] = ENVIRONMENT
df_items_in_month["source"] = "postgres"
df_items_in_month["metric_definition"] = "archived items uploaded in reference month with bitstream count, size and MIME types"

output_items_in_month = export_path("db_items_uploaded_in_month_with_bitstreams.csv")
df_items_in_month.to_csv(output_items_in_month, index=False)

print(f"Items uploaded in the month: {len(df_items_in_month)}")
print(f"Total bitstreams: {df_items_in_month['bitstreams_count'].sum()}")
print(f"Total size in MB: {df_items_in_month['total_size_mb'].sum():.2f}")
print(df_items_in_month.head())
print(f"File saved to: {output_items_in_month.resolve()}")

Items uploaded in the month: 5
Total bitstreams: 38
Total size in MB: 6979.78
                                item_id deposit_date  bitstreams_count  \
0  e38f9258-8508-4838-894c-01c26d4cf9b3   2026-04-23                 2   
1  aeeec067-2091-401a-9503-be401c8c820a   2026-04-17                 3   
2  35b566ce-6199-4f20-aa8e-a2b86133adbc   2026-04-17                 3   
3  ee4abc6b-542d-46ae-8cef-3c0b95e1d017   2026-04-14                 4   
4  d4f2403a-e856-4022-ae77-d81b2a4761e0   2026-04-02                26   

                                          mime_types  total_size_bytes  \
0                text/csv; text/plain; charset=utf-8      4.038200e+04   
1  application/zip; text/plain; text/plain; chars...      1.484833e+06   
2  application/zip; text/plain; text/plain; chars...      2.677080e+05   
3  application/x-rar-compressed; text/plain; char...      7.312185e+09   
4  application/octet-stream; text/plain; charset=...      4.845999e+06   

   total_size_kb  total_size_mb 

### Items uploaded in the reference month with submitters

In [46]:
# - submitter_email
# - total_items
# - item_ids
# - reference_month
# - environment
# - source
# - metric_definition


query_items_by_submitter = """
WITH provenance_dates AS (
    SELECT
        mv.dspace_object_id AS item_id,
        MIN(
            ((regexp_match(
                mv.text_value,
                'on ([0-9]{4}-[0-9]{2}-[0-9]{2})'
            ))[1])::date
        ) AS deposit_date
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'description'
      AND mfr.qualifier = 'provenance'
      AND mv.text_value ~ 'on [0-9]{4}-[0-9]{2}-[0-9]{2}'
    GROUP BY mv.dspace_object_id
)
SELECT
    COALESCE(e.email, '[no email]') AS submitter_email,
    COUNT(DISTINCT i.uuid) AS total_items,
    STRING_AGG(DISTINCT i.uuid::text, ' | ') AS item_ids
FROM item i
JOIN provenance_dates pd
    ON i.uuid = pd.item_id
LEFT JOIN eperson e
    ON i.submitter_id = e.uuid
WHERE i.in_archive = true
  AND DATE_TRUNC('month', pd.deposit_date) = DATE %s
GROUP BY COALESCE(e.email, '[no email]')
ORDER BY total_items DESC, submitter_email;
"""

df_items_by_submitter = pd.read_sql_query(
    query_items_by_submitter,
    conn,
    params=[REPORT_MONTH_DATE]
)

df_items_by_submitter["reference_month"] = REPORT_MONTH
df_items_by_submitter["environment"] = ENVIRONMENT
df_items_by_submitter["source"] = "postgres"
df_items_by_submitter["metric_definition"] = "items per submitter for archived items whose dc.description.provenance deposit date falls in the reference month"

output_items_by_submitter = export_path("db_items_uploaded_in_month_by_submitter.csv")
df_items_by_submitter.to_csv(output_items_by_submitter, index=False)

print("Distribution of items by submitter (items uploaded in the month):")
print(df_items_by_submitter.head())
print(f"File saved to: {output_items_by_submitter.resolve()}")

Distribution of items by submitter (items uploaded in the month):
              submitter_email  total_items  \
0  cristiana.cervini@unibo.it            2   
1   amedeo.raschieri@unimi.it            1   
2     fabio.ardolino@unisi.it            1   
3      irene.buttazzi@upf.edu            1   

                                            item_ids reference_month  \
0  35b566ce-6199-4f20-aa8e-a2b86133adbc | aeeec06...         2026-04   
1               d4f2403a-e856-4022-ae77-d81b2a4761e0         2026-04   
2               ee4abc6b-542d-46ae-8cef-3c0b95e1d017         2026-04   
3               e38f9258-8508-4838-894c-01c26d4cf9b3         2026-04   

  environment    source                                  metric_definition  
0     preprod  postgres  items per submitter for archived items whose d...  
1     preprod  postgres  items per submitter for archived items whose d...  
2     preprod  postgres  items per submitter for archived items whose d...  
3     preprod  postgres  items per

### Items uploaded in the reference month by collection

In [47]:
# - collection_name
# - total_items
# - item_ids
# - reference_month
# - environment
# - source
# - metric_definition

query_items_by_collection = """
WITH provenance_dates AS (
    SELECT
        mv.dspace_object_id AS item_id,
        MIN(
            ((regexp_match(
                mv.text_value,
                'on ([0-9]{4}-[0-9]{2}-[0-9]{2})'
            ))[1])::date
        ) AS deposit_date
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'description'
      AND mfr.qualifier = 'provenance'
      AND mv.text_value ~ 'on [0-9]{4}-[0-9]{2}-[0-9]{2}'
    GROUP BY mv.dspace_object_id
)
SELECT
    COALESCE(mv.text_value, '[no collection title]') AS collection_name,
    COUNT(DISTINCT i.uuid) AS total_items,
    STRING_AGG(DISTINCT i.uuid::text, ' | ') AS item_ids
FROM item i
JOIN provenance_dates pd
    ON i.uuid = pd.item_id
JOIN collection2item c2i
    ON i.uuid = c2i.item_id
JOIN collection c
    ON c2i.collection_id = c.uuid
LEFT JOIN metadatavalue mv
    ON mv.dspace_object_id = c.uuid
LEFT JOIN metadatafieldregistry mfr
    ON mv.metadata_field_id = mfr.metadata_field_id
LEFT JOIN metadataschemaregistry msr
    ON mfr.metadata_schema_id = msr.metadata_schema_id
WHERE i.in_archive = true
  AND DATE_TRUNC('month', pd.deposit_date) = DATE %s
  AND msr.short_id = 'dc'
  AND mfr.element = 'title'
GROUP BY COALESCE(mv.text_value, '[no collection title]')
ORDER BY total_items DESC;
"""

df_items_by_collection = pd.read_sql_query(
    query_items_by_collection,
    conn,
    params=[REPORT_MONTH_DATE]
)

df_items_by_collection["reference_month"] = REPORT_MONTH
df_items_by_collection["environment"] = ENVIRONMENT
df_items_by_collection["source"] = "postgres"
df_items_by_collection["metric_definition"] = "collection distribution for archived items whose dc.description.provenance deposit date falls in the reference month"

output_items_by_collection = export_path("db_items_uploaded_in_month_by_collection.csv")
df_items_by_collection.to_csv(output_items_by_collection, index=False)

print("Distribution by collection of items uploaded in the month:")
print(df_items_by_collection.head())
print(f"File saved to: {output_items_by_collection.resolve()}")

Distribution by collection of items uploaded in the month:
                  collection_name  total_items  \
0  ILC4CLARIN : OPEN Data & Tools            3   
1   ILC4CLARIN : ILC Data & Tools            2   

                                            item_ids reference_month  \
0  d4f2403a-e856-4022-ae77-d81b2a4761e0 | e38f925...         2026-04   
1  35b566ce-6199-4f20-aa8e-a2b86133adbc | aeeec06...         2026-04   

  environment    source                                  metric_definition  
0     preprod  postgres  collection distribution for archived items who...  
1     preprod  postgres  collection distribution for archived items who...  
File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\monthly\2026-04\db_items_uploaded_in_month_by_collection.csv


### Items uploaded in the reference month by languages

In [48]:
# - language
# - total_items
# - item_ids
# - reference_month
# - environment
# - source
# - metric_definition

query_items_by_language = """
WITH provenance_dates AS (
    SELECT
        mv.dspace_object_id AS item_id,
        MIN(
            ((regexp_match(
                mv.text_value,
                'on ([0-9]{4}-[0-9]{2}-[0-9]{2})'
            ))[1])::date
        ) AS deposit_date
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'description'
      AND mfr.qualifier = 'provenance'
      AND mv.text_value ~ 'on [0-9]{4}-[0-9]{2}-[0-9]{2}'
    GROUP BY mv.dspace_object_id
)
SELECT
    COALESCE(mv.text_value, '[no language]') AS language,
    COUNT(DISTINCT i.uuid) AS total_items,
    STRING_AGG(DISTINCT i.uuid::text, ' | ') AS item_ids
FROM item i
JOIN provenance_dates pd
    ON i.uuid = pd.item_id
LEFT JOIN metadatavalue mv
    ON mv.dspace_object_id = i.uuid
LEFT JOIN metadatafieldregistry mfr
    ON mv.metadata_field_id = mfr.metadata_field_id
LEFT JOIN metadataschemaregistry msr
    ON mfr.metadata_schema_id = msr.metadata_schema_id
WHERE i.in_archive = true
  AND DATE_TRUNC('month', pd.deposit_date) = DATE %s
  AND msr.short_id = 'dc'
  AND mfr.element = 'language'
  AND mfr.qualifier = 'iso'
GROUP BY COALESCE(mv.text_value, '[no language]')
ORDER BY total_items DESC;
"""

df_items_by_language = pd.read_sql_query(
    query_items_by_language,
    conn,
    params=[REPORT_MONTH_DATE]
)

df_items_by_language["reference_month"] = REPORT_MONTH
df_items_by_language["environment"] = ENVIRONMENT
df_items_by_language["source"] = "postgres"
df_items_by_language["metric_definition"] = "language distribution for archived items whose dc.description.provenance deposit date falls in the reference month"

output_items_by_language = export_path("db_items_uploaded_in_month_by_language.csv")
df_items_by_language.to_csv(output_items_by_language, index=False)

print("Distribution by language of items uploaded in the month:")
print(df_items_by_language.head())
print(f"File saved to: {output_items_by_language.resolve()}")

Distribution by language of items uploaded in the month:


  language  total_items                                           item_ids  \
0      ita            4  35b566ce-6199-4f20-aa8e-a2b86133adbc | aeeec06...   
1      mul            2  35b566ce-6199-4f20-aa8e-a2b86133adbc | aeeec06...   
2      por            2  35b566ce-6199-4f20-aa8e-a2b86133adbc | aeeec06...   
3      spa            2  35b566ce-6199-4f20-aa8e-a2b86133adbc | aeeec06...   
4      eng            1               d4f2403a-e856-4022-ae77-d81b2a4761e0   

  reference_month environment    source  \
0         2026-04     preprod  postgres   
1         2026-04     preprod  postgres   
2         2026-04     preprod  postgres   
3         2026-04     preprod  postgres   
4         2026-04     preprod  postgres   

                                   metric_definition  
0  language distribution for archived items whose...  
1  language distribution for archived items whose...  
2  language distribution for archived items whose...  
3  language distribution for archived items whose..

### Items uploaded in the reference month (including workflow approval timelines)

In [49]:
# - item_id
# - event_date
# - event_type
# - modifier_name
# - modifier_email
# - modified_by
# - last_modified
# - event_line
# - reference_month
# - environment
# - source
# - metric_definition

query_items_modified_in_month = """
WITH provenance_values AS (
    SELECT
        mv.dspace_object_id AS item_id,
        mv.text_value AS provenance_text
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'description'
      AND mfr.qualifier = 'provenance'
),
event_lines AS (
    SELECT
        item_id,
        regexp_split_to_table(provenance_text, E'\\n') AS event_line
    FROM provenance_values
),
parsed_events AS (
    SELECT
        item_id,
        event_line,

        ((regexp_match(
            event_line,
            'on ([0-9]{4}-[0-9]{2}-[0-9]{2})T'
        ))[1])::date AS event_date,

        (regexp_match(
            event_line,
            'by\\s+(.+?)\\s*\\(([A-Za-z0-9._%%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,})\\)'
        ))[1] AS modifier_name,

        (regexp_match(
            event_line,
            'by\\s+(.+?)\\s*\\(([A-Za-z0-9._%%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,})\\)'
        ))[2] AS modifier_email,

        CASE
            WHEN event_line ~* 'deleted bitstream' THEN 'deleted_bitstream'
            WHEN event_line ~* 'added bitstream' THEN 'added_bitstream'
            WHEN event_line ~* 'Approved for entry into archive' THEN 'workflow_approval'
            WHEN event_line ~* 'Made available in DSpace' THEN 'made_available'
            WHEN event_line ~* 'Submitted by' THEN 'submitted'
            ELSE 'other'
        END AS event_type
    FROM event_lines
    WHERE event_line ~ 'on [0-9]{4}-[0-9]{2}-[0-9]{2}T'
      AND event_line ~* '(deleted bitstream|added bitstream|Approved for entry into archive|Made available in DSpace|Submitted by)'
)
SELECT
    i.uuid AS item_id,
    pe.event_date,
    pe.event_type,
    pe.modifier_name,
    pe.modifier_email,
    COALESCE(pe.modifier_email, pe.modifier_name, '[not available in provenance]') AS modified_by,
    i.last_modified,
    pe.event_line
FROM parsed_events pe
JOIN item i
    ON pe.item_id = i.uuid
WHERE DATE_TRUNC('month', pe.event_date) = DATE %s
ORDER BY pe.event_date DESC, i.last_modified DESC;
"""

df_items_modified_in_month = pd.read_sql_query(
    query_items_modified_in_month,
    conn,
    params=[REPORT_MONTH_DATE]
)

df_items_modified_in_month["reference_month"] = REPORT_MONTH
df_items_modified_in_month["environment"] = ENVIRONMENT
df_items_modified_in_month["source"] = "postgres"
df_items_modified_in_month["metric_definition"] = "item provenance events in reference month parsed from dc.description.provenance"

output_items_modified_in_month = export_path("db_items_modified_in_month_from_provenance.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_items_modified_in_month.to_csv(output_items_modified_in_month, index=False)

print(f"Modification events in the month: {len(df_items_modified_in_month)}")
display(df_items_modified_in_month.head())
print(f"File saved to: {output_items_modified_in_month.resolve()}")

Modification events in the month: 26


,item_id,event_date,event_type,modifier_name,modifier_email,modified_by,last_modified,event_line,reference_month,environment,source,metric_definition
0,e38f9258-8508-4838-894c-01c26d4cf9b3,2026-04-30,workflow_approval,Valeria Quochi,valeria.quochi@ilc.cnr.it,valeria.quochi@ilc.cnr.it,2026-04-30 15:37:19.162000+00:00,Step: editstep - action:editaction Approved fo...,2026-04,preprod,postgres,item provenance events in reference month pars...
1,e38f9258-8508-4838-894c-01c26d4cf9b3,2026-04-30,workflow_approval,Michele Mallia,michele.mallia@ilc.cnr.it,michele.mallia@ilc.cnr.it,2026-04-30 15:37:19.162000+00:00,Step: finaleditstep - action:finaleditaction A...,2026-04,preprod,postgres,item provenance events in reference month pars...
2,e38f9258-8508-4838-894c-01c26d4cf9b3,2026-04-30,made_available,None,None,[not available in provenance],2026-04-30 15:37:19.162000+00:00,Made available in DSpace on 2026-04-30T15:37:1...,2026-04,preprod,postgres,item provenance events in reference month pars...
3,35b566ce-6199-4f20-aa8e-a2b86133adbc,2026-04-30,made_available,None,None,[not available in provenance],2026-04-30 15:36:54.622000+00:00,Made available in DSpace on 2026-04-30T15:36:5...,2026-04,preprod,postgres,item provenance events in reference month pars...
4,35b566ce-6199-4f20-aa8e-a2b86133adbc,2026-04-30,workflow_approval,Valeria Quochi,valeria.quochi@ilc.cnr.it,valeria.quochi@ilc.cnr.it,2026-04-30 15:36:54.622000+00:00,Step: editstep - action:editaction Approved fo...,2026-04,preprod,postgres,item provenance events in reference month pars...


File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\monthly\2026-04\db_items_modified_in_month_from_provenance.csv


### Average items file size in the reference month

In [50]:
# - avg_size_bytes
# - avg_size_kb
# - avg_file_size_mb
# - reference_month
# - environment
# - source
# - metric_definition

query_avg_file_size_month = """
WITH provenance_dates AS (
    SELECT
        mv.dspace_object_id AS item_id,
        MIN(
            ((regexp_match(
                mv.text_value,
                'on ([0-9]{4}-[0-9]{2}-[0-9]{2})'
            ))[1])::date
        ) AS deposit_date
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'description'
      AND mfr.qualifier = 'provenance'
      AND mv.text_value ~ 'on [0-9]{4}-[0-9]{2}-[0-9]{2}'
    GROUP BY mv.dspace_object_id
)
SELECT
    AVG(bs.size_bytes) AS avg_size_bytes,
    AVG(bs.size_bytes) / 1024.0 AS avg_size_kb,
    AVG(bs.size_bytes) / 1024.0 / 1024.0 AS avg_file_size_mb
FROM item i
JOIN provenance_dates pd ON i.uuid = pd.item_id
JOIN item2bundle i2b ON i.uuid = i2b.item_id
JOIN bundle b ON i2b.bundle_id = b.uuid
JOIN bundle2bitstream b2bs ON b.uuid = b2bs.bundle_id
JOIN bitstream bs ON b2bs.bitstream_id = bs.uuid
WHERE i.in_archive = true
  AND (bs.deleted = false OR bs.deleted IS NULL)
  AND DATE_TRUNC('month', pd.deposit_date) = DATE %s;
"""

df_avg_file_size_month = pd.read_sql_query(
    query_avg_file_size_month,
    conn,
    params=[REPORT_MONTH_DATE]
)

# =========================================================
# METADATI EXPORT
# =========================================================
df_avg_file_size_month["reference_month"] = REPORT_MONTH
df_avg_file_size_month["environment"] = ENVIRONMENT
df_avg_file_size_month["source"] = "postgres"
df_avg_file_size_month["metric_definition"] = "average bitstream size for items deposited in reference month"

# =========================================================
# EXPORT CSV
# =========================================================
output_avg_file_size_month = export_path("db_avg_file_size_month.csv")
df_avg_file_size_month.to_csv(output_avg_file_size_month, index=False)

# =========================================================
# PRINT
# =========================================================
print("Average file size in the month:")
print(df_avg_file_size_month)
print(f"File saved to: {output_avg_file_size_month.resolve()}")

Average file size in the month:
   avg_size_bytes    avg_size_kb  avg_file_size_mb reference_month  \
0    1.926006e+08  188086.553917        183.678275         2026-04   

  environment    source                                  metric_definition  
0     preprod  postgres  average bitstream size for items deposited in ...  
File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\monthly\2026-04\db_avg_file_size_month.csv


### Items uploaded in the referencef month by type and author

In [51]:
# - item_id
# - deposit_date
# - last_modified
# - item_types
# - authors
# - reference_month
# - environment
# - source
# - metric_definition

query_items_details = """
WITH provenance_dates AS (
    SELECT
        mv.dspace_object_id AS item_id,
        MIN(
            ((regexp_match(
                mv.text_value,
                'on ([0-9]{4}-[0-9]{2}-[0-9]{2})'
            ))[1])::date
        ) AS deposit_date
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'description'
      AND mfr.qualifier = 'provenance'
      AND mv.text_value ~ 'on [0-9]{4}-[0-9]{2}-[0-9]{2}'
    GROUP BY mv.dspace_object_id
),
item_types AS (
    SELECT
        mv.dspace_object_id AS item_id,
        STRING_AGG(DISTINCT mv.text_value, ' | ') AS item_types
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'type'
    GROUP BY mv.dspace_object_id
),
item_authors AS (
    SELECT
        mv.dspace_object_id AS item_id,
        STRING_AGG(DISTINCT mv.text_value, ' | ') AS authors
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'contributor'
      AND mfr.qualifier = 'author'
    GROUP BY mv.dspace_object_id
)
SELECT
    i.uuid AS item_id,
    pd.deposit_date,
    i.last_modified,
    COALESCE(it.item_types, '[no type]') AS item_types,
    COALESCE(ia.authors, '[no author]') AS authors
FROM item i
JOIN provenance_dates pd
    ON i.uuid = pd.item_id
LEFT JOIN item_types it
    ON i.uuid = it.item_id
LEFT JOIN item_authors ia
    ON i.uuid = ia.item_id
WHERE i.in_archive = true
  AND DATE_TRUNC('month', pd.deposit_date) = DATE %s
ORDER BY pd.deposit_date DESC, i.last_modified DESC;
"""

df_items_details = pd.read_sql_query(
    query_items_details,
    conn,
    params=[REPORT_MONTH_DATE]
)

df_items_details["reference_month"] = REPORT_MONTH
df_items_details["environment"] = ENVIRONMENT
df_items_details["source"] = "postgres"
df_items_details["metric_definition"] = "details of archived items whose dc.description.provenance deposit date falls in the reference month"

output_items_details = export_path("db_items_uploaded_in_month_details.csv")
df_items_details.to_csv(output_items_details, index=False)

print("Details of items uploaded in the month:")
print(df_items_details.head())
print(f"File saved to: {output_items_details.resolve()}")

Details of items uploaded in the month:
                                item_id deposit_date  \
0  e38f9258-8508-4838-894c-01c26d4cf9b3   2026-04-23   
1  35b566ce-6199-4f20-aa8e-a2b86133adbc   2026-04-17   
2  aeeec067-2091-401a-9503-be401c8c820a   2026-04-17   
3  ee4abc6b-542d-46ae-8cef-3c0b95e1d017   2026-04-14   
4  d4f2403a-e856-4022-ae77-d81b2a4761e0   2026-04-02   

                     last_modified item_types  \
0 2026-04-30 15:37:19.162000+00:00     corpus   
1 2026-04-30 15:36:54.622000+00:00     corpus   
2 2026-04-30 15:36:46.302000+00:00     corpus   
3 2026-04-16 12:31:00.146000+00:00     corpus   
4 2026-04-16 12:31:07.143000+00:00     corpus   

                                             authors reference_month  \
0                                     Irene Buttazzi         2026-04   
1               Cervini, Cristiana | Paone, Emanuela         2026-04   
2               Cervini, Cristiana | Paone, Emanuela         2026-04   
3  Fabio Ardolino | Lorenza Brasile | Si

### Most active collections in the reference month

In [52]:
# - collection_id
# - collection_name
# - items_added_in_month
# - modified_items_in_month
# - modification_events_in_month
# - total_activity_score
# - added_item_ids
# - modified_item_ids
# - reference_month
# - environment
# - source
# - metric_definition

query_active_collections_combined = """
WITH provenance_values AS (
    SELECT
        mv.dspace_object_id AS item_id,
        mv.text_value AS provenance_text
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'description'
      AND mfr.qualifier = 'provenance'
),
deposit_dates AS (
    SELECT
        item_id,
        MIN(
            ((regexp_match(
                provenance_text,
                'Submitted by .* on ([0-9]{4}-[0-9]{2}-[0-9]{2})T'
            ))[1])::date
        ) AS deposit_date
    FROM provenance_values
    WHERE provenance_text ~ 'Submitted by .* on [0-9]{4}-[0-9]{2}-[0-9]{2}T'
    GROUP BY item_id
),
event_lines AS (
    SELECT
        item_id,
        regexp_split_to_table(provenance_text, E'\\n') AS event_line
    FROM provenance_values
),
modification_events AS (
    SELECT
        item_id,
        ((regexp_match(
            event_line,
            'on ([0-9]{4}-[0-9]{2}-[0-9]{2})T'
        ))[1])::date AS event_date,
        CASE
            WHEN event_line ~* 'deleted bitstream' THEN 'deleted_bitstream'
            WHEN event_line ~* 'added bitstream' THEN 'added_bitstream'
            WHEN event_line ~* 'Approved for entry into archive' THEN 'workflow_approval'
            WHEN event_line ~* 'Made available in DSpace' THEN 'made_available'
            WHEN event_line ~* 'Submitted by' THEN 'submitted'
            ELSE 'other'
        END AS event_type
    FROM event_lines
    WHERE event_line ~ 'on [0-9]{4}-[0-9]{2}-[0-9]{2}T'
      AND event_line ~* '(deleted bitstream|added bitstream|Approved for entry into archive|Made available in DSpace|Submitted by)'
),
collection_titles AS (
    SELECT
        mv.dspace_object_id AS collection_id,
        MIN(mv.text_value) AS collection_name
    FROM metadatavalue mv
    JOIN metadatafieldregistry mfr
        ON mv.metadata_field_id = mfr.metadata_field_id
    JOIN metadataschemaregistry msr
        ON mfr.metadata_schema_id = msr.metadata_schema_id
    WHERE msr.short_id = 'dc'
      AND mfr.element = 'title'
      AND mv.text_value IS NOT NULL
      AND TRIM(mv.text_value) <> ''
    GROUP BY mv.dspace_object_id
),
items_added AS (
    SELECT
        c.uuid AS collection_id,
        COUNT(DISTINCT i.uuid) AS items_added_in_month,
        STRING_AGG(DISTINCT i.uuid::text, '; ' ORDER BY i.uuid::text) AS added_item_ids
    FROM item i
    JOIN deposit_dates dd
        ON i.uuid = dd.item_id
    JOIN collection2item c2i
        ON i.uuid = c2i.item_id
    JOIN collection c
        ON c2i.collection_id = c.uuid
    WHERE i.in_archive = true
      AND DATE_TRUNC('month', dd.deposit_date) = DATE %s
    GROUP BY c.uuid
),
items_modified AS (
    SELECT
        c.uuid AS collection_id,
        COUNT(*) AS modification_events_in_month,
        COUNT(DISTINCT i.uuid) AS modified_items_in_month,
        STRING_AGG(DISTINCT i.uuid::text, '; ' ORDER BY i.uuid::text) AS modified_item_ids
    FROM modification_events me
    JOIN item i
        ON me.item_id = i.uuid
    JOIN collection2item c2i
        ON i.uuid = c2i.item_id
    JOIN collection c
        ON c2i.collection_id = c.uuid
    WHERE i.in_archive = true
      AND DATE_TRUNC('month', me.event_date) = DATE %s
      AND me.event_type <> 'submitted'
    GROUP BY c.uuid
)
SELECT
    COALESCE(ia.collection_id, im.collection_id) AS collection_id,
    COALESCE(ct.collection_name, '[no collection title]') AS collection_name,

    COALESCE(ia.items_added_in_month, 0) AS items_added_in_month,
    COALESCE(im.modified_items_in_month, 0) AS modified_items_in_month,
    COALESCE(im.modification_events_in_month, 0) AS modification_events_in_month,

    (
        COALESCE(ia.items_added_in_month, 0)
        + COALESCE(im.modification_events_in_month, 0)
    ) AS total_activity_score,

    ia.added_item_ids,
    im.modified_item_ids

FROM items_added ia
FULL OUTER JOIN items_modified im
    ON ia.collection_id = im.collection_id
LEFT JOIN collection_titles ct
    ON COALESCE(ia.collection_id, im.collection_id) = ct.collection_id
ORDER BY total_activity_score DESC, items_added_in_month DESC, modification_events_in_month DESC, collection_name;
"""

df_active_collections_combined = pd.read_sql_query(
    query_active_collections_combined,
    conn,
    params=[REPORT_MONTH_DATE, REPORT_MONTH_DATE]
)

df_active_collections_combined["reference_month"] = REPORT_MONTH
df_active_collections_combined["environment"] = ENVIRONMENT
df_active_collections_combined["source"] = "postgres"
df_active_collections_combined["metric_definition"] = (
    "collections ranked by combined monthly activity: archived items added "
    "plus modification events parsed from dc.description.provenance"
)

output_active_collections_combined = export_path("db_active_collections_combined_in_month.csv")

if EXPORT_OUTPUTS:
    export_dir.mkdir(parents=True, exist_ok=True)
df_active_collections_combined.to_csv(output_active_collections_combined, index=False)

print("Most active collections in the month: items added + modifications")
display(df_active_collections_combined.head())
print(f"Total active collections in the month: {len(df_active_collections_combined)}")
print(f"File saved to: {output_active_collections_combined.resolve()}")

Most active collections in the month: items added + modifications


,collection_id,collection_name,items_added_in_month,modified_items_in_month,modification_events_in_month,total_activity_score,added_item_ids,modified_item_ids,reference_month,environment,source,metric_definition
0,57c1de63-635d-41ef-8e64-119865cb8f80,ILC4CLARIN : OPEN Data & Tools,3,4,13,16,d4f2403a-e856-4022-ae77-d81b2a4761e0; e38f9258...,666e14cc-8e93-4922-ad00-a124d8fe84de; d4f2403a...,2026-04,preprod,postgres,collections ranked by combined monthly activit...
1,79c6fbcd-aaac-42b5-aa7d-9413eb906511,ILC4CLARIN : ILC Data & Tools,2,2,8,10,35b566ce-6199-4f20-aa8e-a2b86133adbc; aeeec067...,35b566ce-6199-4f20-aa8e-a2b86133adbc; aeeec067...,2026-04,preprod,postgres,collections ranked by combined monthly activit...


Total active collections in the month: 2
File saved to: C:\Users\Michele Mallia\Lavoro\CLARIN\ILC4CLARIN\Report\notebooks\exports\db\monthly\2026-04\db_active_collections_combined_in_month.csv
